In [ ]:
import urllib.request

files = {
    "EDITIONS": "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz",
    "WORKS": "https://openlibrary.org/data/ol_dump_works_latest.txt.gz",
    "AUTHORS": "https://openlibrary.org/data/ol_dump_authors_latest.txt.gz",
}

for name, url in files.items():
    print(f"\n--- {name} ---")
    req = urllib.request.Request(url, method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            size = r.headers.get("Content-Length")
            print("HTTP :", r.status)
            print("Taille :", round(int(size)/1024**3, 2), "GB" if size else "inconnue")
            print("Type :", r.headers.get("Content-Type"))
    except Exception as e:
        print("Erreur :", str(e))

In [ ]:
from huggingface_hub import hf_hub_upload, hf_hub_download
from kaggle_secrets import UserSecretsClient
import os, hashlib

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO_ID = "Moses-team/nexus-science-backup"

V10_BACKUP_DIR = "v10_recovery_v10_3"

commit, stage_path = v10_read_commit()
stage_name = commit["stage_filename"]
remote_stage = f"{V10_BACKUP_DIR}/{stage_name}"

hf_hub_upload(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    path_or_fileobj=stage_path,
    path_in_repo=remote_stage,
    token=HF_TOKEN
)

print("=== TRANSFERT STAGE V10.3 ===")
print("Génération :", commit["generation"])
print("Fichier :", remote_stage)
print("Taille locale :", os.path.getsize(stage_path))
print("STATUT : STAGE ENVOYE")


# Bibliothèque Scientifique — Nexus Science
## Phase 1 : Collecte et filtrage Open Library
Environnement cloud Kaggle — traitement en flux

In [ ]:
import os, gzip, json, urllib.request

BASE = "/kaggle/working/nexus_science"
os.makedirs(BASE, exist_ok=True)

print("Dossier de travail :", BASE)
print("Environnement prêt pour le traitement Open Library.")

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
subjects = collections.Counter()
languages = collections.Counter()
years = collections.Counter()
count = 0
valid = 0

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            count += 1

            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                valid += 1

                for s in rec.get("subjects", []):
                    if isinstance(s, str):
                        subjects[s.lower()] += 1

                for lang in rec.get("languages", []):
                    if isinstance(lang, dict):
                        languages[lang.get("key", "")] += 1

                date = str(rec.get("publish_date", ""))
                if len(date) >= 4 and date[:4].isdigit():
                    years[date[:4]] += 1

            except Exception:
                pass

print("Lignes examinées :", count)
print("Notices JSON valides :", valid)
print("\nTOP 40 SUJETS :")
for s, n in subjects.most_common(40):
    print(n, "|", s)

print("\nTOP LANGUES :")
for s, n in languages.most_common(20):
    print(n, "|", s)

print("\nDATES les plus fréquentes :")
for s, n in years.most_common(20):
    print(n, "|", s)

In [ ]:
SCIENCE_KEYWORDS = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological",
    "medicine", "medical", "health", "anatomy",
    "engineering", "technology", "computer science",
    "astronomy", "astrophysics", "geology", "geography",
    "ecology", "environment", "botany", "zoology",
    "statistics", "research", "laboratory", "microbiology",
    "genetics", "pharmacology", "neuroscience", "agriculture",
    "economics", "psychology", "sociology", "education",
    "philosophy of science", "nursing", "veterinary"
]

print("Filtre scientifique initialisé.")
print("Nombre de mots-clés :", len(SCIENCE_KEYWORDS))
print("Période cible : 1720–2026")
print("Langues cibles : multilingues")

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
count = 0
selected = 0
matched_subjects = collections.Counter()
examples = []

keywords = [k.lower() for k in SCIENCE_KEYWORDS]

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                count += 1

                title = str(rec.get("title", "")).lower()
                subjects = rec.get("subjects", [])
                subject_text = " ".join(
                    s.lower() for s in subjects if isinstance(s, str)
                )
                text_to_check = title + " " + subject_text

                if any(k in text_to_check for k in keywords):
                    selected += 1
                    for s in subjects:
                        if isinstance(s, str):
                            matched_subjects[s] += 1

                    if len(examples) < 15:
                        examples.append(rec.get("title", "Sans titre"))

            except Exception:
                pass

print("Notices examinées :", count)
print("Notices sélectionnées :", selected)
print("Proportion :", round(selected/count*100, 2) if count else 0, "%")
print("\nEXEMPLES :")
for title in examples:
    print("-", title)

print("\nSUJETS RETENUS :")
for subject, n in matched_subjects.most_common(20):
    print(n, "|", subject)

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
count = selected = excluded = 0
examples = []

science = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "medicine", "medical",
    "anatomy", "engineering", "computer science", "astronomy",
    "geology", "ecology", "botany", "zoology", "statistics",
    "microbiology", "genetics", "pharmacology", "neuroscience",
    "agriculture", "nursing", "veterinary", "technology",
    "environmental science", "biotechnology", "physics",
    "natural science", "research methodology"
]

noise = [
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "bill [", "regulations",
    "establishment order", "amendment order",
    "house of lords", "government notice"
]

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                count += 1

                title = str(rec.get("title", "")).lower()
                subjects = [
                    s.lower() for s in rec.get("subjects", [])
                    if isinstance(s, str)
                ]
                searchable = title + " " + " ".join(subjects)

                relevant = any(k in searchable for k in science)
                administrative = any(k in title for k in noise)

                if relevant and not administrative:
                    selected += 1
                    if len(examples) < 25:
                        examples.append(title)
                elif relevant and administrative:
                    excluded += 1

            except Exception:
                pass

print("Notices examinées :", count)
print("Retenues V2 :", selected)
print("Documents administratifs écartés :", excluded)
print("\nEXEMPLES RETENUS :")
for t in examples:
    print("-", t)

In [ ]:
# NEXUS SCIENCE — GLOBAL OPEN LIBRARY EDITIONS SCANNER V1
# Streaming + checkpoint + restart-safe
# Aucune écriture dans Neon.

import os, json, gzip, urllib.request, time, re

BASE = "/kaggle/working/nexus_science"
os.makedirs(BASE, exist_ok=True)

URL = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
OUT = os.path.join(BASE, "scientific_editions.jsonl")
CHECKPOINT = os.path.join(BASE, "scanner_checkpoint.json")

KEYWORDS = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological",
    "medicine", "medical", "health", "anatomy",
    "engineering", "technology", "computer science",
    "astronomy", "astrophysics", "geology", "geography",
    "ecology", "environment", "botany", "zoology",
    "statistics", "research", "laboratory", "microbiology",
    "genetics", "pharmacology", "neuroscience", "agriculture",
    "economics", "psychology", "sociology", "education",
    "nursing", "veterinary", "biotechnology", "mathematics",
    "matematica", "ciencia", "ciencias", "biologia",
    "fisica", "quimica", "medicina", "ingenieria",
    "astronomia", "mathematiques", "scientifique",
    "biologie", "physique", "chimie", "médecine",
    "matemática", "ciência", "biologia", "medicina",
    "engenharia", "astronomia"
]

# Termes fortement administratifs : exclusions prudentes sur le titre seulement
NOISE = [
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "government notice",
    "house of lords bill"
]

def matches(record):
    title = str(record.get("title", "")).lower()
    subjects = record.get("subjects", [])
    if not isinstance(subjects, list):
        subjects = []

    subject_text = " ".join(
        s.lower() for s in subjects if isinstance(s, str)
    )
    searchable = title + " " + subject_text

    relevant = any(k in searchable for k in KEYWORDS)
    excluded = any(k in title for k in NOISE)

    return relevant and not excluded

# Reprise du compteur de lignes
resume_line = 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, "r", encoding="utf-8") as f:
        state = json.load(f)
        resume_line = state.get("scan_lines", 0)

print("Scanner Nexus Science")
print("Lignes déjà parcourues :", resume_line)
print("Fichier de sortie :", OUT)
print("Mode : flux gzip / checkpoint")

scan_lines = resume_line
valid = 0
selected = 0
started = time.time()
last_checkpoint = scan_lines

# append pour préserver les résultats précédents
with open(OUT, "a", encoding="utf-8") as output:
    request = urllib.request.Request(
        URL, headers={"User-Agent": "NexusScience-Research/1.0"}
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        with gzip.GzipFile(fileobj=response, mode="rb") as gz:

            current_line = 0

            for raw_line in gz:
                current_line += 1

                # Relecture du début pour reprendre le checkpoint
                if current_line <= resume_line:
                    if current_line % 1000000 == 0:
                        print("Reprise :", current_line)
                    continue

                scan_lines += 1

                parts = raw_line.rstrip(b"\n").split(b"\t")
                if len(parts) < 4:
                    continue

                try:
                    record = json.loads(parts[-1])
                    valid += 1

                    if matches(record):
                        output.write(json.dumps(
                            record, ensure_ascii=False
                        ) + "\n")
                        selected += 1

                except Exception:
                    pass

                if scan_lines - last_checkpoint >= 10000:
                    output.flush()

                    with open(CHECKPOINT, "w", encoding="utf-8") as cp:
                        json.dump(
                            {"scan_lines": scan_lines},
                            cp
                        )

                    last_checkpoint = scan_lines
                    elapsed = round(time.time() - started, 1)

                    print(
                        "Progression lignes :", scan_lines,
                        "| JSON valides :", valid,
                        "| retenues :", selected,
                        "| secondes :", elapsed
                    )

# Dernier checkpoint
with open(CHECKPOINT, "w", encoding="utf-8") as cp:
    json.dump({"scan_lines": scan_lines}, cp)

print("\nSCAN TERMINÉ")
print("Lignes parcourues :", scan_lines)
print("JSON valides :", valid)
print("Retenues pendant cette session :", selected)
print("Temps (sec) :", round(time.time() - started, 1))
print("Résultats :", OUT)

In [ ]:
!ls -lh /kaggle/working/nexus_science

In [ ]:
import json

with open("/kaggle/working/nexus_science/scanner_checkpoint.json", "r") as f:
    print(json.load(f))

In [ ]:
import os

p = "/kaggle/working/nexus_science/scientific_editions.jsonl"

print("Fichier présent :", os.path.exists(p))
print("Taille (Go) :", round(os.path.getsize(p) / 1024**3, 3))

with open(p, "rb") as f:
    f.seek(max(0, os.path.getsize(p) - 2000))
    print("\nFIN DU FICHIER :\n")
    print(f.read().decode("utf-8", errors="replace"))

In [ ]:
import os

p = "/kaggle/working/nexus_science/scientific_editions.jsonl"

print("Fichier présent :", os.path.exists(p))
if os.path.exists(p):
    print("Taille (Go) :", round(os.path.getsize(p) / 1024**3, 3))

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/nexus_science")

print("=== SAUVEGARDE NEXUS : INVENTAIRE ===")

if base.exists():
    for p in sorted(base.rglob("*")):
        if p.is_file():
            print(f"{p.name} | {p.stat().st_size:,} octets | {p}")
else:
    print("Le dossier nexus_science est absent.")

print("\n=== AUTRES FICHIERS VOLUMINEUX ===")

root = Path("/kaggle/working")
for p in root.iterdir():
    if p.is_file():
        print(f"{p.name} | {round(p.stat().st_size / 1024**3, 3)} Go")

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/temp"),
    Path("/tmp"),
]

print("=== RECHERCHE DE FICHIERS NEXUS ===")

for root in roots:
    if not root.exists():
        continue

    print(f"\nDossier : {root}")

    for p in root.rglob("*"):
        try:
            if p.is_file() and p.stat().st_size > 100_000_000:
                size = p.stat().st_size / 1024**3
                print(f"{size:.3f} Go | {p}")
        except (PermissionError, OSError):
            pass

In [ ]:
import os, shutil

print("=== ENVIRONNEMENT NEXUS ===")
print("Répertoire :", os.getcwd())
print("Espace libre :", round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2), "Go")
print("Espace total :", round(shutil.disk_usage("/kaggle/working").total / 1024**3, 2), "Go")
print("Python :", os.sys.version)

In [ ]:
import os

base = "/kaggle/working/nexus_science"

print("=== ÉTAT DES SAUVEGARDES NEXUS ===")

if os.path.exists(base):
    for f in os.listdir(base):
        p = os.path.join(base, f)
        if os.path.isfile(p):
            print(f"{f} | {round(os.path.getsize(p)/1024**2, 2)} Mo")
        else:
            print(f"{f} | dossier")
else:
    print("Dossier Nexus absent.")

print("\n=== CONFIGURATION ===")
print("RUN_FULL_SCAN =", RUN_FULL_SCAN)

In [ ]:
import os

base = "/kaggle/working/nexus_science"

print("=== ÉTAT DES SAUVEGARDES NEXUS ===")

if os.path.exists(base):
    for f in os.listdir(base):
        p = os.path.join(base, f)
        if os.path.isfile(p):
            print(f"{f} | {round(os.path.getsize(p)/1024**2, 2)} Mo")
        else:
            print(f"{f} | dossier")
else:
    print("Dossier Nexus absent.")

print("\n=== ÉTAT DU SCANNER ===")
print("Cette cellule n'exécute aucun scanner.")

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"

CHUNKS.mkdir(parents=True, exist_ok=True)

print("Dossier V3 :", BASE)
print("Dossier segments :", CHUNKS)
print("Dossiers créés :", BASE.exists() and CHUNKS.exists())

In [ ]:
import json, os
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

BASE.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

manifest = {
    "version": 3,
    "project": "Bibliothèque Scientifique — Nexus Science",
    "source": "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz",
    "status": "prepared",
    "scan_lines": 0,
    "valid_json": 0,
    "selected": 0,
    "invalid_json": 0,
    "out_of_period": 0,
    "completed_chunks": [],
    "current_chunk": None,
    "updated_at_utc": None
}

tmp = MANIFEST.with_suffix(".tmp")

with open(tmp, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
    f.flush()
    os.fsync(f.fileno())

os.replace(tmp, MANIFEST)

# Vérification
loaded = json.loads(MANIFEST.read_text(encoding="utf-8"))

print("=== MANIFESTE V3 ===")
print("Version :", loaded["version"])
print("Statut :", loaded["status"])
print("Segments validés :", len(loaded["completed_chunks"]))
print("Manifeste vérifié :", loaded["version"] == 3)
print("Fichier :", MANIFEST)

In [ ]:
import hashlib
import json
import tempfile
from pathlib import Path

def validate_chunk(path, expected_count, expected_sha256):
    path = Path(path)
    count = 0

    with open(path, "rb") as f:
        for line in f:
            json.loads(line)
            count += 1

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    return {
        "valid_json": count == expected_count,
        "valid_hash": digest == expected_sha256,
        "records": count,
        "sha256": digest
    }

# Test isolé : aucun fichier Nexus réel n'est modifié.
with tempfile.TemporaryDirectory(prefix="nexus_v3_test_") as tmp:
    test_dir = Path(tmp)
    test_file = test_dir / "chunk_000001.jsonl"

    records = [
        {"title": "Physics", "year": 1900},
        {"title": "Mathematics", "year": 1920},
        {"title": "Biology", "year": 2020}
    ]

    with open(test_file, "wb") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False).encode("utf-8")
                + b"\n"
            )

    digest = hashlib.sha256(test_file.read_bytes()).hexdigest()
    result = validate_chunk(test_file, len(records), digest)

    print("=== TEST MOTEUR V3 ===")
    print("Notices :", result["records"])
    print("JSON valides :", result["valid_json"])
    print("SHA-256 vérifié :", result["valid_hash"])
    print("Test réussi :", result["valid_json"] and result["valid_hash"])

In [ ]:
import hashlib
import json
import os
from pathlib import Path

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_json_write(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def finalize_chunk(chunk_path, records_count):
    chunk_path = Path(chunk_path)

    # Contrôle des données avant validation
    with open(chunk_path, "rb") as f:
        count = 0
        for line in f:
            json.loads(line)
            count += 1

    if count != records_count:
        raise ValueError("Le nombre de notices ne correspond pas.")

    return {
        "file": chunk_path.name,
        "records": count,
        "bytes": chunk_path.stat().st_size,
        "sha256": sha256_file(chunk_path)
    }

# Simulation isolée
with tempfile.TemporaryDirectory(prefix="nexus_v3_chunks_") as tmp:
    folder = Path(tmp)
    results = []

    for index, records in enumerate([
        [{"title": "Physics"}, {"title": "Biology"}],
        [{"title": "Chemistry"}, {"title": "Astronomy"}]
    ], start=1):

        chunk = folder / f"chunk_{index:06d}.jsonl"

        with open(chunk, "wb") as f:
            for record in records:
                f.write(
                    json.dumps(record, ensure_ascii=False).encode("utf-8")
                    + b"\n"
                )

        metadata = finalize_chunk(chunk, len(records))
        results.append(metadata)

    test_manifest = {
        "version": 3,
        "status": "test",
        "completed_chunks": results
    }

    atomic_json_write(folder / "manifest_test.json", test_manifest)

    verified = json.loads(
        (folder / "manifest_test.json").read_text(encoding="utf-8")
    )

    print("=== GESTIONNAIRE V3 ===")
    print("Segments :", len(verified["completed_chunks"]))
    print("Notices :", sum(c["records"] for c in verified["completed_chunks"]))
    print("Taille cumulée :", sum(c["bytes"] for c in verified["completed_chunks"]))
    print("SHA-256 présents :", all(c["sha256"] for c in verified["completed_chunks"]))
    print("Manifeste relu :", len(verified["completed_chunks"]) == 2)
    print("TEST :", "OK" if len(verified["completed_chunks"]) == 2 and all(c["sha256"] for c in verified["completed_chunks"]) else "ÉCHEC")

In [ ]:
import os
import json
import hashlib
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

BASE.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

def atomic_json_write(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

def validate_chunk(path, expected_count):
    path = Path(path)
    count = 0

    with open(path, "rb") as f:
        for line in f:
            json.loads(line)
            count += 1

    if count != expected_count:
        raise ValueError(
            f"Nombre incorrect : {count} au lieu de {expected_count}"
        )

    return {
        "records": count,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path)
    }

def load_manifest():
    if not MANIFEST.exists():
        raise FileNotFoundError("Manifeste V3 absent.")

    data = json.loads(MANIFEST.read_text(encoding="utf-8"))

    if data.get("version") != 3:
        raise ValueError("Version du manifeste incompatible.")

    return data

def register_chunk(path, metadata):
    manifest = load_manifest()

    name = Path(path).name

    # Empêcher l'enregistrement en double
    if any(c["file"] == name for c in manifest["completed_chunks"]):
        raise ValueError(f"Segment déjà enregistré : {name}")

    entry = {
        "file": name,
        "records": metadata["records"],
        "bytes": metadata["bytes"],
        "sha256": metadata["sha256"]
    }

    manifest["completed_chunks"].append(entry)
    manifest["selected"] += metadata["records"]
    manifest["status"] = "running"

    atomic_json_write(MANIFEST, manifest)

    return entry

class NexusChunkWriter:

    def __init__(self, chunk_size=CHUNK_SIZE):
        self.chunk_size = chunk_size
        self.count = 0
        self.chunk_index = 0
        self.file = None
        self.path = None

    def open_chunk(self):
        self.chunk_index += 1

        self.path = CHUNKS / (
            f"chunk_{self.chunk_index:06d}.jsonl"
        )

        if self.path.exists():
            raise FileExistsError(
                f"Le segment existe déjà : {self.path}"
            )

        self.file = open(self.path, "xb")
        self.count = 0

    def write_record(self, record):
        if self.file is None:
            self.open_chunk()

        payload = (
            json.dumps(record, ensure_ascii=False)
            .encode("utf-8") + b"\n"
        )

        self.file.write(payload)
        self.count += 1

        if self.count >= self.chunk_size:
            self.finalize()

    def finalize(self):
        if self.file is None:
            return None

        self.file.flush()
        os.fsync(self.file.fileno())
        self.file.close()
        self.file = None

        metadata = validate_chunk(self.path, self.count)

        entry = register_chunk(self.path, metadata)

        print(
            "SEGMENT CERTIFIÉ :",
            entry["file"],
            "| notices :",
            entry["records"],
            "| SHA-256 :",
            entry["sha256"][:16]
        )

        self.path = None
        self.count = 0

        return entry

    def close(self):
        return self.finalize()

print("=== MOTEUR V3 INSTALLÉ ===")
print("Segment maximal :", CHUNK_SIZE, "notices")
print("Validation JSON : active")
print("SHA-256 : actif")
print("Manifeste atomique : actif")
print("Téléchargement : désactivé")
print("Neon : aucune connexion")

In [ ]:
import json
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

# Vérifier que notre espace de test est propre
manifest_before = json.loads(MANIFEST.read_text(encoding="utf-8"))

if manifest_before["completed_chunks"] or manifest_before["selected"] != 0:
    raise RuntimeError("Le manifeste n'est pas vierge. Test refusé.")

if list(CHUNKS.iterdir()):
    raise RuntimeError("Le dossier chunks n'est pas vide. Test refusé.")

test_records = [
    {"title": "Physics", "year": 1900},
    {"title": "Biology", "year": 1920},
    {"title": "Chemistry", "year": 2020}
]

writer = NexusChunkWriter(chunk_size=3)

try:
    for record in test_records:
        writer.write_record(record)

    # Le 3e enregistrement déclenche déjà la certification.
    result = load_manifest()

    assert len(result["completed_chunks"]) == 1
    assert result["selected"] == 3

    entry = result["completed_chunks"][0]
    chunk_path = CHUNKS / entry["file"]

    verified = validate_chunk(chunk_path, 3)

    assert verified["sha256"] == entry["sha256"]
    assert verified["bytes"] == entry["bytes"]

    print("=== TEST V3 RÉEL ===")
    print("Segment :", entry["file"])
    print("Notices :", verified["records"])
    print("SHA-256 concordant :", True)
    print("Manifeste :", True)
    print("TEST :", "OK")

finally:
    # Nettoyage du test et retour au manifeste initial
    if writer.file is not None:
        writer.file.close()

    for f in CHUNKS.iterdir():
        if f.is_file():
            f.unlink()

    atomic_json_write(MANIFEST, manifest_before)

print("\n=== ÉTAT APRÈS NETTOYAGE ===")
print("Segments :", len(load_manifest()["completed_chunks"]))
print("Notices :", load_manifest()["selected"])
print("Dossier de segments vide :", not list(CHUNKS.iterdir()))

In [ ]:
import json, tempfile
from pathlib import Path

def simulate_recovery_protocol():
    with tempfile.TemporaryDirectory(prefix="nexus_v3_recovery_") as tmp:
        root = Path(tmp)
        chunk = root / "chunk_000001.jsonl"

        records = [
            {"title": "Physics"},
            {"title": "Biology"}
        ]

        with open(chunk, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")

        metadata = finalize_chunk(chunk, 2)

        source = {
            "url": "https://example.org/dump.gz",
            "etag": "test-etag-001",
            "length": "1000"
        }

        checkpoint = {
            "version": 3,
            "source": source,
            "scan_lines": 5000,
            "completed_chunks": [{
                "file": chunk.name,
                "records": metadata["records"],
                "bytes": metadata["bytes"],
                "sha256": metadata["sha256"],
                "source_lines": 5000
            }]
        }

        cp = root / "checkpoint.json"
        atomic_json_write(cp, checkpoint)

        recovered = json.loads(cp.read_text(encoding="utf-8"))

        # 1. Contrôle de source
        source_ok = recovered["source"] == source

        # 2. Validation segment
        segment = root / recovered["completed_chunks"][0]["file"]
        segment_ok = (
            validate_chunk(segment, 2)["sha256"]
            == recovered["completed_chunks"][0]["sha256"]
        )

        # 3. Refus d'une autre source
        altered_source = dict(source)
        altered_source["etag"] = "different-etag"

        source_change_refused = altered_source != recovered["source"]

        # 4. Progression récupérée
        recovered_lines = recovered["scan_lines"]
        next_chunk = len(recovered["completed_chunks"]) + 1

        return {
            "source_ok": source_ok,
            "segment_ok": segment_ok,
            "source_change_refused": source_change_refused,
            "recovered_lines": recovered_lines,
            "next_chunk": next_chunk
        }

result = simulate_recovery_protocol()

print("=== TEST PROTOCOLE DE REPRISE ===")
print("Identité source :", result["source_ok"])
print("Segment certifié :", result["segment_ok"])
print("Source différente refusée :", result["source_change_refused"])
print("Lignes récupérées :", result["recovered_lines"])
print("Prochain segment :", result["next_chunk"])
print("TEST :", "OK" if all([
    result["source_ok"],
    result["segment_ok"],
    result["source_change_refused"],
    result["recovered_lines"] == 5000,
    result["next_chunk"] == 2
]) else "ÉCHEC")

In [ ]:
import json, hashlib, os
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

def v3_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_atomic_write(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def v3_load_manifest():
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def v3_next_chunk_number():
    manifest = v3_load_manifest()
    numbers = []

    for entry in manifest.get("completed_chunks", []):
        name = entry["file"]
        if name.startswith("chunk_") and name.endswith(".jsonl"):
            numbers.append(int(name[6:12]))

    return max(numbers, default=0) + 1

def v3_verify_all_chunks():
    manifest = v3_load_manifest()
    verified = 0

    for entry in manifest.get("completed_chunks", []):
        path = CHUNKS / entry["file"]

        if not path.exists():
            raise FileNotFoundError(entry["file"])

        if path.stat().st_size != entry["bytes"]:
            raise ValueError("Taille incohérente : " + entry["file"])

        if v3_hash(path) != entry["sha256"]:
            raise ValueError("SHA-256 incohérent : " + entry["file"])

        verified += 1

    return verified

print("=== GESTIONNAIRE INTÉGRÉ V3 ===")
print("Prochain segment :", v3_next_chunk_number())
print("Segments vérifiés :", v3_verify_all_chunks())
print("Manifeste :", MANIFEST)
print("Scan massif : non lancé")

In [ ]:
import gzip, json, tempfile, hashlib, os
from pathlib import Path

def run_v3_local_stream_test():
    with tempfile.TemporaryDirectory(prefix="nexus_v3_stream_") as tmp:
        root = Path(tmp)
        source = root / "mini_dump.gz"
        outdir = root / "chunks"
        outdir.mkdir()

        # Mini-dump artificiel de 20 notices
        records = [
            {"title": f"Physics Book {i}", "publish_date": str(1900 + i)}
            for i in range(20)
        ]

        with gzip.open(source, "wb") as gz:
            for i, record in enumerate(records):
                line = f"/works/OL{i}M\tedition\t".encode() + json.dumps(record).encode() + b"\n"
                gz.write(line)

        # Lecture du vrai fichier gzip
        parsed = 0
        selected = []

        with gzip.open(source, "rb") as gz:
            for line in gz:
                parts = line.rstrip(b"\r\n").split(b"\t")
                record = json.loads(parts[-1])
                parsed += 1
                selected.append(record)

        # Création et certification d'un segment local
        chunk = outdir / "chunk_000001.jsonl"
        with open(chunk, "wb") as f:
            for record in selected[:10]:
                f.write(json.dumps(record).encode() + b"\n")

        digest = hashlib.sha256(chunk.read_bytes()).hexdigest()

        # Checkpoint simulé après 10 lignes
        checkpoint = {
            "version": 3,
            "scan_lines": 10,
            "selected": 10,
            "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
            "completed_chunks": [{
                "file": chunk.name,
                "records": 10,
                "bytes": chunk.stat().st_size,
                "sha256": digest
            }]
        }

        cp = root / "checkpoint.json"
        with open(cp, "w", encoding="utf-8") as f:
            json.dump(checkpoint, f)

        recovered = json.loads(cp.read_text())

        # Vérification du segment et du checkpoint
        verified_hash = hashlib.sha256(chunk.read_bytes()).hexdigest()
        segment_ok = verified_hash == recovered["completed_chunks"][0]["sha256"]

        # Relecture du dump et saut des 10 lignes déjà checkpointées
        resumed_count = 0
        with gzip.open(source, "rb") as gz:
            for line_number, line in enumerate(gz, start=1):
                if line_number <= recovered["scan_lines"]:
                    continue
                json.loads(line.rstrip(b"\r\n").split(b"\t")[-1])
                resumed_count += 1

        return {
            "source_lines": parsed,
            "checkpoint_lines": recovered["scan_lines"],
            "segment_ok": segment_ok,
            "resumed_lines": resumed_count,
            "test_ok": parsed == 20 and segment_ok and resumed_count == 10
        }

test = run_v3_local_stream_test()

print("=== TEST FLUX GZIP V3 ===")
print("Lignes mini-dump :", test["source_lines"])
print("Lignes checkpointées :", test["checkpoint_lines"])
print("Segment vérifié :", test["segment_ok"])
print("Lignes après reprise simulée :", test["resumed_lines"])
print("TEST :", "OK" if test["test_ok"] else "ÉCHEC")

In [ ]:
import json, hashlib, tempfile
from pathlib import Path

def test_interruption_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_crash_v3_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        records = [
            {"id": i, "title": f"Science {i}"}
            for i in range(1, 31)
        ]

        chunk_size = 10

        # 1. Premier segment certifié
        first = records[:10]
        chunk1 = chunks / "chunk_000001.jsonl"

        with open(chunk1, "w", encoding="utf-8") as f:
            for r in first:
                f.write(json.dumps(r) + "\n")

        digest1 = hashlib.sha256(chunk1.read_bytes()).hexdigest()

        committed_line = 10

        # 2. Simuler une panne après 17 lignes
        interrupted_records = records[10:17]

        partial = chunks / "chunk_000002.partial"

        with open(partial, "w", encoding="utf-8") as f:
            for r in interrupted_records:
                f.write(json.dumps(r) + "\n")

        # 3. Panne : le segment partiel n'est pas certifié
        partial.unlink()

        # 4. Reprise depuis la dernière frontière certifiée
        resumed = records[committed_line:]

        chunk2 = chunks / "chunk_000002.jsonl"

        with open(chunk2, "w", encoding="utf-8") as f:
            for r in resumed[:chunk_size]:
                f.write(json.dumps(r) + "\n")

        digest2 = hashlib.sha256(chunk2.read_bytes()).hexdigest()

        # 5. Vérification des segments
        valid1 = hashlib.sha256(chunk1.read_bytes()).hexdigest() == digest1
        valid2 = hashlib.sha256(chunk2.read_bytes()).hexdigest() == digest2

        ids1 = [json.loads(x)["id"] for x in chunk1.read_text().splitlines()]
        ids2 = [json.loads(x)["id"] for x in chunk2.read_text().splitlines()]

        recovered = ids1 + ids2

        return {
            "first_segment_valid": valid1,
            "second_segment_valid": valid2,
            "partial_deleted": not partial.exists(),
            "recovery_starts_at": resumed[0]["id"],
            "recovered_records": len(recovered),
            "no_duplicates": len(recovered) == len(set(recovered)),
            "continuous_sequence": recovered == list(range(1, 21)),
            "test_ok": (
                valid1 and valid2
                and not partial.exists()
                and len(recovered) == 20
                and len(recovered) == len(set(recovered))
                and recovered == list(range(1, 21))
            )
        }

result = test_interruption_v3()

print("=== TEST INTERRUPTION V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

In [ ]:
import hashlib, json, tempfile
from pathlib import Path

def test_corruption_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_corruption_v3_") as tmp:
        root = Path(tmp)
        chunk = root / "chunk_000001.jsonl"

        records = [
            {"id": i, "title": f"Science {i}"}
            for i in range(1, 6)
        ]

        with open(chunk, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")

        original_hash = hashlib.sha256(chunk.read_bytes()).hexdigest()

        # Vérification initiale
        initial_ok = (
            hashlib.sha256(chunk.read_bytes()).hexdigest()
            == original_hash
        )

        # Simuler une modification malveillante ou accidentelle
        content = chunk.read_bytes()
        chunk.write_bytes(content.replace(b"Science 3", b"Science X"))

        corrupted_hash = hashlib.sha256(chunk.read_bytes()).hexdigest()

        corruption_detected = corrupted_hash != original_hash

        # Restaurer le fichier original
        chunk.write_bytes(content)

        restored_ok = (
            hashlib.sha256(chunk.read_bytes()).hexdigest()
            == original_hash
        )

        return {
            "initial_integrity": initial_ok,
            "corruption_detected": corruption_detected,
            "restoration_verified": restored_ok,
            "test_ok": initial_ok and corruption_detected and restored_ok
        }

result = test_corruption_v3()

print("=== TEST CORRUPTION V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

In [ ]:
import json, hashlib, tempfile
from pathlib import Path

def test_manifest_coherence_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_manifest_v3_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        # Création d'un segment
        chunk = chunks / "chunk_000001.jsonl"
        chunk.write_text(
            '{"id":1}\n{"id":2}\n',
            encoding="utf-8"
        )

        digest = hashlib.sha256(chunk.read_bytes()).hexdigest()

        manifest = {
            "source_id": "source_A",
            "completed_chunks": [{
                "file": chunk.name,
                "sha256": digest,
                "records": 2
            }]
        }

        # 1. Segment présent et hash valide
        entry = manifest["completed_chunks"][0]
        valid_initial = (
            chunk.exists()
            and hashlib.sha256(chunk.read_bytes()).hexdigest()
            == entry["sha256"]
        )

        # 2. Simuler corruption
        chunk.write_text('{"id":999}\n', encoding="utf-8")

        corruption_detected = (
            hashlib.sha256(chunk.read_bytes()).hexdigest()
            != entry["sha256"]
        )

        # 3. Simuler fichier absent
        chunk.unlink()
        missing_detected = not chunk.exists()

        # 4. Identité de source
        source_A_accepted = manifest["source_id"] == "source_A"
        source_B_rejected = manifest["source_id"] != "source_B"

        return {
            "initial_segment_valid": valid_initial,
            "corruption_detected": corruption_detected,
            "missing_segment_detected": missing_detected,
            "correct_source_accepted": source_A_accepted,
            "different_source_rejected": source_B_rejected,
            "test_ok": all([
                valid_initial,
                corruption_detected,
                missing_detected,
                source_A_accepted,
                source_B_rejected
            ])
        }

result = test_manifest_coherence_v3()

print("=== TEST COHÉRENCE MANIFESTE V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

In [ ]:
import os, json, gzip, hashlib, time, re
from pathlib import Path
from urllib.request import Request, urlopen

# =========================================================
# NEXUS SCIENCE — SCANNER V3 (INSTALLATION SANS SCAN)
# =========================================================

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
CHUNK_SIZE = 100_000

ROOT.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

def atomic_write_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def new_manifest():
    return {
        "version": 3,
        "project": "Bibliothèque Scientifique — Nexus Science",
        "source": SOURCE,
        "source_identity": None,
        "status": "prepared",
        "scan_lines": 0,
        "valid_json": 0,
        "selected": 0,
        "invalid_json": 0,
        "out_of_period": 0,
        "completed_chunks": [],
        "updated_at_utc": None
    }

if not MANIFEST.exists():
    atomic_write_json(MANIFEST, new_manifest())

def load_manifest_v3():
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def verify_chunks_v3():
    manifest = load_manifest_v3()
    results = []

    for item in manifest["completed_chunks"]:
        path = CHUNKS / item["file"]

        if not path.exists():
            results.append((item["file"], "ABSENT"))
            continue

        actual_hash = sha256_file(path)
        if actual_hash != item["sha256"]:
            results.append((item["file"], "CORRUPTION"))
            continue

        count = 0
        invalid = 0
        with open(path, "rb") as f:
            for line in f:
                count += 1
                try:
                    json.loads(line)
                except Exception:
                    invalid += 1

        if count != item["records"] or invalid:
            results.append((item["file"], "JSONL_INVALIDE"))
        else:
            results.append((item["file"], "OK"))

    return results

def next_chunk_v3():
    manifest = load_manifest_v3()
    numbers = []

    for item in manifest["completed_chunks"]:
        match = re.search(r"chunk_(\d+)", item["file"])
        if match:
            numbers.append(int(match.group(1)))

    return max(numbers, default=0) + 1

# =========================================================
# CONTRÔLE INITIAL — AUCUN SCAN
# =========================================================

manifest = load_manifest_v3()

print("=== INSTALLATION MOTEUR NEXUS V3 ===")
print("Répertoire :", ROOT)
print("Manifeste :", MANIFEST)
print("Checkpoint :", CHECKPOINT)
print("Segments enregistrés :", len(manifest["completed_chunks"]))
print("Prochain segment :", next_chunk_v3())
print("Taille segment :", CHUNK_SIZE)
print("Scan massif : NON")
print("Installation : OK")

In [ ]:
import os, json, gzip, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

CHUNK_SIZE = 100_000
SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

# Filtrage configurable
SCIENCE_KEYWORDS = [
    "science", "scientific", "physics", "mathematics", "math",
    "chemistry", "biology", "astronomy", "astrophysics",
    "engineering", "medicine", "medical", "geology",
    "statistics", "computer science", "algorithm",
    "research", "technology", "ecology", "genetics",
    "neuroscience", "quantum", "anatomy", "botany",
    "zoology", "microbiology", "biochemistry", "thermodynamics",
    "électronique", "physique", "mathématique", "chimie",
    "biologie", "médecine", "astronomie", "géologie",
    "scientifique", "recherche", "technologie", "ingénierie",
    "estatística", "matemática", "ciência", "fisica",
    "química", "biologia", "medicina", "engenharia"
]

EXCLUSIONS = [
    "fiction", "romance", "poetry", "comic", "comics",
    "novel", "fantasy", "thriller"
]

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def scientific_record(record):
    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])
    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    if any(word in text for word in EXCLUSIONS):
        return False

    return any(word in text for word in SCIENCE_KEYWORDS)

def extract_year(record):
    for key in ("publish_date", "date"):
        value = record.get(key)
        if value is not None:
            match = re.search(r"\b(1[7-9]\d{2}|20\d{2})\b", str(value))
            if match:
                return int(match.group(1))
    return None

def in_period(record):
    year = extract_year(record)
    return year is not None and 1720 <= year <= 2026

def validate_jsonl(path, expected_count):
    count = 0
    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1
    return count == expected_count

def certify_chunk(path, expected_count):
    if not validate_jsonl(path, expected_count):
        raise RuntimeError("Validation JSONL échouée")

    return {
        "records": expected_count,
        "bytes": path.stat().st_size,
        "sha256": hash_file(path)
    }

def committed_source_line(manifest):
    if not manifest["completed_chunks"]:
        return 0
    return manifest["completed_chunks"][-1]["source_line_end"]

def verify_manifest_chunks():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

    for item in manifest["completed_chunks"]:
        path = CHUNKS / item["file"]

        if not path.exists():
            raise RuntimeError(f"Segment absent : {item['file']}")

        if hash_file(path) != item["sha256"]:
            raise RuntimeError(f"Hash incorrect : {item['file']}")

        if not validate_jsonl(path, item["records"]):
            raise RuntimeError(f"JSONL invalide : {item['file']}")

    return manifest

class NexusV3Engine:
    def __init__(self):
        self.manifest = verify_manifest_chunks()
        self.next_number = max(
            [
                int(re.search(r"chunk_(\d+)", x["file"]).group(1))
                for x in self.manifest["completed_chunks"]
            ],
            default=0
        ) + 1

    def commit_segment(self, records, source_line_end, counters):
        if not records:
            return None

        final_name = f"chunk_{self.next_number:06d}.jsonl"
        final_path = CHUNKS / final_name
        temp_path = CHUNKS / (final_name + ".partial")

        # Écriture non certifiée
        with open(temp_path, "wb") as f:
            for record in records:
                f.write(
                    json.dumps(record, ensure_ascii=False).encode("utf-8")
                    + b"\n"
                )
            f.flush()
            os.fsync(f.fileno())

        # Validation avant certification
        info = certify_chunk(temp_path, len(records))

        # Renommage du segment validé
        os.replace(temp_path, final_path)

        entry = {
            "file": final_name,
            "records": info["records"],
            "bytes": info["bytes"],
            "sha256": info["sha256"],
            "source_line_end": source_line_end,
            "committed_at_utc": utc_now()
        }

        # Manifeste mis à jour uniquement après certification
        self.manifest["completed_chunks"].append(entry)
        self.manifest["scan_lines"] = source_line_end
        self.manifest["selected"] += len(records)
        self.manifest["updated_at_utc"] = utc_now()
        self.manifest["status"] = "running"

        atomic_json(MANIFEST, self.manifest)

        # Checkpoint dérivé du manifeste certifié
        checkpoint = {
            "version": 3,
            "source": SOURCE,
            "scan_lines": source_line_end,
            "selected": self.manifest["selected"],
            "chunk_number": self.next_number,
            "source_identity": self.manifest.get("source_identity"),
            "completed_chunks": self.manifest["completed_chunks"]
        }

        atomic_json(CHECKPOINT, checkpoint)

        self.next_number += 1
        return entry

print("=== MOTEUR DE TRAITEMENT V3 INSTALLÉ ===")
print("Filtrage scientifique : prêt")
print("Validation JSONL : prête")
print("Certification SHA-256 : prête")
print("Checkpoint post-certification : prêt")
print("Reprise par frontière certifiée : préparée")
print("Scan massif : NON")

In [ ]:
import json, hashlib, os, tempfile
from pathlib import Path

def test_transaction_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_tx_v3_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        manifest_path = root / "manifest.json"
        checkpoint_path = root / "checkpoint.json"

        source_identity = {
            "url": "mini_source.gz",
            "sha256": "SIMULATION_SOURCE_123"
        }

        manifest = {
            "version": 3,
            "source_identity": source_identity,
            "scan_lines": 0,
            "selected": 0,
            "completed_chunks": []
        }

        records = [{"id": i} for i in range(1, 11)]
        source_line_end = 1000

        # Écriture temporaire
        partial = chunks / "chunk_000001.jsonl.partial"
        final = chunks / "chunk_000001.jsonl"

        with open(partial, "wb") as f:
            for r in records:
                f.write(json.dumps(r).encode() + b"\n")
            f.flush()
            os.fsync(f.fileno())

        # Validation JSONL
        parsed = []
        with open(partial, "rb") as f:
            for line in f:
                parsed.append(json.loads(line))

        valid_jsonl = len(parsed) == 10

        digest = hashlib.sha256(partial.read_bytes()).hexdigest()
        size = partial.stat().st_size

        # Certification
        os.replace(partial, final)

        entry = {
            "file": final.name,
            "records": len(parsed),
            "bytes": size,
            "sha256": digest,
            "source_line_end": source_line_end
        }

        manifest["completed_chunks"].append(entry)
        manifest["scan_lines"] = source_line_end
        manifest["selected"] = len(parsed)

        # Manifeste atomique
        tmp_manifest = root / "manifest.tmp"
        tmp_manifest.write_text(
            json.dumps(manifest, indent=2),
            encoding="utf-8"
        )
        os.replace(tmp_manifest, manifest_path)

        # Checkpoint dérivé du manifeste
        checkpoint = {
            "version": 3,
            "source_identity": source_identity,
            "scan_lines": manifest["scan_lines"],
            "selected": manifest["selected"],
            "completed_chunks": manifest["completed_chunks"]
        }

        tmp_checkpoint = root / "checkpoint.tmp"
        tmp_checkpoint.write_text(
            json.dumps(checkpoint, indent=2),
            encoding="utf-8"
        )
        os.replace(tmp_checkpoint, checkpoint_path)

        # Relecture
        recovered_manifest = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )
        recovered_checkpoint = json.loads(
            checkpoint_path.read_text(encoding="utf-8")
        )

        recovered_file = chunks / entry["file"]
        recovered_hash = hashlib.sha256(
            recovered_file.read_bytes()
        ).hexdigest()

        source_match = (
            recovered_manifest["source_identity"]
            == source_identity
            == recovered_checkpoint["source_identity"]
        )

        boundary_match = (
            recovered_manifest["scan_lines"]
            == recovered_checkpoint["scan_lines"]
            == source_line_end
        )

        hash_match = recovered_hash == entry["sha256"]

        records_match = (
            recovered_manifest["selected"]
            == recovered_checkpoint["selected"]
            == 10
        )

        return {
            "jsonl_valid": valid_jsonl,
            "segment_certified": hash_match,
            "source_identity_match": source_match,
            "boundary_match": boundary_match,
            "records_match": records_match,
            "checkpoint_recovered": checkpoint_path.exists(),
            "test_ok": all([
                valid_jsonl,
                hash_match,
                source_match,
                boundary_match,
                records_match,
                checkpoint_path.exists()
            ])
        }

result = test_transaction_v3()

print("=== TEST TRANSACTION INTÉGRÉE V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

In [ ]:
import json, hashlib, tempfile, os
from pathlib import Path

def test_crash_windows_v3():
    results = {}

    with tempfile.TemporaryDirectory(prefix="nexus_crash_windows_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        manifest_path = root / "manifest.json"
        checkpoint_path = root / "checkpoint.json"

        def save_atomic(path, data):
            temp = path.with_suffix(".tmp")
            temp.write_text(json.dumps(data), encoding="utf-8")
            os.replace(temp, path)

        initial = {
            "scan_lines": 0,
            "selected": 0,
            "completed_chunks": []
        }
        save_atomic(manifest_path, initial)
        save_atomic(checkpoint_path, initial)

        # -----------------------------------------------
        # SCÉNARIO 1 : segment renommé, manifeste absent
        # -----------------------------------------------
        orphan = chunks / "chunk_000001.jsonl"
        orphan.write_text('{"id":1}\n{"id":2}\n', encoding="utf-8")

        orphan_hash = hashlib.sha256(orphan.read_bytes()).hexdigest()
        orphan_valid = orphan_hash == hashlib.sha256(
            orphan.read_bytes()
        ).hexdigest()

        manifest1 = json.loads(manifest_path.read_text())
        orphan_not_registered = (
            len(manifest1["completed_chunks"]) == 0
        )

        results["orphan_detected_as_unregistered"] = (
            orphan.exists() and orphan_not_registered and orphan_valid
        )

        # Ne pas adopter automatiquement l'orphelin.
        orphan.unlink()

        # -----------------------------------------------
        # SCÉNARIO 2 : manifeste avancé, checkpoint ancien
        # -----------------------------------------------
        chunk2 = chunks / "chunk_000001.jsonl"
        chunk2.write_text('{"id":1}\n{"id":2}\n', encoding="utf-8")

        digest2 = hashlib.sha256(chunk2.read_bytes()).hexdigest()

        committed = {
            "scan_lines": 500,
            "selected": 2,
            "completed_chunks": [{
                "file": chunk2.name,
                "records": 2,
                "sha256": digest2,
                "source_line_end": 500
            }]
        }

        save_atomic(manifest_path, committed)

        # Le checkpoint reste volontairement à zéro.
        stale_checkpoint = {
            "scan_lines": 0,
            "selected": 0,
            "completed_chunks": []
        }
        save_atomic(checkpoint_path, stale_checkpoint)

        # Le manifeste est la référence maîtresse.
        recovered = json.loads(manifest_path.read_text())
        reconstructed = {
            "scan_lines": recovered["scan_lines"],
            "selected": recovered["selected"],
            "completed_chunks": recovered["completed_chunks"]
        }

        results["manifest_authoritative"] = (
            reconstructed["scan_lines"] == 500
            and reconstructed["selected"] == 2
        )

        results["stale_checkpoint_reconstructable"] = (
            reconstructed == committed
        )

        # -----------------------------------------------
        # SCÉNARIO 3 : segment certifié corrompu
        # -----------------------------------------------
        chunk2.write_text('{"id":999}\n', encoding="utf-8")

        corruption_detected = (
            hashlib.sha256(chunk2.read_bytes()).hexdigest()
            != committed["completed_chunks"][0]["sha256"]
        )

        results["corruption_blocks_recovery"] = corruption_detected

        results["test_ok"] = all([
            results["orphan_detected_as_unregistered"],
            results["manifest_authoritative"],
            results["stale_checkpoint_reconstructable"],
            results["corruption_blocks_recovery"]
        ])

    return results

result = test_crash_windows_v3()

print("=== TEST FENÊTRES DE PANNE V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

In [ ]:
import json, hashlib, os, re
from pathlib import Path

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

def v3_atomic_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def v3_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_validate_jsonl(path, expected):
    count = 0
    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1
    return count == expected

def v3_recovery_controller():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    registered = set()
    verified = 0

    # Vérifier tous les segments enregistrés
    for item in manifest["completed_chunks"]:
        filename = item["file"]
        registered.add(filename)
        path = CHUNKS / filename

        if not path.exists():
            raise RuntimeError(f"SEGMENT ABSENT : {filename}")

        if v3_hash(path) != item["sha256"]:
            raise RuntimeError(f"CORRUPTION HASH : {filename}")

        if not v3_validate_jsonl(path, item["records"]):
            raise RuntimeError(f"JSONL INVALIDE : {filename}")

        if "source_line_end" not in item:
            raise RuntimeError(
                f"FRONTIÈRE SOURCE ABSENTE : {filename}"
            )

        verified += 1

    # Repérer les fichiers non enregistrés
    orphans = []
    for path in CHUNKS.glob("chunk_*.jsonl"):
        if path.name not in registered:
            orphans.append(path.name)

    # Ne jamais adopter automatiquement les orphelins
    if orphans:
        return {
            "status": "ORPHANS_BLOCK_RECOVERY",
            "verified": verified,
            "orphans": sorted(orphans),
            "checkpoint_rebuilt": False
        }

    # Reconstruction du checkpoint depuis le manifeste
    checkpoint = {
        "version": 3,
        "source": manifest.get("source"),
        "source_identity": manifest.get("source_identity"),
        "scan_lines": manifest["scan_lines"],
        "selected": manifest["selected"],
        "completed_chunks": manifest["completed_chunks"]
    }

    v3_atomic_json(CHECKPOINT, checkpoint)

    return {
        "status": "RECOVERY_READY",
        "verified": verified,
        "orphans": [],
        "checkpoint_rebuilt": True,
        "scan_lines": manifest["scan_lines"],
        "selected": manifest["selected"]
    }

print("=== CONTRÔLEUR DE RÉCUPÉRATION V3 ===")
result = v3_recovery_controller()

for key, value in result.items():
    print(f"{key} : {value}")

print("SCAN MASSIF : NON")

In [ ]:
import os, json, gzip, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

# ============================================================
# NEXUS SCIENCE — SCANNER V3 INTÉGRÉ
# INSTALLATION SEULE : AUCUN SCAN
# ============================================================

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

CHUNK_SIZE = 100_000
MIN_YEAR = 1720
MAX_YEAR = 2026

RUN_SCAN = False   # VERROU DE SÉCURITÉ

ROOT.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# UTILITAIRES
# ------------------------------------------------------------

def v3_now():
    return datetime.now(timezone.utc).isoformat()

def v3_atomic_json(path, data):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")

    with open(temp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, path)

def v3_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_manifest():
    if not MANIFEST.exists():
        raise RuntimeError("Manifeste absent")
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def v3_validate(path, expected_records):
    count = 0

    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1

    return count == expected_records

# ------------------------------------------------------------
# IDENTITÉ SOURCE
# ------------------------------------------------------------

def v3_source_identity():
    req = Request(
        SOURCE,
        headers={
            "User-Agent": "NexusScienceScanner/3.0",
            "Accept-Encoding": "identity"
        },
        method="HEAD"
    )

    with urlopen(req, timeout=30) as response:
        headers = response.headers

        return {
            "url": SOURCE,
            "etag": headers.get("ETag"),
            "last_modified": headers.get("Last-Modified"),
            "content_length": headers.get("Content-Length")
        }

# ------------------------------------------------------------
# FILTRAGE
# ------------------------------------------------------------

SCIENCE_KEYWORDS = [
    "science", "scientific", "physics", "mathematics",
    "math", "chemistry", "biology", "astronomy",
    "astrophysics", "engineering", "medicine",
    "medical", "geology", "statistics", "computer science",
    "algorithm", "research", "technology", "ecology",
    "genetics", "neuroscience", "quantum", "anatomy",
    "botany", "zoology", "microbiology", "biochemistry",
    "thermodynamics", "électronique", "physique",
    "mathématique", "chimie", "biologie", "médecine",
    "astronomie", "géologie", "scientifique", "recherche",
    "technologie", "ingénierie", "matemática", "ciência",
    "química", "medicina", "engenharia"
]

EXCLUSIONS = [
    "romance", "poetry", "comic", "comics",
    "thriller", "fantasy"
]

def v3_is_scientific(record):
    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])

    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    if any(term in text for term in EXCLUSIONS):
        return False

    return any(term in text for term in SCIENCE_KEYWORDS)

def v3_year(record):
    for key in ("publish_date", "date"):
        value = record.get(key)

        if value is not None:
            match = re.search(
                r"\b(1[7-9]\d{2}|20\d{2})\b",
                str(value)
            )
            if match:
                return int(match.group(1))

    return None

def v3_in_period(record):
    year = v3_year(record)
    return year is not None and MIN_YEAR <= year <= MAX_YEAR

# ------------------------------------------------------------
# VÉRIFICATION DES SEGMENTS ENREGISTRÉS
# ------------------------------------------------------------

def v3_verify_registered(manifest):
    registered = set()

    for item in manifest["completed_chunks"]:
        name = item["file"]
        registered.add(name)
        path = CHUNKS / name

        if not path.exists():
            raise RuntimeError("SEGMENT ABSENT : " + name)

        if v3_sha256(path) != item["sha256"]:
            raise RuntimeError("HASH INCORRECT : " + name)

        if not v3_validate(path, item["records"]):
            raise RuntimeError("JSONL INVALIDE : " + name)

        if "source_line_end" not in item:
            raise RuntimeError(
                "FRONTIÈRE SOURCE MANQUANTE : " + name
            )

    # Orphelins : refus de démarrer sans intervention
    orphans = [
        p.name for p in CHUNKS.glob("chunk_*.jsonl")
        if p.name not in registered
    ]

    partials = [
        p.name for p in CHUNKS.glob("*.partial")
    ]

    if orphans or partials:
        raise RuntimeError(
            f"Récupération bloquée : orphelins={orphans}, "
            f"partiels={partials}"
        )

# ------------------------------------------------------------
# MOTEUR
# ------------------------------------------------------------

class NexusScannerV3:

    def __init__(self):
        self.manifest = v3_manifest()
        v3_verify_registered(self.manifest)

        self.next_number = max(
            [
                int(re.search(r"chunk_(\d+)", x["file"]).group(1))
                for x in self.manifest["completed_chunks"]
            ],
            default=0
        ) + 1

        self.identity = self.manifest.get("source_identity")
        self.resume_line = self.manifest.get("scan_lines", 0)

    def verify_source(self, identity):
        previous = self.manifest.get("source_identity")

        if previous is not None and previous != identity:
            raise RuntimeError(
                "SOURCE DIFFÉRENTE : reprise refusée"
            )

        if previous is None and self.resume_line != 0:
            raise RuntimeError(
                "Checkpoint sans identité source : reprise refusée"
            )

        self.identity = identity

    def commit(self, records, source_line_end, counters):
        if not records:
            # Une frontière sans segment ne sera pas commitée
            return

        name = f"chunk_{self.next_number:06d}.jsonl"
        temp = CHUNKS / (name + ".partial")
        final = CHUNKS / name

        # Écriture du segment temporaire
        with open(temp, "wb") as f:
            for record in records:
                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ).encode("utf-8") + b"\n"
                )

            f.flush()
            os.fsync(f.fileno())

        # Validation avant certification
        if not v3_validate(temp, len(records)):
            raise RuntimeError("Validation JSONL échouée")

        digest = v3_sha256(temp)
        size = temp.stat().st_size

        # Certification et renommage
        os.replace(temp, final)

        entry = {
            "file": name,
            "records": len(records),
            "bytes": size,
            "sha256": digest,
            "source_line_end": source_line_end,
            "committed_at_utc": v3_now()
        }

        # Le manifeste devient la référence maîtresse
        updated = dict(self.manifest)
        updated["completed_chunks"] = (
            self.manifest["completed_chunks"] + [entry]
        )

        updated["scan_lines"] = source_line_end
        updated["selected"] = (
            self.manifest["selected"] + len(records)
        )

        for key, value in counters.items():
            updated[key] = self.manifest.get(key, 0) + value

        updated["source_identity"] = self.identity
        updated["status"] = "running"
        updated["updated_at_utc"] = v3_now()

        v3_atomic_json(MANIFEST, updated)

        # Checkpoint dérivé du manifeste
        checkpoint = {
            "version": 3,
            "source": SOURCE,
            "source_identity": updated["source_identity"],
            "scan_lines": updated["scan_lines"],
            "selected": updated["selected"],
            "completed_chunks": updated["completed_chunks"]
        }

        v3_atomic_json(CHECKPOINT, checkpoint)

        self.manifest = updated
        self.next_number += 1

    def scan(self):
        raise NotImplementedError(
            "Le flux réel nécessite le raccordement final "
            "et son test avant activation."
        )

# ------------------------------------------------------------
# INSTALLATION ET CONTRÔLE
# ------------------------------------------------------------

scanner = NexusScannerV3()

print("=== NEXUS SCIENCE — SCANNER V3 ===")
print("Moteur intégré : installé")
print("Source : configurée")
print("Filtrage : configuré")
print("Certification : configurée")
print("Gestionnaire de récupération : raccordé")
print("Prochain segment :", scanner.next_number)
print("Frontière de reprise :", scanner.resume_line)
print("RUN_SCAN :", RUN_SCAN)
print("SCAN MASSIF : NON")
print("INSTALLATION : OK")

In [ ]:
import gzip, json, hashlib, time, os
from pathlib import Path
from urllib.request import Request, urlopen

# =========================================================
# NEXUS V3 — TEST LIMITÉ DU FLUX OPEN LIBRARY
# =========================================================

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
LIMIT_LINES = 20_000

TEST_ROOT = Path("/kaggle/working/nexus_science_v3/integration_test")
TEST_ROOT.mkdir(parents=True, exist_ok=True)

TEST_CHUNK = TEST_ROOT / "test_chunk.jsonl"

def test_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def test_scientific_filter(record):
    keywords = [
        "science", "scientific", "physics", "mathematics",
        "math", "chemistry", "biology", "astronomy",
        "engineering", "medicine", "medical", "geology",
        "statistics", "computer science", "research",
        "technology", "ecology", "genetics", "neuroscience",
        "quantum", "anatomy", "botany", "zoology",
        "microbiology", "biochemistry", "thermodynamics",
        "physique", "mathématique", "chimie", "biologie",
        "médecine", "astronomie", "géologie", "scientifique",
        "recherche", "technologie", "ingénierie",
        "matemática", "ciência", "química", "medicina",
        "engenharia"
    ]

    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])

    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    return any(k in text for k in keywords)

def test_period_filter(record):
    value = record.get("publish_date", record.get("date", ""))

    import re
    match = re.search(
        r"\b(1[7-9]\d{2}|20\d{2})\b",
        str(value)
    )

    if not match:
        return False

    year = int(match.group(1))
    return 1720 <= year <= 2026

# ---------------------------------------------------------
# 1. IDENTITÉ SOURCE
# ---------------------------------------------------------

identity = None

try:
    req = Request(
        SOURCE,
        method="HEAD",
        headers={"User-Agent": "NexusScienceScanner/3.0"}
    )

    with urlopen(req, timeout=30) as response:
        identity = {
            "url": SOURCE,
            "etag": response.headers.get("ETag"),
            "last_modified": response.headers.get("Last-Modified"),
            "content_length": response.headers.get("Content-Length")
        }

    print("IDENTITÉ SOURCE : OK")
    print("ETag :", identity["etag"])
    print("Last-Modified :", identity["last_modified"])
    print("Content-Length :", identity["content_length"])

except Exception as e:
    raise RuntimeError(
        f"Échec de vérification de la source : {e}"
    )

# ---------------------------------------------------------
# 2. LECTURE LIMITÉE
# ---------------------------------------------------------

physical_lines = 0
valid_json = 0
invalid_json = 0
in_period = 0
selected = 0

records_buffer = []
start_time = time.time()

try:
    req = Request(
        SOURCE,
        headers={"User-Agent": "NexusScienceScanner/3.0"}
    )

    with urlopen(req, timeout=60) as response:
        with gzip.GzipFile(fileobj=response) as gz:

            while physical_lines < LIMIT_LINES:
                raw = gz.readline()

                if not raw:
                    break

                physical_lines += 1

                try:
                    parts = raw.rstrip(b"\r\n").split(b"\t")
                    record = json.loads(parts[-1])

                    if not isinstance(record, dict):
                        invalid_json += 1
                        continue

                    valid_json += 1

                except Exception:
                    invalid_json += 1
                    continue

                if not test_period_filter(record):
                    continue

                in_period += 1

                if test_scientific_filter(record):
                    records_buffer.append(record)
                    selected += 1

except Exception as e:
    raise RuntimeError(f"Échec du flux limité : {e}")

# ---------------------------------------------------------
# 3. ÉCRITURE DU SEGMENT DE TEST
# ---------------------------------------------------------

with open(TEST_CHUNK, "wb") as f:
    for record in records_buffer:
        f.write(
            json.dumps(record, ensure_ascii=False).encode("utf-8")
            + b"\n"
        )

    f.flush()
    os.fsync(f.fileno())

# ---------------------------------------------------------
# 4. VALIDATION
# ---------------------------------------------------------

validated = 0
jsonl_errors = 0

with open(TEST_CHUNK, "rb") as f:
    for line in f:
        try:
            json.loads(line)
            validated += 1
        except Exception:
            jsonl_errors += 1

digest = test_hash(TEST_CHUNK)
size_bytes = TEST_CHUNK.stat().st_size

test_ok = (
    physical_lines > 0
    and valid_json + invalid_json == physical_lines
    and validated == selected
    and jsonl_errors == 0
)

print()
print("=== TEST FLUX RÉEL V3 ===")
print("Lignes physiques :", physical_lines)
print("JSON valides :", valid_json)
print("JSON invalides :", invalid_json)
print("Dans la période :", in_period)
print("Notices sélectionnées :", selected)
print("Notices validées :", validated)
print("Erreurs JSONL :", jsonl_errors)
print("Taille segment (octets) :", size_bytes)
print("SHA-256 :", digest)
print("Temps (secondes) :", round(time.time() - start_time, 2))
print("TEST :", "OK" if test_ok else "ÉCHEC")
print("SCAN MASSIF : NON")

In [ ]:
import os, json, gzip, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

# ============================================================
# NEXUS SCIENCE V3 — MOTEUR SEGMENTÉ ET REPRENABLE
# ============================================================

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

CHUNK_SIZE = 100_000
MIN_YEAR = 1720
MAX_YEAR = 2026

RUN_SCAN = False

ROOT.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# UTILITAIRES
# ------------------------------------------------------------

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def atomic_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def load_manifest():
    if not MANIFEST.exists():
        raise RuntimeError("Manifeste absent")
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def source_identity():
    req = Request(
        SOURCE,
        method="HEAD",
        headers={"User-Agent": "NexusScienceScanner/3.0"}
    )

    with urlopen(req, timeout=30) as response:
        return {
            "url": SOURCE,
            "etag": response.headers.get("ETag"),
            "last_modified": response.headers.get("Last-Modified"),
            "content_length": response.headers.get("Content-Length")
        }

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

def validate_jsonl(path, expected):
    count = 0

    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1

    return count == expected

def verify_manifest_segments(manifest):
    registered = set()

    for entry in manifest["completed_chunks"]:
        filename = entry["file"]
        registered.add(filename)
        path = CHUNKS / filename

        if not path.exists():
            raise RuntimeError(f"SEGMENT ABSENT : {filename}")

        if sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"HASH INCORRECT : {filename}")

        if not validate_jsonl(path, entry["records"]):
            raise RuntimeError(f"JSONL INVALIDE : {filename}")

        if "source_line_end" not in entry:
            raise RuntimeError(
                f"FRONTIÈRE SOURCE MANQUANTE : {filename}"
            )

    # Refus d'adopter automatiquement les segments orphelins
    orphans = [
        p.name for p in CHUNKS.glob("chunk_*.jsonl")
        if p.name not in registered
    ]

    partials = list(CHUNKS.glob("*.partial"))

    if orphans or partials:
        raise RuntimeError(
            f"Récupération bloquée : orphelins={orphans}, "
            f"partiels={[p.name for p in partials]}"
        )

# ------------------------------------------------------------
# FILTRAGE
# ------------------------------------------------------------

KEYWORDS = [
    "science", "scientific", "physics", "mathematics",
    "math", "chemistry", "biology", "astronomy",
    "astrophysics", "engineering", "medicine", "medical",
    "geology", "statistics", "computer science", "algorithm",
    "research", "technology", "ecology", "genetics",
    "neuroscience", "quantum", "anatomy", "botany",
    "zoology", "microbiology", "biochemistry",
    "thermodynamics", "physique", "mathématique", "chimie",
    "biologie", "médecine", "astronomie", "géologie",
    "scientifique", "recherche", "technologie", "ingénierie",
    "matemática", "ciência", "química", "medicina",
    "engenharia"
]

EXCLUSIONS = [
    "romance", "poetry", "comic", "comics", "thriller"
]

def is_scientific(record):
    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])

    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    if any(word in text for word in EXCLUSIONS):
        return False

    return any(word in text for word in KEYWORDS)

def extract_year(record):
    for key in ("publish_date", "date"):
        value = record.get(key)

        if value is not None:
            match = re.search(
                r"\b(1[7-9]\d{2}|20\d{2})\b",
                str(value)
            )
            if match:
                return int(match.group(1))

    return None

def in_period(record):
    year = extract_year(record)
    return year is not None and MIN_YEAR <= year <= MAX_YEAR

# ------------------------------------------------------------
# MANIFESTE ET REPRISE
# ------------------------------------------------------------

def initialize_manifest_if_needed():
    if not MANIFEST.exists():
        atomic_json(MANIFEST, {
            "version": 3,
            "project": "Bibliothèque Scientifique — Nexus Science",
            "source": SOURCE,
            "source_identity": None,
            "status": "prepared",
            "scan_lines": 0,
            "valid_json": 0,
            "selected": 0,
            "invalid_json": 0,
            "out_of_period": 0,
            "completed_chunks": [],
            "updated_at_utc": None
        })

initialize_manifest_if_needed()

class NexusScannerV3:

    def __init__(self):
        self.manifest = load_manifest()
        verify_manifest_segments(self.manifest)

        self.next_number = max(
            [
                int(re.search(
                    r"chunk_(\d+)",
                    entry["file"]
                ).group(1))
                for entry in self.manifest["completed_chunks"]
            ],
            default=0
        ) + 1

        self.committed_line = self.manifest["scan_lines"]
        self.buffer = []
        self.interval_counters = {
            "valid_json": 0,
            "invalid_json": 0,
            "out_of_period": 0
        }

    def verify_source(self, identity):
        previous = self.manifest.get("source_identity")

        if previous is not None and previous != identity:
            raise RuntimeError(
                "Identité de source différente : reprise refusée."
            )

        if previous is None and self.committed_line != 0:
            raise RuntimeError(
                "Checkpoint sans identité source : reprise refusée."
            )

        self.identity = identity

    def commit(self, source_line_end):
        if not self.buffer:
            return

        name = f"chunk_{self.next_number:06d}.jsonl"
        temp = CHUNKS / (name + ".partial")
        final = CHUNKS / name

        if final.exists():
            raise RuntimeError(
                "Refus d'écraser un segment existant."
            )

        # Écriture temporaire
        with open(temp, "wb") as f:
            for record in self.buffer:
                f.write(
                    json.dumps(
                        record, ensure_ascii=False
                    ).encode("utf-8") + b"\n"
                )

            f.flush()
            os.fsync(f.fileno())

        # Validation
        if not validate_jsonl(temp, len(self.buffer)):
            raise RuntimeError("Validation du segment échouée")

        digest = sha256_file(temp)
        size = temp.stat().st_size

        # Certification : renommage
        os.replace(temp, final)

        entry = {
            "file": name,
            "records": len(self.buffer),
            "bytes": size,
            "sha256": digest,
            "source_line_end": source_line_end,
            "committed_at_utc": now_utc()
        }

        # Manifeste maître
        updated = dict(self.manifest)
        updated["completed_chunks"] = (
            self.manifest["completed_chunks"] + [entry]
        )
        updated["scan_lines"] = source_line_end
        updated["selected"] = (
            self.manifest["selected"] + len(self.buffer)
        )

        for key, value in self.interval_counters.items():
            updated[key] = self.manifest.get(key, 0) + value

        updated["source_identity"] = self.identity
        updated["status"] = "running"
        updated["updated_at_utc"] = now_utc()

        atomic_json(MANIFEST, updated)

        # Checkpoint secondaire
        checkpoint = {
            "version": 3,
            "source": SOURCE,
            "source_identity": updated["source_identity"],
            "scan_lines": updated["scan_lines"],
            "selected": updated["selected"],
            "completed_chunks": updated["completed_chunks"]
        }

        atomic_json(CHECKPOINT, checkpoint)

        # Mise à jour mémoire
        self.manifest = updated
        self.committed_line = source_line_end
        self.next_number += 1
        self.buffer = []
        self.interval_counters = {
            "valid_json": 0,
            "invalid_json": 0,
            "out_of_period": 0
        }

    def scan_limited(self, max_lines=None):
        identity = source_identity()
        self.verify_source(identity)

        physical_lines = 0
        start_time = time.time()

        req = Request(
            SOURCE,
            headers={"User-Agent": "NexusScienceScanner/3.0"}
        )

        with urlopen(req, timeout=60) as response:
            remote_identity = {
                "url": SOURCE,
                "etag": response.headers.get("ETag"),
                "last_modified": response.headers.get("Last-Modified"),
                "content_length": response.headers.get("Content-Length")
            }

            if remote_identity != identity:
                raise RuntimeError(
                    "Identité incohérente entre HEAD et GET."
                )

            with gzip.GzipFile(fileobj=response) as gz:

                # Rejouer le flux jusqu'à la frontière certifiée
                while physical_lines < self.committed_line:
                    raw = gz.readline()
                    if not raw:
                        raise RuntimeError(
                            "Source trop courte pour le checkpoint."
                        )
                    physical_lines += 1

                while True:
                    if max_lines is not None:
                        if physical_lines - self.committed_line >= max_lines:
                            break

                    raw = gz.readline()
                    if not raw:
                        break

                    physical_lines += 1

                    try:
                        parts = raw.rstrip(b"\r\n").split(b"\t")
                        record = json.loads(parts[-1])

                        if not isinstance(record, dict):
                            raise ValueError("JSON non objet")

                        self.interval_counters["valid_json"] += 1

                    except Exception:
                        self.interval_counters["invalid_json"] += 1
                        continue

                    if not in_period(record):
                        self.interval_counters["out_of_period"] += 1
                        continue

                    if is_scientific(record):
                        self.buffer.append(record)

                        if len(self.buffer) >= CHUNK_SIZE:
                            self.commit(physical_lines)

                # Segment final du test limité
                if self.buffer:
                    self.commit(physical_lines)

        return {
            "physical_lines": physical_lines,
            "committed_line": self.committed_line,
            "selected_total": self.manifest["selected"],
            "elapsed_seconds": round(time.time() - start_time, 2)
        }

# ------------------------------------------------------------
# INSTALLATION — AUCUNE EXÉCUTION
# ------------------------------------------------------------

scanner_v3 = NexusScannerV3()

print("=== SCANNER SEGMENTÉ V3 ===")
print("Prochain segment :", scanner_v3.next_number)
print("Frontière certifiée :", scanner_v3.committed_line)
print("Source : configurée")
print("Reprise : préparée")
print("RUN_SCAN :", RUN_SCAN)
print("SCAN MASSIF : NON")
print("INSTALLATION : OK")

In [ ]:
import os, json, hashlib

BASE = "/kaggle/working/nexus_science_v3"
MANIFEST = os.path.join(BASE, "manifest_v3.json")
CHUNKS = os.path.join(BASE, "chunks")

print("=== AUDIT FINAL V3 ===")
print("Dossier V3 :", os.path.isdir(BASE))
print("Dossier segments :", os.path.isdir(CHUNKS))
print("Manifeste :", os.path.isfile(MANIFEST))

if os.path.isfile(MANIFEST):
    with open(MANIFEST, "r", encoding="utf-8") as f:
        m = json.load(f)

    print("\nVersion :", m.get("version"))
    print("Statut :", m.get("status"))
    print("Lignes :", m.get("scan_lines"))
    print("Notices :", m.get("selected"))
    print("Segments enregistrés :", len(m.get("completed_chunks", [])))

if os.path.isdir(CHUNKS):
    files = sorted(os.listdir(CHUNKS))
    print("\nFichiers dans chunks :", len(files))
    print("Fichiers :", files[:10])

print("\n=== SCAN MASSIF : NON ===")

In [ ]:
import os

test_file = "/kaggle/working/nexus_science_v3/integration_test/test_chunk.jsonl"

if os.path.isfile(test_file):
    size = os.path.getsize(test_file)
    notices = 2093
    avg = size / notices

    # Estimation indicative, pas une prédiction garantie
    scenarios = [3_000_000, 5_000_000, 7_000_000]

    print("=== CAPACITÉ DE STOCKAGE NEXUS ===")
    print("Taille du test :", round(size / 1024**2, 2), "Mo")
    print("Taille moyenne :", round(avg, 1), "octets / notice")

    for n in scenarios:
        estimated = avg * n
        print(
            f"{n:,} notices : "
            f"{estimated / 1024**3:.2f} Go estimés"
        )

    free = 19.5
    print("\nEspace disponible initial : ~", free, "Go")
else:
    print("Fichier test introuvable.")

import os, json, shutil
from datetime import datetime, timezone

BASE = "/kaggle/working/nexus_science_v3"
PLAN = os.path.join(BASE, "storage_plan_v3.json")

os.makedirs(BASE, exist_ok=True)

free_bytes = shutil.disk_usage("/kaggle/working").free
free_gb = free_bytes / 1024**3

plan = {
    "version": 3,
    "chunk_size_notices": 100000,
    "reserve_disk_gb": 2.0,
    "available_disk_gb_at_setup": round(free_gb, 2),
    "estimated_average_bytes_per_notice": 1051,
    "estimated_notice_scenarios": [3000000, 5000000, 7000000],
    "scan_authorized": False,
    "backup_confirmed": False,
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

with open(PLAN, "w", encoding="utf-8") as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)

print("=== PLAN DE STOCKAGE V3 ===")
print("Espace libre :", round(free_gb, 2), "Go")
print("Segments :", plan["chunk_size_notices"], "notices/segment")
print("Réserve minimale :", plan["reserve_disk_gb"], "Go")
print("Plan enregistré :", os.path.isfile(PLAN))
print("Scan autorisé :", plan["scan_authorized"])
print("Sauvegarde confirmée :", plan["backup_confirmed"])
print("STATUT : PRÉPARATION")

import os, json, hashlib, shutil
from datetime import datetime, timezone

BASE = "/kaggle/working/nexus_science_v3"
CHUNKS = os.path.join(BASE, "chunks")
MANIFEST = os.path.join(BASE, "manifest_v3.json")
PROTECTION = os.path.join(BASE, "protection_v3")
os.makedirs(PROTECTION, exist_ok=True)

def v3_protection_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_protection_inventory():
    with open(MANIFEST, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    entries = manifest.get("completed_chunks", [])
    inventory = []

    for entry in entries:
        filename = entry.get("filename") or entry.get("file")
        if not filename:
            raise ValueError("Entrée de segment sans nom de fichier.")

        path = os.path.join(CHUNKS, filename)

        if not os.path.isfile(path):
            raise FileNotFoundError(f"Segment manquant : {filename}")

        inventory.append({
            "filename": filename,
            "bytes": os.path.getsize(path),
            "sha256": v3_protection_hash(path)
        })

    return inventory

def v3_disk_guard(required_bytes=0, reserve_gb=2):
    free = shutil.disk_usage(BASE).free
    reserve = int(reserve_gb * 1024**3)
    return {
        "free_bytes": free,
        "reserve_bytes": reserve,
        "safe": free >= required_bytes + reserve
    }

def v3_export_inventory():
    inventory = v3_protection_inventory()

    report = {
        "version": 3,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "segments": inventory,
        "segment_count": len(inventory),
        "total_bytes": sum(x["bytes"] for x in inventory),
        "backup_verified": False,
        "backup_destination": None
    }

    target = os.path.join(PROTECTION, "inventory_v3.json")
    temp = target + ".partial"

    with open(temp, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, target)
    return report

print("=== PROTECTION V3 INSTALLÉE ===")
print("Inventaire certifiable :", callable(v3_protection_inventory))
print("Contrôle disque :", callable(v3_disk_guard))
print("Export inventaire :", callable(v3_export_inventory))
print("Destination externe : non configurée")
print("Sauvegarde réelle : non confirmée")
print("Scan massif : NON")
print("INSTALLATION : OK")

In [ ]:
!pip install -q huggingfacprint("=== FONCTIONS NEXUS DISPONIBLES ===")

for name, obj in sorted(globals().items()):
    if name.startswith("nexus_") and callable(obj):
        print(name)

print("\n=== VARIABLES NEXUS ===")

for name in sorted(globals()):
    if name.startswith(("NEXUS_", "HF_", "CHUNK_", "SCAN_")):
        obj = globals()[name]
        if not callable(obj):
            print(name, "=", str(obj)[:120])e-hub==1.32.0

In [ ]:
from huggingface_hub import hf_hub_upload, hf_hub_download
from kaggle_secrets import UserSecretsClient
import os, hashlib

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO_ID = "Moses-team/nexus-science-backup"

V10_BACKUP_DIR = "v10_recovery_v10_3"

commit, stage_path = v10_read_commit()
stage_name = commit["stage_filename"]
remote_stage = f"{V10_BACKUP_DIR}/{stage_name}"

hf_hub_upload(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    path_or_fileobj=stage_path,
    path_in_repo=remote_stage,
    token=HF_TOKEN
)

print("=== TRANSFERT STAGE V10.3 ===")
print("Génération :", commit["generation"])
print("Fichier :", remote_stage)
print("Taille locale :", os.path.getsize(stage_path))
print("STATUT : STAGE ENVOYE")


In [ ]:
from pathlib import Path

racines = [Path("/kaggle/input"), Path("/kaggle/working")]

mots_cles = ("edition", "work", "author")
extensions = (".gz", ".txt", ".json", ".jsonl", ".csv", ".zip")

print("=== RECHERCHE DES DUMPS OPEN LIBRARY ===")

for racine in racines:
    print(f"\nDossier : {racine}")
    if not racine.exists():
        print("Dossier absent")
        continue

    trouves = []
    for fichier in racine.rglob("*"):
        if fichier.is_file():
            nom = fichier.name.lower()
            if any(mot in nom for mot in mots_cles) and nom.endswith(extensions):
                trouves.append(fichier)

    if trouves:
        for fichier in trouves:
            print(f"{fichier} — {fichier.stat().st_size / (1024**3):.3f} Go")
    else:
        print("Aucun dump correspondant trouvé.")

print("\n=== FIN DE LA RECHERCHE ===")


In [ ]:
import requests

base = "https://openlibrary.org/data/"
fichiers = [
    "ol_dump_editions_latest.txt.gz",
    "ol_dump_works_latest.txt.gz",
    "ol_dump_authors_latest.txt.gz",
]

print("=== DISPONIBILITÉ DES SOURCES OPEN LIBRARY ===")

for nom in fichiers:
    try:
        r = requests.head(base + nom, timeout=30, allow_redirects=True)
        taille = r.headers.get("Content-Length")
        go = f"{int(taille)/(1024**3):.2f} Go" if taille and taille.isdigit() else "Taille inconnue"
        print(f"{nom} | HTTP {r.status_code} | {go}")
    except Exception as e:
        print(f"{nom} | ERREUR : {e}")


In [ ]:
import requests
import gzip
import json

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

print("=== TEST DE LECTURE EN FLUX ===")

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()
    response.raw.decode_content = False

    with gzip.GzipFile(fileobj=response.raw) as dump:
        for i in range(5):
            ligne = dump.readline()
            if not ligne:
                break

            parties = ligne.decode("utf-8").rstrip("\n").split("\t", 2)
            print(f"\nLigne {i+1}")
            print("Identifiant :", parties[0] if len(parties) > 0 else "?")
            print("Révision :", parties[1] if len(parties) > 1 else "?")

            if len(parties) > 2:
                notice = json.loads(parties[2])
                print("Champs :", list(notice.keys())[:20])

print("\n=== TEST TERMINÉ ===")


In [ ]:
import requests
import gzip

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

print("=== INSPECTION DU FORMAT BRUT ===")

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()

    with gzip.GzipFile(fileobj=response.raw) as dump:
        for i in range(3):
            ligne = dump.readline()

            if not ligne:
                break

            texte = ligne.decode("utf-8", errors="replace").rstrip("\n")
            print(f"\nLIGNE {i+1}")
            print("Longueur :", len(texte))
            print("Début :", repr(texte[:700]))
            print("Nombre de tabulations :", texte.count("\t"))

print("\n=== INSPECTION TERMINÉE ===")


In [ ]:
import requests
import gzip
import json

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

print("=== VALIDATION DU PARSEUR OPEN LIBRARY ===")

valides = 0
invalides = 0

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()

    with gzip.GzipFile(fileobj=response.raw) as dump:
        for i in range(10):
            ligne = dump.readline()
            if not ligne:
                break

            try:
                champs = ligne.decode("utf-8").rstrip("\n").split("\t", 4)

                if len(champs) != 5:
                    raise ValueError(f"{len(champs)} champs au lieu de 5")

                type_notice, olid, revision, date_modif, contenu = champs
                notice = json.loads(contenu)

                assert notice.get("key") == olid

                valides = {
                    "OLID": olid,
                    "titre": notice.get("title"),
                    "works": [w.get("key") for w in notice.get("works", [])],
                    "authors": [a.get("key") for a in notice.get("authors", [])]
                }

                print(f"\nNotice {i+1} : OK")
                print(valides)
                valides += 1

            except Exception as e:
                invalides += 1
                print(f"Notice {i+1} : ERREUR — {e}")

print("\n=== BILAN ===")
print("Notices valides :", valides)
print("Notices invalides :", invalides)


In [ ]:
import requests
import gzip
import json

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

print("=== VALIDATION CORRIGÉE ===")

compteur_valides = 0
compteur_invalides = 0

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()

    with gzip.GzipFile(fileobj=response.raw) as dump:
        for i in range(10):
            ligne = dump.readline()
            if not ligne:
                break

            try:
                champs = ligne.decode("utf-8").rstrip("\n").split("\t", 4)
                if len(champs) != 5:
                    raise ValueError("Structure incorrecte")

                type_notice, olid, revision, date_modif, contenu = champs
                notice = json.loads(contenu)

                if notice.get("key") != olid:
                    raise ValueError("Identifiant incohérent")

                resultat = {
                    "OLID": olid,
                    "titre": notice.get("title"),
                    "works": [x.get("key") for x in notice.get("works", [])],
                    "authors": [x.get("key") for x in notice.get("authors", [])]
                }

                compteur_valides += 1
                print(f"Notice {i+1}: OK — {resultat['OLID']}")

            except Exception as erreur:
                compteur_invalides += 1
                print(f"Notice {i+1}: ERREUR — {erreur}")

print("\n=== BILAN FINAL ===")
print("Notices valides :", compteur_valides)
print("Notices invalides :", compteur_invalides)
print("Statut :", "VALIDÉ" if compteur_valides == 10 and compteur_invalides == 0 else "À EXAMINER")


In [ ]:
import requests, gzip, json, re, hashlib, time
from pathlib import Path

URL = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
MAX_SCAN = 100_000
TARGET = 500

# Filtre exploratoire : candidats scientifiques, pas certification définitive.
MOTS = re.compile(
    r"\b(science|scientific|physics|chemistry|biology|"
    r"mathematics|mathematical|medicine|medical|engineering|"
    r"astronomy|geology|ecology|genetics|neuroscience|"
    r"computer science|technology|research|statistics)\b",
    re.I
)

sortie = Path("/kaggle/working/nexus_pilot_500.jsonl")
examines = candidats = invalides = 0
empreinte = hashlib.sha256()

print("=== PILOTE NEXUS : 500 CANDIDATS MAXIMUM ===")

with requests.get(URL, stream=True, timeout=(30, 180)) as response:
    response.raise_for_status()

    with gzip.GzipFile(fileobj=response.raw) as dump, sortie.open("w", encoding="utf-8") as fichier:
        while examines < MAX_SCAN and candidats < TARGET:
            ligne = dump.readline()
            if not ligne:
                break

            examines += 1
            empreinte.update(ligne)

            try:
                champs = ligne.decode("utf-8").rstrip("\n").split("\t", 4)
                if len(champs) != 5:
                    raise ValueError("Format incorrect")

                type_notice, olid, revision, date_modif, contenu = champs
                notice = json.loads(contenu)

                if type_notice != "/type/edition" or notice.get("key") != olid:
                    raise ValueError("Identité incohérente")

                titre = notice.get("title") or ""
                sujets = notice.get("subjects") or []
                texte = titre + " " + " ".join(
                    s for s in sujets if isinstance(s, str)
                )

                if not MOTS.search(texte):
                    continue

                resultat = {
                    "olid": olid,
                    "revision": revision,
                    "modified": date_modif,
                    "title": titre,
                    "subjects": sujets,
                    "works": [x["key"] for x in notice.get("works", [])
                              if isinstance(x, dict) and "key" in x],
                    "authors": [x["key"] for x in notice.get("authors", [])
                                if isinstance(x, dict) and "key" in x],
                    "source": "Open Library"
                }

                fichier.write(json.dumps(resultat, ensure_ascii=False) + "\n")
                candidats += 1

            except Exception:
                invalides += 1

print("\n=== BILAN DU PILOTE ===")
print("Lignes examinées :", examines)
print("Candidats retenus :", candidats)
print("Lignes invalides :", invalides)
print("SHA-256 des lignes examinées :", empreinte.hexdigest())
print("Fichier :", sortie)
print("Taille :", round(sortie.stat().st_size / 1024, 2), "Ko")
print("Statut :", "ÉCHANTILLON PRÉPARÉ" if candidats else "AUCUN CANDIDAT")


In [ ]:
import json
from pathlib import Path
from collections import Counter

fichier = Path("/kaggle/working/nexus_pilot_500.jsonl")

notices = [
    json.loads(ligne)
    for ligne in fichier.open(encoding="utf-8")
    if ligne.strip()
]

ids = [n["olid"] for n in notices]
sans_titre = [n for n in notices if not n.get("title")]
sans_work = [n for n in notices if not n.get("works")]
sans_author = [n for n in notices if not n.get("authors")]

sujets = Counter(
    sujet
    for n in notices
    for sujet in n.get("subjects", [])
    if isinstance(sujet, str)
)

print("=== AUDIT QUALITÉ DU PILOTE ===")
print("Notices :", len(notices))
print("Identifiants uniques :", len(set(ids)))
print("Doublons :", len(ids) - len(set(ids)))
print("Titres absents :", len(sans_titre))
print("Références Works absentes :", len(sans_work))
print("Références Authors absentes :", len(sans_author))

print("\n=== 20 SUJETS LES PLUS FRÉQUENTS ===")
for sujet, nombre in sujets.most_common(20):
    print(nombre, "|", sujet)

print("\n=== 10 PREMIERS TITRES ===")
for n in notices[:10]:
    print("-", n.get("title"))

print("\n=== FIN DE L'AUDIT ===")


In [ ]:
import json
from pathlib import Path
from collections import defaultdict, Counter

fichier = Path("/kaggle/working/nexus_pilot_500.jsonl")
notices = [json.loads(x) for x in fichier.open(encoding="utf-8") if x.strip()]

# Regroupement des titres répétés
groupes = defaultdict(list)
for n in notices:
    titre = " ".join((n.get("title") or "").lower().split())
    groupes[titre].append(n)

repetes = {k: v for k, v in groupes.items() if len(v) > 1}

print("=== TITRES RÉPÉTÉS ===")
print("Groupes :", len(repetes))
for titre, items in sorted(repetes.items(), key=lambda x: -len(x[1]))[:15]:
    print(f"\n{len(items)} éditions | {titre}")
    for n in items[:5]:
        print(" ", n["olid"], "| Works:", n.get("works", []))

print("\n=== NOTICES SANS RÉFÉRENCE ===")
for champ in ("works", "authors"):
    absentes = [n for n in notices if not n.get(champ)]
    print(f"\nSans {champ} : {len(absentes)}")
    for n in absentes[:10]:
        print(n["olid"], "|", n.get("title"))

print("\n=== RÉPARTITION DES SUJETS ===")
compteur = Counter(
    sujet.strip()
    for n in notices
    for sujet in n.get("subjects", [])
    if isinstance(sujet, str) and sujet.strip()
)
for sujet, nombre in compteur.most_common(30):
    print(nombre, "|", sujet)


In [ ]:
import json
from pathlib import Path

fichier = Path("/kaggle/working/nexus_pilot_500.jsonl")
notices = [json.loads(x) for x in fichier.open(encoding="utf-8") if x.strip()]

works = set()
authors = set()

for n in notices:
    works.update(n.get("works", []))
    authors.update(n.get("authors", []))

resultat = {
    "edition_count": len(notices),
    "unique_work_ids": len(works),
    "unique_author_ids": len(authors),
    "editions_without_work": sum(not n.get("works") for n in notices),
    "editions_without_author": sum(not n.get("authors") for n in notices),
    "work_ids": sorted(works),
    "author_ids": sorted(authors)
}

sortie = Path("/kaggle/working/nexus_reference_manifest.json")
sortie.write_text(
    json.dumps(resultat, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("=== MANIFESTE DES RÉFÉRENCES ===")
print("Éditions :", resultat["edition_count"])
print("Works uniques :", resultat["unique_work_ids"])
print("Authors uniques :", resultat["unique_author_ids"])
print("Éditions sans Works :", resultat["editions_without_work"])
print("Éditions sans Authors :", resultat["editions_without_author"])
print("Manifeste :", sortie)
print("STATUT : MANIFESTE PRÉPARÉ")


In [ ]:
import json
import requests
import time
from pathlib import Path

manifest = json.loads(
    Path("/kaggle/working/nexus_reference_manifest.json").read_text(encoding="utf-8")
)

session = requests.Session()
session.headers.update({
    "User-Agent": "NexusScienceBibliographicPilot/1.0"
})

def verifier_references(identifiants, categorie):
    resultats = []
    print(f"\n=== TEST {categorie.upper()} ===")

    for identifiant in identifiants[:10]:
        url = f"https://openlibrary.org{identifiant}.json"

        try:
            response = session.get(url, timeout=30)

            if response.status_code == 200:
                notice = response.json()
                identite_ok = notice.get("key") == identifiant

                resultat = {
                    "id": identifiant,
                    "status": "OK" if identite_ok else "IDENTITE_INCOHERENTE",
                    "title": notice.get("title"),
                    "name": notice.get("name")
                }
            else:
                resultat = {
                    "id": identifiant,
                    "status": f"HTTP_{response.status_code}"
                }

        except Exception as erreur:
            resultat = {
                "id": identifiant,
                "status": "ERREUR",
                "detail": str(erreur)
            }

        resultats.append(resultat)
        print(resultat)
        time.sleep(1)

    return resultats

works = verifier_references(manifest["work_ids"], "Works")
authors = verifier_references(manifest["author_ids"], "Authors")

sortie = {
    "works_test": works,
    "authors_test": authors
}

Path("/kaggle/working/nexus_reference_test.json").write_text(
    json.dumps(sortie, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n=== BILAN ===")
print("Works valides :", sum(x["status"] == "OK" for x in works), "/ 10")
print("Authors valides :", sum(x["status"] == "OK" for x in authors), "/ 10")
print("Fichier : nexus_reference_test.json")


In [ ]:
import requests, json, time
from pathlib import Path

BASE = Path("/kaggle/working")
MANIFEST = json.loads(
    (BASE / "nexus_reference_manifest.json").read_text(encoding="utf-8")
)

session = requests.Session()
session.headers.update({
    "User-Agent": "NexusScienceBibliographicPilot/1.0"
})

def charger_cache(nom):
    chemin = BASE / nom
    cache = {}
    if chemin.exists():
        for ligne in chemin.open(encoding="utf-8"):
            try:
                item = json.loads(ligne)
                cache[item["id"]] = item
            except Exception:
                continue
    return cache

def resoudre(identifiants, categorie, nom_fichier):
    cache = charger_cache(nom_fichier)
    chemin = BASE / nom_fichier

    restants = [x for x in identifiants if x not in cache]

    print(f"\n=== {categorie.upper()} ===")
    print("Références :", len(identifiants))
    print("Déjà traitées :", len(cache))
    print("À traiter :", len(restants))

    with chemin.open("a", encoding="utf-8") as sortie:
        for numero, identifiant in enumerate(restants, 1):
            url = f"https://openlibrary.org{identifiant}.json"
            resultat = {"id": identifiant, "status": "ERROR"}

            for tentative in range(4):
                try:
                    r = session.get(url, timeout=40)

                    if r.status_code == 200:
                        notice = r.json()
                        if notice.get("key") != identifiant:
                            resultat["status"] = "IDENTITY_MISMATCH"
                        else:
                            resultat = {
                                "id": identifiant,
                                "status": "OK",
                                "notice": notice
                            }
                        break

                    if r.status_code == 404:
                        resultat["status"] = "NOT_FOUND"
                        break

                    if r.status_code == 429 or r.status_code >= 500:
                        resultat["status"] = f"HTTP_{r.status_code}"
                        time.sleep(5 * (tentative + 1))
                        continue

                    resultat["status"] = f"HTTP_{r.status_code}"
                    break

                except Exception as erreur:
                    resultat["status"] = "ERROR"
                    resultat["detail"] = str(erreur)
                    time.sleep(5 * (tentative + 1))

            sortie.write(json.dumps(resultat, ensure_ascii=False) + "\n")
            sortie.flush()
            cache[identifiant] = resultat

            if numero % 25 == 0:
                ok = sum(x["status"] == "OK" for x in cache.values())
                print(f"{categorie}: {len(cache)}/{len(identifiants)} | Valides: {ok}")

            time.sleep(1.1)

    return cache

works = resoudre(
    MANIFEST["work_ids"], "Works", "nexus_works_resolved.jsonl"
)

authors = resoudre(
    MANIFEST["author_ids"], "Authors", "nexus_authors_resolved.jsonl"
)

rapport = {
    "works": {
        "total": len(works),
        "valides": sum(x["status"] == "OK" for x in works.values()),
        "echecs": sum(x["status"] != "OK" for x in works.values())
    },
    "authors": {
        "total": len(authors),
        "valides": sum(x["status"] == "OK" for x in authors.values()),
        "echecs": sum(x["status"] != "OK" for x in authors.values())
    }
}

(BASE / "nexus_resolution_report.json").write_text(
    json.dumps(rapport, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n=== RAPPORT FINAL ===")
print(json.dumps(rapport, ensure_ascii=False, indent=2))
print("STATUT : RÉSOLUTION TERMINÉE")


In [ ]:
import json
import hashlib
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working")

FICHIERS = {
    "editions": BASE / "nexus_pilot_500.jsonl",
    "works": BASE / "nexus_works_resolved.jsonl",
    "authors": BASE / "nexus_authors_resolved.jsonl"
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloc in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def charger(path):
    lignes, erreurs = [], 0
    with path.open(encoding="utf-8") as f:
        for ligne in f:
            if not ligne.strip():
                continue
            try:
                lignes.append(json.loads(ligne))
            except Exception:
                erreurs += 1
    return lignes, erreurs

print("=== AUDIT D'INTÉGRITÉ NEXUS ===")

donnees = {}
for nom, chemin in FICHIERS.items():
    if not chemin.exists():
        raise FileNotFoundError(f"Fichier manquant : {chemin}")
    donnees[nom], erreurs = charger(chemin)
    print(f"{nom}: {len(donnees[nom])} lignes | JSON invalides: {erreurs}")

editions = donnees["editions"]
works = donnees["works"]
authors = donnees["authors"]

# Index des notices résolues
def indexer(items):
    index = {}
    doublons = 0
    incoherences = 0

    for item in items:
        identifiant = item.get("id")
        notice = item.get("notice", {})
        if identifiant in index:
            doublons += 1
        index[identifiant] = item

        if item.get("status") == "OK":
            if notice.get("key") != identifiant:
                incoherences += 1

    return index, doublons, incoherences

index_works, doublons_w, incoherences_w = indexer(works)
index_authors, doublons_a, incoherences_a = indexer(authors)

ids_editions = [e.get("olid") for e in editions]
doublons_editions = len(ids_editions) - len(set(ids_editions))

refs_works = {
    ref for e in editions for ref in e.get("works", [])
}
refs_authors = {
    ref for e in editions for ref in e.get("authors", [])
}

works_absents = sorted(refs_works - set(index_works))
authors_absents = sorted(refs_authors - set(index_authors))

works_non_ok = [
    k for k in refs_works
    if index_works.get(k, {}).get("status") != "OK"
]
authors_non_ok = [
    k for k in refs_authors
    if index_authors.get(k, {}).get("status") != "OK"
]

rapport = {
    "project": "Bibliothèque Scientifique — Nexus Science",
    "pilot": {
        "editions": len(editions),
        "unique_works_referenced": len(refs_works),
        "unique_authors_referenced": len(refs_authors)
    },
    "integrity": {
        "duplicate_editions": doublons_editions,
        "duplicate_work_cache_ids": doublons_w,
        "duplicate_author_cache_ids": doublons_a,
        "work_identity_mismatches": incoherences_w,
        "author_identity_mismatches": incoherences_a,
        "missing_work_references": len(works_absents),
        "missing_author_references": len(authors_absents),
        "non_valid_work_references": len(works_non_ok),
        "non_valid_author_references": len(authors_non_ok)
    },
    "files_sha256": {
        nom: sha256(path) for nom, path in FICHIERS.items()
    }
}

controle_ok = (
    len(editions) == 500
    and len(refs_works) == 438
    and len(refs_authors) == 435
    and doublons_editions == 0
    and doublons_w == 0
    and doublons_a == 0
    and incoherences_w == 0
    and incoherences_a == 0
    and not works_absents
    and not authors_absents
    and not works_non_ok
    and not authors_non_ok
)

rapport["integrity"]["audit_passed"] = controle_ok

sortie = BASE / "nexus_pilot_integrity_audit.json"
sortie.write_text(
    json.dumps(rapport, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n=== RÉSULTAT ===")
print(json.dumps(rapport["integrity"], ensure_ascii=False, indent=2))
print("\nSHA-256 des trois fichiers :")
for nom, valeur in rapport["files_sha256"].items():
    print(nom, ":", valeur)

print("\nSTATUT :", "AUDIT INTÈGRE" if controle_ok else "ANOMALIES À EXAMINER")
print("Rapport :", sortie)


In [ ]:
import json
from pathlib import Path

files = [
    "nexus_pilot_500.jsonl",
    "nexus_works_resolved.jsonl",
    "nexus_authors_resolved.jsonl"
]

for filename in files:
    path = Path("/kaggle/working") / filename
    print(f"\n=== {filename} ===")

    with path.open(encoding="utf-8") as f:
        record = json.loads(next(line for line in f if line.strip()))

    print("Champs :", list(record.keys()))
    print("Exemple :", json.dumps(record, ensure_ascii=False)[:1800])

In [ ]:
import json, re, hashlib
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working")
ED_FILE = BASE / "nexus_pilot_500.jsonl"
WO_FILE = BASE / "nexus_works_resolved.jsonl"

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

editions = load_jsonl(ED_FILE)
works = load_jsonl(WO_FILE)

work_map = {
    r["id"]: r["notice"]
    for r in works
    if r.get("status") == "OK" and isinstance(r.get("notice"), dict)
}

SCI = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological", "medicine",
    "medical", "engineering", "astronomy", "geology", "ecology",
    "genetics", "neuroscience", "computer science", "technology",
    "statistics", "research", "laboratory", "algorithm", "robotics",
    "thermodynamics", "quantum", "microbiology", "biochemistry",
    "anatomy", "epidemiology", "mechanics", "mineralogy",
    "meteorology", "agriculture", "pharmacology", "environmental science",
    "econometrics", "econometric", "operations research",
    "data science", "artificial intelligence", "machine learning",
    "mathematiques", "physique", "chimie", "biologie", "medecine",
    "ingenierie", "informatique", "astronomie", "geologie",
    "statistique", "recherche scientifique", "neurosciences",
    "épidémiologie", "génétique", "écologie", "technologie",
    "ciencia", "matemática", "matemáticas", "física", "química",
    "biologia", "medicina", "engenharia", "engenharia",
    "estatística", "astronomia", "geologia", "informática",
    "investigação científica", "tecnologia"
]

QUANT = [
    "economics", "econometric", "statistics", "sociology",
    "demography", "political science", "quantitative",
    "finance", "public policy", "social science", "education research",
    "economic analysis", "economia", "sociologia", "demografia",
    "ciências sociais", "ciencias sociales", "quantitativo",
    "quantitative analysis"
]

INSTITUTION = [
    "technical report", "research report", "scientific report",
    "laboratory report", "institute", "research institute",
    "technical paper", "working paper", "government report",
    "rapport scientifique", "rapport technique",
    "instituto de pesquisa", "relatório técnico"
]

def norm(text):
    return re.sub(r"\s+", " ", str(text or "").lower()).strip()

def hits(text, words):
    text = norm(text)
    return [w for w in words if w in text]

results = []

for ed in editions:
    work_ids = ed.get("works") or []
    work_records = [work_map[w] for w in work_ids if w in work_map]

    ed_text = norm(" ".join([
        ed.get("title", ""),
        " ".join(ed.get("subjects") or [])
    ]))

    wo_text = norm(" ".join([
        " ".join(w.get("title", "") for w in work_records),
        " ".join(
            s for w in work_records
            for s in (w.get("subjects") or [])
        )
    ]))

    combined = ed_text + " " + wo_text

    sci_hits = hits(combined, SCI)
    quant_hits = hits(combined, QUANT)
    inst_hits = hits(combined, INSTITUTION)

    score = len(set(sci_hits)) * 2 + len(set(quant_hits))
    reasons = []

    if sci_hits:
        reasons.append("scientific_keywords:" + ",".join(sorted(set(sci_hits))))
    if quant_hits:
        reasons.append("quantitative_keywords:" + ",".join(sorted(set(quant_hits))))
    if inst_hits:
        reasons.append("institutional_keywords:" + ",".join(sorted(set(inst_hits))))

    if score >= 4:
        category = "SCIENTIFIC"
    elif score >= 1 or inst_hits:
        category = "REVIEW"
    else:
        category = "REVIEW"

    if not work_records:
        reasons.append("work_reference_unresolved_or_missing")

    results.append({
        "olid": ed.get("olid"),
        "title": ed.get("title"),
        "subjects": ed.get("subjects", []),
        "works": work_ids,
        "authors": ed.get("authors", []),
        "work_titles": [w.get("title") for w in work_records],
        "work_subjects": [w.get("subjects", []) for w in work_records],
        "classification": category,
        "score": score,
        "reasons": reasons,
        "source": ed.get("source")
    })

out_file = BASE / "nexus_pilot_classified_v1.jsonl"

with out_file.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

counts = Counter(r["classification"] for r in results)

report = {
    "version": "V1",
    "total": len(results),
    "scientific": counts["SCIENTIFIC"],
    "review": counts["REVIEW"],
    "excluded": 0,
    "note": "Classification keyword-based and provisional; review required.",
    "sha256": hashlib.sha256(out_file.read_bytes()).hexdigest()
}

(BASE / "nexus_pilot_classification_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== CLASSIFICATION V1 ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Fichier :", out_file)
print("STATUT : CLASSIFICATION PRELIMINAIRE")

In [ ]:
import json
from collections import Counter

path = "/kaggle/working/nexus_pilot_classified_v1.jsonl"

data = []
with open(path, encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

scientific = [x for x in data if x["classification"] == "SCIENTIFIC"]
review = [x for x in data if x["classification"] == "REVIEW"]

def subjects_count(items):
    c = Counter()
    for x in items:
        for s in x.get("subjects", []):
            c[s] += 1
    return c.most_common(20)

print("=== SUJETS SCIENTIFIQUES ===")
for s, n in subjects_count(scientific):
    print(n, "|", s)

print("\n=== SUJETS A EXAMINER ===")
for s, n in subjects_count(review):
    print(n, "|", s)

print("\n=== 30 EXEMPLES SCIENTIFIC ===")
for x in scientific[:30]:
    print("-", x.get("title"), "|", x.get("subjects"))

print("\n=== 30 EXEMPLES REVIEW ===")
for x in review[:30]:
    print("-", x.get("title"), "|", x.get("subjects"))

In [ ]:
import json, re, hashlib
from collections import Counter

src = "/kaggle/working/nexus_pilot_classified_v1.jsonl"
out = "/kaggle/working/nexus_pilot_classified_v2.jsonl"
report_path = "/kaggle/working/nexus_pilot_classification_report_v2.json"

with open(src, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

POSITIVE = {
    "science": 2, "science/mathematics": 3, "general science": 2,
    "mathematics": 2, "physics": 3, "chemistry": 3,
    "biology": 3, "life sciences": 3, "medical": 3,
    "medicine": 3, "engineering": 3, "geology": 3,
    "astronomy": 3, "ecology": 3, "computer science": 3,
    "computing": 2, "statistics": 3, "probability": 3,
    "econometrics": 3, "oceanography": 3, "botany": 3,
    "biochemistry": 3, "technology": 1, "industrial arts": 1,
    "quantitative": 2, "demography": 2, "social research": 2,
    "scientific research": 3, "research": 1,
    "radiation biology": 3, "medical toxicology": 3
}

NEGATIVE = {
    "english law": 3, "law and legislation": 3,
    "legal status": 3, "contracts": 2, "contract management": 2,
    "curriculum": 2, "teaching": 2, "teacher's guide": 2,
    "juvenile nonfiction": 2, "parenting": 2,
    "management decision making": 1, "government contracts": 3,
    "politics/current events": 2
}

def norm(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()

def score_record(x):
    title = norm(x.get("title", ""))
    subjects = [norm(s) for s in x.get("subjects", [])]

    work_subjects = x.get("work_subjects", [])
    if isinstance(work_subjects, list):
        subjects += [norm(s) for s in work_subjects]

    unique_subjects = list(set(subjects))
    positive_hits, negative_hits = [], []

    for s in unique_subjects:
        for term, weight in POSITIVE.items():
            if term in s:
                positive_hits.append((term, weight))

    for s in unique_subjects + [title]:
        for term, weight in NEGATIVE.items():
            if term in s:
                negative_hits.append((term, weight))

    # Les sujets précis priment sur les mots génériques du titre.
    positive_score = sum(w for _, w in positive_hits)
    negative_score = sum(w for _, w in negative_hits)

    title_hits = [
        term for term in POSITIVE
        if term in title and term not in ["science", "technology", "research"]
    ]

    # Bonus limité pour le titre.
    positive_score += min(2, len(title_hits))

    final_score = positive_score - negative_score

    classification = "SCIENTIFIC" if final_score >= 3 else "REVIEW"

    return {
        "classification": classification,
        "score": final_score,
        "positive_hits": sorted(set(t for t, _ in positive_hits)),
        "negative_hits": sorted(set(t for t, _ in negative_hits)),
        "title_hits": title_hits
    }

classified = []

for x in data:
    result = score_record(x)
    y = dict(x)
    y.update(result)
    y["classification_version"] = "V2"
    classified.append(y)

with open(out, "w", encoding="utf-8") as f:
    for x in classified:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

counts = Counter(x["classification"] for x in classified)

sha = hashlib.sha256(open(out, "rb").read()).hexdigest()

report = {
    "version": "V2",
    "total": len(classified),
    "scientific": counts["SCIENTIFIC"],
    "review": counts["REVIEW"],
    "excluded": 0,
    "method": "subject weighted scoring with title bonus and penalty",
    "sha256": sha
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("=== CLASSIFICATION V2 ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\n=== 25 SCIENTIFIC ===")
for x in [r for r in classified if r["classification"] == "SCIENTIFIC"][:25]:
    print(x["score"], "|", x["title"], "|", x["positive_hits"])

print("\n=== 25 REVIEW ===")
for x in [r for r in classified if r["classification"] == "REVIEW"][:25]:
    print(x["score"], "|", x["title"], "|", x["negative_hits"])

print("\nFichier :", out)
print("STATUT : CLASSIFICATION V2 PRELIMINAIRE")

In [ ]:
import json
from collections import Counter

path = "/kaggle/working/nexus_pilot_classified_v2.jsonl"

data = []
with open(path, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

selected = [x for x in data if x["classification"] == "SCIENTIFIC"]

work_ids = [
    w for x in selected
    for w in (x.get("works") or [])
    if isinstance(w, str)
]

unique_works = set(work_ids)
title_counts = Counter(x.get("title", "") for x in selected)

print("=== AUDIT DE DEDUPLICATION ===")
print("Editions sélectionnées :", len(selected))
print("Références Work :", len(work_ids))
print("Works uniques :", len(unique_works))
print("Works manquants :", sum(not x.get("works") for x in selected))
print("Titres répétés :", sum(n > 1 for n in title_counts.values()))

print("\n=== 20 TITRES LES PLUS REPETES ===")
for title, n in title_counts.most_common(20):
    if n > 1:
        print(n, "|", title)

print("\n=== STATUT ===")
print("AUDIT DE DEDUPLICATION TERMINE")

In [ ]:
import os, json, hashlib
from collections import defaultdict, Counter

BASE = "/kaggle/working"
PACKAGE = os.path.join(BASE, "nexus_science_v3")
os.makedirs(PACKAGE, exist_ok=True)

EDITION_FILE = os.path.join(BASE, "nexus_pilot_classified_v2.jsonl")
WORK_FILE = os.path.join(BASE, "nexus_works_resolved.jsonl")
AUTHOR_FILE = os.path.join(BASE, "nexus_authors_resolved.jsonl")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def norm_ids(values):
    if values is None:
        return []
    if isinstance(values, str):
        values = [values]
    result = []
    for v in values:
        if isinstance(v, dict):
            v = v.get("key") or v.get("id") or v.get("olid")
        if isinstance(v, str) and v.strip():
            result.append(v.strip())
    return list(dict.fromkeys(result))

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

editions = load_jsonl(EDITION_FILE)
works_cache = load_jsonl(WORK_FILE)
authors_cache = load_jsonl(AUTHOR_FILE)

work_map = {x["id"]: x for x in works_cache if x.get("id")}
author_map = {x["id"]: x for x in authors_cache if x.get("id")}

selected = [x for x in editions if x.get("classification") == "SCIENTIFIC"]

unique_editions = {}
unique_works = {}
unique_authors = {}
quarantine = []
warnings = []

for e in selected:
    edition_id = e.get("olid")
    work_ids = norm_ids(e.get("works"))
    author_ids = norm_ids(e.get("authors"))

    if not edition_id:
        quarantine.append({"reason": "MISSING_EDITION_ID", "edition": e})
        continue

    if len(work_ids) != 1:
        quarantine.append({
            "reason": "MISSING_OR_AMBIGUOUS_WORK",
            "edition_id": edition_id,
            "work_ids": work_ids,
            "edition": e
        })
        continue

    wid = work_ids[0]
    cached_work = work_map.get(wid)

    if not cached_work or cached_work.get("status") != "valid":
        quarantine.append({
            "reason": "UNRESOLVED_WORK",
            "edition_id": edition_id,
            "work_id": wid,
            "edition": e
        })
        continue

    unique_editions[edition_id] = e
    unique_works[wid] = cached_work

    for aid in author_ids:
        cached_author = author_map.get(aid)
        if cached_author and cached_author.get("status") == "valid":
            unique_authors[aid] = cached_author
        else:
            warnings.append({
                "edition_id": edition_id,
                "author_id": aid,
                "reason": "AUTHOR_UNRESOLVED"
            })

# Génération des fichiers
files = {
    "editions.jsonl": list(unique_editions.values()),
    "works.jsonl": list(unique_works.values()),
    "authors.jsonl": list(unique_authors.values()),
    "quarantine.jsonl": quarantine,
    "warnings.jsonl": warnings
}

for filename, rows in files.items():
    save_jsonl(os.path.join(PACKAGE, filename), rows)

manifest = {
    "version": "V3",
    "classification": "V2 provisional",
    "status": "PACKAGE_PREPARED",
    "selected_input": len(selected),
    "editions_ready": len(unique_editions),
    "unique_works_ready": len(unique_works),
    "unique_authors_ready": len(unique_authors),
    "quarantined": len(quarantine),
    "warnings": len(warnings),
    "canonical_database_write": False,
    "files": {}
}

for filename in files:
    path = os.path.join(PACKAGE, filename)
    manifest["files"][filename] = {
        "rows": len(files[filename]),
        "bytes": os.path.getsize(path),
        "sha256": sha256(path)
    }

manifest_path = os.path.join(PACKAGE, "manifest_v3.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== GESTIONNAIRE NEXUS V3 ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nDOSSIER :", PACKAGE)
print("STATUT :", manifest["status"])

In [ ]:
import json
from collections import Counter

p = "/kaggle/working/nexus_science_v3/quarantine.jsonl"

with open(p, encoding="utf-8") as f:
    q = [json.loads(line) for line in f if line.strip()]

print("=== MOTIFS ===")
print(Counter(x.get("reason") for x in q))

print("\n=== 3 EXEMPLES ===")
for x in q[:3]:
    print(json.dumps(x, ensure_ascii=False, indent=2)[:5000])

In [ ]:
import json
from collections import Counter

p = "/kaggle/working/nexus_works_resolved.jsonl"

with open(p, encoding="utf-8") as f:
    works = [json.loads(line) for line in f if line.strip()]

print("=== STATUTS CACHE ===")
print(Counter(x.get("status") for x in works))

print("\n=== STRUCTURE ===")
if works:
    print("Clés :", list(works[0].keys()))
    print("Exemple :", json.dumps(works[0], ensure_ascii=False, indent=2)[:2500])

print("\n=== IDENTIFIANTS ===")
print("IDs présents :", sum(bool(x.get("id")) for x in works))
print("Cache total :", len(works))

In [ ]:
import os, json, hashlib

BASE = "/kaggle/working"
PACKAGE = os.path.join(BASE, "nexus_science_v3")
os.makedirs(PACKAGE, exist_ok=True)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

editions = load_jsonl(BASE + "/nexus_pilot_classified_v2.jsonl")
works_cache = load_jsonl(BASE + "/nexus_works_resolved.jsonl")
authors_cache = load_jsonl(BASE + "/nexus_authors_resolved.jsonl")

work_map = {x["id"]: x for x in works_cache if x.get("id")}
author_map = {x["id"]: x for x in authors_cache if x.get("id")}

selected = [x for x in editions if x.get("classification") == "SCIENTIFIC"]

ready_editions = {}
ready_works = {}
ready_authors = {}
quarantine = []
warnings = []

for e in selected:
    eid = e.get("olid")
    work_ids = e.get("works") or []
    author_ids = e.get("authors") or []

    if isinstance(work_ids, str):
        work_ids = [work_ids]
    if isinstance(author_ids, str):
        author_ids = [author_ids]

    if not eid:
        quarantine.append({"reason": "MISSING_EDITION_ID", "edition": e})
        continue

    if len(work_ids) != 1:
        quarantine.append({
            "reason": "MISSING_OR_AMBIGUOUS_WORK",
            "edition_id": eid,
            "work_ids": work_ids,
            "edition": e
        })
        continue

    wid = work_ids[0]
    w = work_map.get(wid)

    # CORRECTION : le cache utilise OK, et non valid
    if not w or w.get("status") != "OK":
        quarantine.append({
            "reason": "UNRESOLVED_WORK",
            "edition_id": eid,
            "work_id": wid,
            "edition": e
        })
        continue

    ready_editions[eid] = e
    ready_works[wid] = w

    for aid in author_ids:
        a = author_map.get(aid)
        if a and a.get("status") == "OK":
            ready_authors[aid] = a
        else:
            warnings.append({
                "edition_id": eid,
                "author_id": aid,
                "reason": "AUTHOR_UNRESOLVED"
            })

files = {
    "editions.jsonl": list(ready_editions.values()),
    "works.jsonl": list(ready_works.values()),
    "authors.jsonl": list(ready_authors.values()),
    "quarantine.jsonl": quarantine,
    "warnings.jsonl": warnings
}

for name, rows in files.items():
    save_jsonl(os.path.join(PACKAGE, name), rows)

manifest = {
    "version": "V3",
    "classification": "V2 provisional",
    "status": "PACKAGE_PREPARED",
    "selected_input": len(selected),
    "editions_ready": len(ready_editions),
    "unique_works_ready": len(ready_works),
    "unique_authors_ready": len(ready_authors),
    "quarantined": len(quarantine),
    "warnings": len(warnings),
    "canonical_database_write": False,
    "files": {}
}

for name, rows in files.items():
    p = os.path.join(PACKAGE, name)
    manifest["files"][name] = {
        "rows": len(rows),
        "bytes": os.path.getsize(p),
        "sha256": sha256(p)
    }

with open(os.path.join(PACKAGE, "manifest_v3.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== GESTIONNAIRE NEXUS V3 CORRIGE ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nSTATUT :", manifest["status"])

In [ ]:
import json, os, hashlib

BASE = "/kaggle/working/nexus_science_v3"

files = ["manifest_v3.json", "editions.jsonl", "works.jsonl",
         "authors.jsonl", "quarantine.jsonl", "warnings.jsonl"]

print("=== NEXUS V4 — PRÉPARATION ===")

for f in files:
    path = os.path.join(BASE, f)
    if not os.path.exists(path):
        print("ABSENT:", f)
        continue

    with open(path, "rb") as stream:
        raw = stream.read()

    print(f"\n{f} | {len(raw)} bytes | SHA256:", hashlib.sha256(raw).hexdigest())

    if f.endswith(".json"):
        data = json.loads(raw)
        print(json.dumps(data, ensure_ascii=False, indent=2)[:3500])
    elif f.endswith(".jsonl") and raw:
        lines = raw.decode("utf-8").splitlines()
        print("Lignes:", len(lines))
        print("Exemple:", lines[0][:2500] if lines else "VIDE")

In [ ]:
import os, json, hashlib, uuid
from collections import defaultdict

BASE = "/kaggle/working/nexus_science_v3"
OUT = "/kaggle/working/nexus_science_v4"
os.makedirs(OUT, exist_ok=True)

with open(os.path.join(BASE, "manifest_v3.json"), encoding="utf-8") as f:
    manifest = json.load(f)

def load_jsonl(name):
    path = os.path.join(BASE, name)
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

names = ["editions.jsonl", "works.jsonl", "authors.jsonl",
          "quarantine.jsonl", "warnings.jsonl"]

loaded = {}
checks = {}

for name in names:
    path = os.path.join(BASE, name)
    expected = manifest["files"][name]
    actual_hash = sha256_file(path)
    rows = load_jsonl(name)

    checks[name] = {
        "hash_ok": actual_hash == expected["sha256"],
        "rows_ok": len(rows) == expected["rows"],
        "rows": len(rows),
        "sha256": actual_hash
    }
    loaded[name] = rows

V4_INPUT_VALID = all(
    c["hash_ok"] and c["rows_ok"] for c in checks.values()
)

print("=== NEXUS V4 — INPUT ===")
print("Validation intégrale :", V4_INPUT_VALID)
for name, c in checks.items():
    print(name, "|", c["rows"], "| hash:", c["hash_ok"],
          "| lignes:", c["rows_ok"])

if not V4_INPUT_VALID:
    raise RuntimeError("Paquet V3 invalide — génération V4 bloquée.")

editions = loaded["editions.jsonl"]
works_cache = loaded["works.jsonl"]
authors_cache = loaded["authors.jsonl"]
quarantine = loaded["quarantine.jsonl"]

works_index = {x["id"]: x["notice"] for x in works_cache}
authors_index = {x["id"]: x["notice"] for x in authors_cache}

print("\nPaquet V3 validé et chargé.")

In [ ]:
# Identifiants internes reproductibles
NEXUS_NAMESPACE = uuid.UUID("a20f3c70-37b4-4f4e-9f90-21b3b52c8421")

def nexus_uuid(entity, external_id):
    return str(uuid.uuid5(NEXUS_NAMESPACE, entity + ":" + external_id))

work_map = {}
author_map = {}
edition_map = {}

for w in works_cache:
    work_map[w["id"]] = nexus_uuid("work", w["id"])

for a in authors_cache:
    author_map[a["id"]] = nexus_uuid("agent", a["id"])

for e in editions:
    edition_map[e["olid"]] = nexus_uuid("edition", e["olid"])

# Plan Works
work_plan = []
for w in works_cache:
    notice = w["notice"]
    work_plan.append({
        "external_id": w["id"],
        "work_id": work_map[w["id"]],
        "work_type": "monograph",
        "title": notice.get("title"),
        "record_status": "provisional",
        "source": "Open Library",
        "operation": "PROPOSE_INSERT"
    })

# Plan Agents
author_plan = []
for a in authors_cache:
    notice = a["notice"]
    author_plan.append({
        "external_id": a["id"],
        "agent_id": author_map[a["id"]],
        "agent_type": "person_or_organization_unresolved",
        "name": notice.get("name"),
        "record_status": "provisional",
        "source": "Open Library",
        "operation": "PROPOSE_INSERT"
    })

# Plan Editions et relations
edition_plan = []
relation_plan = []

for e in editions:
    olid = e["olid"]
    linked_works = e.get("works", [])
    linked_authors = e.get("authors", [])

    edition_plan.append({
        "external_id": olid,
        "edition_id": edition_map[olid],
        "title_statement": e.get("title"),
        "edition_type": "other",
        "record_status": "provisional",
        "source": "Open Library",
        "operation": "PROPOSE_INSERT"
    })

    for wid in linked_works:
        if wid in work_map:
            relation_plan.append({
                "relation": "edition_work",
                "edition_id": edition_map[olid],
                "work_id": work_map[wid],
                "external_edition_id": olid,
                "external_work_id": wid
            })

    for aid in linked_authors:
        if aid in author_map:
            relation_plan.append({
                "relation": "edition_agent",
                "edition_id": edition_map[olid],
                "agent_id": author_map[aid],
                "role_code": "contributor",
                "external_edition_id": olid,
                "external_author_id": aid
            })

# Export fichiers de plan
def write_jsonl(name, data):
    path = os.path.join(OUT, name)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    return path

write_jsonl("works_plan.jsonl", work_plan)
write_jsonl("authors_plan.jsonl", author_plan)
write_jsonl("editions_plan.jsonl", edition_plan)
write_jsonl("relations_plan.jsonl", relation_plan)

print("=== V4 — PLANS GENERES ===")
print("Works :", len(work_plan))
print("Agents :", len(author_plan))
print("Editions :", len(edition_plan))
print("Relations :", len(relation_plan))
print("Dossier :", OUT)

In [ ]:
# Audit V4
issues = []

def check_duplicates(values, label):
    seen = set()
    dup = set()
    for v in values:
        if v in seen:
            dup.add(v)
        seen.add(v)
    if dup:
        issues.append(f"{label}: {len(dup)} doublons")

check_duplicates([x["work_id"] for x in work_plan], "Work UUID")
check_duplicates([x["agent_id"] for x in author_plan], "Agent UUID")
check_duplicates([x["edition_id"] for x in edition_plan], "Edition UUID")

check_duplicates([x["external_id"] for x in work_plan], "Work OLID")
check_duplicates([x["external_id"] for x in author_plan], "Author OLID")
check_duplicates([x["external_id"] for x in edition_plan], "Edition OLID")

# Références
missing_work_refs = [
    w for e in editions
    for w in e.get("works", [])
    if w not in work_map
]

missing_author_refs = [
    a for e in editions
    for a in e.get("authors", [])
    if a not in author_map
]

# UUID et relations
uuid_valid = all(
    uuid.UUID(x["work_id"]) for x in work_plan
) and all(
    uuid.UUID(x["agent_id"]) for x in author_plan
) and all(
    uuid.UUID(x["edition_id"]) for x in edition_plan
)

relation_valid = all(
    r["edition_id"] in set(edition_map.values())
    for r in relation_plan
)

report = {
    "version": "V4",
    "status": "DRY_RUN_PLAN_READY" if not issues and
              not missing_work_refs and not missing_author_refs and
              uuid_valid and relation_valid else "REVIEW_REQUIRED",
    "input_manifest_sha256": sha256_file(
        os.path.join(BASE, "manifest_v3.json")
    ),
    "counts": {
        "works": len(work_plan),
        "authors": len(author_plan),
        "editions": len(edition_plan),
        "relations": len(relation_plan),
        "quarantine_v3": len(quarantine)
    },
    "validation": {
        "input_hashes_valid": V4_INPUT_VALID,
        "duplicate_issues": issues,
        "missing_work_refs": len(missing_work_refs),
        "missing_author_refs": len(missing_author_refs),
        "uuid_valid": uuid_valid,
        "relations_valid": relation_valid
    },
    "canonical_database_write": False,
    "scientific_classification_certified": False
}

with open(os.path.join(OUT, "report_v4.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== NEXUS SCIENCE V4 ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nSTATUT :", report["status"])
print("ECRITURE NEON :", report["canonical_database_write"])
print("Rapport :", os.path.join(OUT, "report_v4.json"))

In [ ]:
# V4.1 — expressions et relations correctes
expression_map = {
    wid: nexus_uuid("expression", wid)
    for wid in work_map
}

expression_plan = []
title_plan = []
edition_work_plan = []
edition_expression_plan = []
work_agent_plan = []

def norm_title(s):
    return " ".join(str(s or "").casefold().strip().split())

for wid, notice in works_index.items():
    eid = expression_map[wid]
    title = notice.get("title", "")

    expression_plan.append({
        "expression_id": eid,
        "work_id": work_map[wid],
        "expression_type": "original",
        "record_status": "provisional"
    })

    if title.strip():
        title_plan.append({
            "expression_title_id": nexus_uuid("expression_title", wid),
            "expression_id": eid,
            "title_type": "preferred",
            "title_text": title.strip(),
            "normalized_title": norm_title(title),
            "is_preferred": True,
            "sequence_no": 1
        })

for e in editions:
    olid = e["olid"]
    for wid in e.get("works", []):
        if wid not in work_map:
            continue

        edition_work_plan.append({
            "edition_id": edition_map[olid],
            "work_id": work_map[wid],
            "content_role": "main",
            "sequence_no": 1
        })

        edition_expression_plan.append({
            "edition_id": edition_map[olid],
            "work_id": work_map[wid],
            "expression_id": expression_map[wid]
        })

for wid, notice in works_index.items():
    for i, role in enumerate(notice.get("authors", []), 1):
        aid = role.get("author", {}).get("key")
        if aid in author_map:
            work_agent_plan.append({
                "work_id": work_map[wid],
                "agent_id": author_map[aid],
                "role_code": "author",
                "sequence_no": i
            })

write_jsonl("expressions_plan.jsonl", expression_plan)
write_jsonl("titles_plan.jsonl", title_plan)
write_jsonl("edition_works_plan.jsonl", edition_work_plan)
write_jsonl("edition_expressions_plan.jsonl", edition_expression_plan)
write_jsonl("work_agents_plan.jsonl", work_agent_plan)

print("Expressions :", len(expression_plan))
print("Titres :", len(title_plan))
print("Edition-Work :", len(edition_work_plan))
print("Edition-Expression :", len(edition_expression_plan))
print("Work-Agent :", len(work_agent_plan))

In [ ]:
# V4.1 — identifiants externes + noms + provenance

agent_name_plan = []
identifier_plan = []
source_record_plan = []

for a in authors_cache:
    aid = a["id"]
    name = a["notice"].get("name", "")
    if name.strip():
        agent_name_plan.append({
            "agent_name_id": nexus_uuid("agent_name", aid),
            "agent_id": author_map[aid],
            "name_type": "preferred",
            "name_text": name.strip(),
            "normalized_name": norm_title(name),
            "is_preferred": True
        })

for w in works_cache:
    identifier_plan.append({
        "entity_type": "work",
        "entity_id": work_map[w["id"]],
        "scheme_code": "OLID",
        "identifier_value": w["id"].split("/")[-1],
        "normalized_value": w["id"].split("/")[-1]
    })

for a in authors_cache:
    identifier_plan.append({
        "entity_type": "agent",
        "entity_id": author_map[a["id"]],
        "scheme_code": "OLID",
        "identifier_value": a["id"].split("/")[-1],
        "normalized_value": a["id"].split("/")[-1]
    })

for e in editions:
    identifier_plan.append({
        "entity_type": "edition",
        "entity_id": edition_map[e["olid"]],
        "scheme_code": "OLID",
        "identifier_value": e["olid"].split("/")[-1],
        "normalized_value": e["olid"].split("/")[-1]
    })

# Empreintes des notices sources
for e in editions:
    payload = json.dumps(e, sort_keys=True, ensure_ascii=False)
    source_record_plan.append({
        "entity_type": "edition",
        "external_record_id": e["olid"],
        "payload_sha256": hashlib.sha256(
            payload.encode("utf-8")
        ).hexdigest(),
        "payload": e
    })

write_jsonl("agent_names_plan.jsonl", agent_name_plan)
write_jsonl("identifiers_plan.jsonl", identifier_plan)
write_jsonl("source_records_plan.jsonl", source_record_plan)

print("Noms :", len(agent_name_plan))
print("Identifiants :", len(identifier_plan))
print("Provenance Editions :", len(source_record_plan))
print("Source :", "Open Library")

In [ ]:
# V4.1 — audit final du plan structurel

audit_v41 = {
    "input_valid": V4_INPUT_VALID,
    "works": len(work_plan),
    "agents": len(author_plan),
    "editions": len(edition_plan),
    "expressions": len(expression_plan),
    "titles": len(title_plan),
    "edition_works": len(edition_work_plan),
    "edition_expressions": len(edition_expression_plan),
    "work_agents": len(work_agent_plan),
    "agent_names": len(agent_name_plan),
    "identifiers": len(identifier_plan),
    "source_records_proposed": len(source_record_plan),
    "quarantine": len(quarantine)
}

# Vérification des références
audit_v41["broken_edition_work"] = sum(
    1 for r in edition_work_plan
    if r["edition_id"] not in set(edition_map.values())
    or r["work_id"] not in set(work_map.values())
)

audit_v41["broken_expression"] = sum(
    1 for r in edition_expression_plan
    if r["expression_id"] not in set(expression_map.values())
)

audit_v41["broken_work_agent"] = sum(
    1 for r in work_agent_plan
    if r["work_id"] not in set(work_map.values())
    or r["agent_id"] not in set(author_map.values())
)

audit_v41["canonical_database_write"] = False
audit_v41["status"] = (
    "V4_1_PLAN_READY"
    if audit_v41["input_valid"]
    and audit_v41["broken_edition_work"] == 0
    and audit_v41["broken_expression"] == 0
    and audit_v41["broken_work_agent"] == 0
    else "REVIEW_REQUIRED"
)

with open(os.path.join(OUT, "report_v4_1.json"), "w", encoding="utf-8") as f:
    json.dump(audit_v41, f, ensure_ascii=False, indent=2)

print("=== NEXUS V4.1 ===")
print(json.dumps(audit_v41, ensure_ascii=False, indent=2))
print("\nSTATUT :", audit_v41["status"])
print("NEON WRITE :", audit_v41["canonical_database_write"])

In [ ]:
from pathlib import Path
import json, hashlib

BASE = Path("/kaggle/working/nexus_science_v4_1")
OUT = Path("/kaggle/working/nexus_science_v4_2")
OUT.mkdir(parents=True, exist_ok=True)

required = [
    "works_plan.jsonl",
    "authors_plan.jsonl",
    "editions_plan.jsonl",
    "expressions_plan.jsonl",
    "titles_plan.jsonl",
    "edition_works_plan.jsonl",
    "edition_expressions_plan.jsonl",
    "work_agents_plan.jsonl",
    "agent_names_plan.jsonl",
    "identifiers_plan.jsonl",
    "source_records_plan.jsonl",
]

# Recherche automatique des fichiers V4.1
ROOT = Path("/kaggle/working")
files = {}

for name in required:
    matches = list(ROOT.rglob(name))
    if matches:
        files[name] = matches[0]

missing = [n for n in required if n not in files]

print("=== V4.2 PREPARATION ===")
print("Fichiers trouvés :", len(files), "/", len(required))
print("Fichiers manquants :", missing)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

if missing:
    raise RuntimeError("Plans V4.1 incomplets. Arrêt.")

plans = {name: load_jsonl(path) for name, path in files.items()}

for name, records in plans.items():
    print(f"{name}: {len(records)}")

print("\nSTATUT : PLANS CHARGÉS")

In [ ]:
import json
from collections import Counter

identifiers = plans["identifiers_plan.jsonl"]
authors = plans["authors_plan.jsonl"]

print("=== STRUCTURE IDENTIFIANTS ===")
if identifiers:
    print("Clés :", list(identifiers[0].keys()))
    print("Exemple :", json.dumps(identifiers[0], ensure_ascii=False, indent=2))

print("\n=== STRUCTURE AUTEURS ===")
if authors:
    print("Clés :", list(authors[0].keys()))
    print("Exemple :", json.dumps(authors[0], ensure_ascii=False, indent=2))

print("\n=== ANALYSE IDENTIFIANTS ===")

scheme_counts = Counter(
    str(r.get("scheme_code", r.get("scheme", "INCONNU")))
    for r in identifiers
)

print("Répartition par schéma :")
for k, v in scheme_counts.items():
    print(k, ":", v)

# Détection des lignes comportant une référence agent
agent_fields = {"agent_id", "agent_uuid"}

agent_identifier_rows = [
    r for r in identifiers
    if any(r.get(k) for k in agent_fields)
]

print("\nIdentifiants liés aux agents :", len(agent_identifier_rows))

# OLID d'agents à exclure du plan bibliographique
agent_olid_rows = [
    r for r in agent_identifier_rows
    if str(r.get("scheme_code", r.get("scheme", ""))).lower() == "olid"
]

print("OLID d'agents détectés :", len(agent_olid_rows))

# Séparation sans suppression du plan original
identifiers_v42 = [
    r for r in identifiers
    if r not in agent_olid_rows
]

print("\nIdentifiants conservés :", len(identifiers_v42))
print("OLID d'agents isolés :", len(agent_olid_rows))

print("\nSTATUT : AUDIT TERMINÉ")

In [ ]:
import json, hashlib, uuid
from pathlib import Path
from collections import Counter

OUT = Path("/kaggle/working/nexus_science_v4_2")
OUT.mkdir(parents=True, exist_ok=True)

def write_jsonl(name, records):
    path = OUT / name
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

# 1. Normalisation identifiants bibliographiques
identifiers_clean = []

for r in identifiers_v42:
    row = dict(r)
    row["scheme_code"] = str(row["scheme_code"]).lower()
    identifiers_clean.append(row)

# 2. Provenance séparée pour les auteurs
author_source_map = []

for a in authors:
    ext = str(a["external_id"]).strip()
    external_id = ext.split("/")[-1]

    author_source_map.append({
        "agent_id": a["agent_id"],
        "source_code": "openlibrary",
        "external_record_id": ext,
        "external_key": external_id,
        "source": a.get("source", "Open Library"),
        "name_snapshot": a.get("name"),
        "record_snapshot": a,
        "operation": "PROPOSE_LINK"
    })

# 3. Contrôles UUID
def valid_uuid(value):
    try:
        uuid.UUID(str(value))
        return True
    except Exception:
        return False

uuid_checks = {
    "works": all(valid_uuid(r["work_id"]) for r in plans["works_plan.jsonl"]),
    "authors": all(valid_uuid(r["agent_id"]) for r in authors),
    "editions": all(valid_uuid(r["edition_id"]) for r in plans["editions_plan.jsonl"]),
    "expressions": all(valid_uuid(r["expression_id"]) for r in plans["expressions_plan.jsonl"]),
    "identifiers": all(valid_uuid(r["entity_id"]) for r in identifiers_clean)
}

# 4. Vérification OLID bibliographiques
olid_checks = all(
    r["scheme_code"] == "olid"
    and r["entity_type"] in ("work", "edition", "expression", "digital_asset")
    and bool(r["identifier_value"])
    and bool(r["normalized_value"])
    for r in identifiers_clean
)

# 5. Vérification références auteurs
author_external_ids = [r["external_record_id"] for r in author_source_map]
author_agent_ids = [r["agent_id"] for r in author_source_map]

author_checks = {
    "external_ids_present": all(author_external_ids),
    "agent_ids_valid": all(valid_uuid(x) for x in author_agent_ids),
    "unique_external_ids": len(author_external_ids) == len(set(author_external_ids)),
    "unique_agent_ids": len(author_agent_ids) == len(set(author_agent_ids))
}

# 6. Vérification identifiants en double
identifier_keys = [
    (r["entity_type"], r["entity_id"], r["scheme_code"], r["normalized_value"])
    for r in identifiers_clean
]

duplicate_identifiers = len(identifier_keys) - len(set(identifier_keys))

# 7. Export package
for name, records in plans.items():
    write_jsonl(name, records)

write_jsonl("identifiers_plan_v42.jsonl", identifiers_clean)
write_jsonl("author_source_map_v42.jsonl", author_source_map)

# 8. Rapport
report = {
    "version": "V4.2",
    "status": "TECHNICAL_VALIDATION",
    "canonical_database_write": False,
    "scientific_classification_certified": False,
    "counts": {
        "works": len(plans["works_plan.jsonl"]),
        "authors": len(authors),
        "editions": len(plans["editions_plan.jsonl"]),
        "expressions": len(plans["expressions_plan.jsonl"]),
        "identifiers": len(identifiers_clean),
        "author_source_map": len(author_source_map)
    },
    "uuid_checks": uuid_checks,
    "olid_checks": olid_checks,
    "author_checks": author_checks,
    "duplicate_identifiers": duplicate_identifiers,
    "source_records_v41": len(plans["source_records_plan.jsonl"]),
    "author_provenance_separated": True
}

report["technical_validation"] = (
    all(uuid_checks.values())
    and olid_checks
    and all(author_checks.values())
    and duplicate_identifiers == 0
)

report["status"] = (
    "V4_2_PACKAGE_VALIDATED"
    if report["technical_validation"]
    else "V4_2_REVIEW_REQUIRED"
)

with open(OUT / "report_v42.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

manifest = {}

for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name != "manifest_v42.json":
        manifest[p.name] = {
            "sha256": sha256_file(p),
            "bytes": p.stat().st_size
        }

with open(OUT / "manifest_v42.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== RAPPORT V4.2 ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\n=== MANIFEST V4.2 ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nPACKAGE :", OUT)

In [ ]:
from collections import Counter
import json

print("=== TYPES D'ENTITÉS ===")
print(Counter(r.get("entity_type") for r in identifiers_clean))

print("\n=== SCHEMAS ===")
print(Counter(r.get("scheme_code") for r in identifiers_clean))

print("\n=== VALEURS VIDES ===")
for field in ("entity_type", "entity_id", "scheme_code",
               "identifier_value", "normalized_value"):
    bad = [r for r in identifiers_clean if not r.get(field)]
    print(field, ":", len(bad))

print("\n=== ENTITÉS INCOMPATIBLES ===")
allowed = {"work", "edition", "expression", "digital_asset"}

bad_entities = [
    r for r in identifiers_clean
    if r.get("entity_type") not in allowed
]

print("Total :", len(bad_entities))
for r in bad_entities[:15]:
    print(json.dumps(r, ensure_ascii=False))

print("\n=== IDENTIFIANTS OLID ===")
bad_olid = [
    r for r in identifiers_clean
    if r.get("scheme_code") != "olid"
    or r.get("entity_type") not in allowed
    or not r.get("identifier_value")
    or not r.get("normalized_value")
]

print("Total en échec :", len(bad_olid))
for r in bad_olid[:15]:
    print(json.dumps(r, ensure_ascii=False, indent=2))

print("\nSTATUT : DIAGNOSTIC")

In [ ]:
from pathlib import Path
import json, hashlib

OUT = Path("/kaggle/working/nexus_science_v4_2")

# Séparation exacte par type d'entité
bibliographic_identifiers_v42 = [
    r for r in identifiers_clean
    if r["entity_type"] in {"work", "edition", "expression", "digital_asset"}
]

agent_identifiers_v42 = [
    r for r in identifiers_clean
    if r["entity_type"] == "agent"
]

# Vérification du partage
assert len(bibliographic_identifiers_v42) == 724
assert len(agent_identifiers_v42) == 289
assert len(bibliographic_identifiers_v42) + len(agent_identifiers_v42) == len(identifiers_clean)

# Validation bibliographique
bibliographic_valid = all(
    r["scheme_code"] == "olid"
    and r["entity_type"] in {"work", "edition", "expression", "digital_asset"}
    and r["identifier_value"]
    and r["normalized_value"]
    for r in bibliographic_identifiers_v42
)

# Vérification agents
agent_valid = all(
    r["scheme_code"] == "olid"
    and r["entity_type"] == "agent"
    and r["identifier_value"]
    and r["normalized_value"]
    for r in agent_identifiers_v42
)

# Export séparé
def export(name, records):
    p = OUT / name
    with open(p, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return p

export("bibliographic_identifiers_v42.jsonl", bibliographic_identifiers_v42)
export("agent_olid_provenance_v42.jsonl", agent_identifiers_v42)

# Rapport final technique
report_v42_final = {
    "version": "V4.2.2",
    "status": "V4_2_2_PACKAGE_VALIDATED" if bibliographic_valid and agent_valid else "REVIEW_REQUIRED",
    "canonical_database_write": False,
    "scientific_classification_certified": False,
    "counts": {
        "works": 353,
        "authors": 289,
        "editions": 371,
        "expressions": 353,
        "bibliographic_identifiers": len(bibliographic_identifiers_v42),
        "agent_olid_provenance": len(agent_identifiers_v42)
    },
    "checks": {
        "bibliographic_identifiers_valid": bibliographic_valid,
        "agent_olid_preserved": agent_valid,
        "all_original_identifiers_accounted_for": (
            len(bibliographic_identifiers_v42) + len(agent_identifiers_v42)
            == len(identifiers_clean)
        ),
        "original_v41_plans_preserved": True,
        "canonical_write": False
    }
}

with open(OUT / "report_v42_final.json", "w", encoding="utf-8") as f:
    json.dump(report_v42_final, f, ensure_ascii=False, indent=2)

print("=== V4.2.2 FINAL ===")
print(json.dumps(report_v42_final, ensure_ascii=False, indent=2))
print("\nFICHIERS :")
print("Bibliographie :", OUT / "bibliographic_identifiers_v42.jsonl")
print("Agents :", OUT / "agent_olid_provenance_v42.jsonl")
print("Rapport :", OUT / "report_v42_final.json")

In [ ]:
import json
from collections import Counter

print("=== V4.3 — AUDIT DES ENUMS ===")

def values(records, field):
    return Counter(r.get(field) for r in records)

enum_fields = {
    "works_plan.jsonl": ["work_type", "record_status"],
    "authors_plan.jsonl": ["agent_type", "record_status"],
    "editions_plan.jsonl": ["edition_type", "record_status"],
    "expressions_plan.jsonl": ["expression_type", "record_status"],
    "titles_plan.jsonl": ["title_type", "language_code"],
    "edition_works_plan.jsonl": ["content_role"],
    "work_agents_plan.jsonl": ["role_code"],
    "identifiers_plan_v42.jsonl": ["entity_type", "scheme_code"],
}

for filename, fields in enum_fields.items():
    records = plans.get(filename, [])
    if filename == "identifiers_plan_v42.jsonl":
        records = identifiers_clean
    print("\n", filename, ":", len(records))
    for field in fields:
        print(field, "=>", values(records, field))

print("\n=== EXEMPLES STRUCTURELS ===")

for filename in [
    "works_plan.jsonl",
    "authors_plan.jsonl",
    "editions_plan.jsonl",
    "expressions_plan.jsonl",
    "titles_plan.jsonl",
    "edition_works_plan.jsonl",
    "edition_expressions_plan.jsonl",
    "work_agents_plan.jsonl",
    "agent_names_plan.jsonl",
    "source_records_plan.jsonl",
]:
    records = plans.get(filename, [])
    if records:
        print("\n", filename)
        print(json.dumps(records[0], ensure_ascii=False, indent=2))

print("\nSTATUT : AUDIT DES TYPES")

In [ ]:
import json
from collections import Counter

print("=== V4.3 — AUDIT RELATIONNEL ===")

works = plans["works_plan.jsonl"]
authors = plans["authors_plan.jsonl"]
editions = plans["editions_plan.jsonl"]
expressions = plans["expressions_plan.jsonl"]
titles = plans["titles_plan.jsonl"]
ew = plans["edition_works_plan.jsonl"]
ee = plans["edition_expressions_plan.jsonl"]
wa = plans["work_agents_plan.jsonl"]
an = plans["agent_names_plan.jsonl"]
sr = plans["source_records_plan.jsonl"]

# Sets de références
work_ids = {r["work_id"] for r in works}
agent_ids = {r["agent_id"] for r in authors}
edition_ids = {r["edition_id"] for r in editions}
expression_ids = {r["expression_id"] for r in expressions}

# Contrôle des références
checks = {
    "edition_works_work_refs": all(r["work_id"] in work_ids for r in ew),
    "edition_works_edition_refs": all(r["edition_id"] in edition_ids for r in ew),
    "edition_expressions_work_refs": all(r["work_id"] in work_ids for r in ee),
    "edition_expressions_edition_refs": all(r["edition_id"] in edition_ids for r in ee),
    "edition_expressions_expression_refs": all(r["expression_id"] in expression_ids for r in ee),
    "work_agents_work_refs": all(r["work_id"] in work_ids for r in wa),
    "work_agents_agent_refs": all(r["agent_id"] in agent_ids for r in wa),
    "agent_names_agent_refs": all(r["agent_id"] in agent_ids for r in an),
}

# Unicité des relations
relation_duplicates = {
    "edition_works": len(ew) - len(set((r["edition_id"], r["work_id"]) for r in ew)),
    "edition_expressions": len(ee) - len(set((r["edition_id"], r["work_id"], r["expression_id"]) for r in ee)),
    "work_agents": len(wa) - len(set((r["work_id"], r["agent_id"], r["role_code"]) for r in wa)),
    "agent_names": len(an) - len(set(r["agent_name_id"] for r in an))
}

# Contrôle des langues
missing_languages = sum(
    1 for r in titles
    if not r.get("language_code")
)

# Contrôle des agents
agent_types = Counter(r.get("agent_type") for r in authors)

# Contrôle de provenance
source_types = Counter(r.get("entity_type") for r in sr)

report = {
    "relation_checks": checks,
    "relation_duplicates": relation_duplicates,
    "missing_title_languages": missing_languages,
    "agent_types": dict(agent_types),
    "source_record_entity_types": dict(source_types),
    "counts": {
        "works": len(works),
        "agents": len(authors),
        "editions": len(editions),
        "expressions": len(expressions),
        "edition_works": len(ew),
        "edition_expressions": len(ee),
        "work_agents": len(wa),
        "agent_names": len(an),
        "source_records": len(sr)
    }
}

print(json.dumps(report, ensure_ascii=False, indent=2))

print("\nSTATUT : AUDIT RELATIONNEL TERMINÉ")

In [ ]:
from pathlib import Path
import json, hashlib

AUDIT = Path("/kaggle/working/nexus_science_v4_3")
AUDIT.mkdir(parents=True, exist_ok=True)

issues = []

# 1. Langues manquantes
for r in titles:
    if not r.get("language_code"):
        issues.append({
            "severity": "BLOCKING",
            "category": "missing_title_language",
            "entity_id": r.get("expression_id"),
            "title": r.get("title_text"),
            "action": "Resolve language from source or retain unknown"
        })

# 2. Types d'agents non compatibles
for a in authors:
    if a.get("agent_type") == "person_or_organization_unresolved":
        issues.append({
            "severity": "BLOCKING",
            "category": "unresolved_agent_type",
            "entity_id": a.get("agent_id"),
            "external_id": a.get("external_id"),
            "name": a.get("name"),
            "action": "Resolve agent type or use an approved compatible provisional type"
        })

# 3. Provenance insuffisante pour works et agents
for entity_type in ("work", "agent"):
    issues.append({
        "severity": "REVIEW",
        "category": "source_record_coverage",
        "entity_type": entity_type,
        "action": "Prepare source record and source link plans"
    })

# 4. Classification scientifique non certifiée
issues.append({
    "severity": "REVIEW",
    "category": "scientific_classification",
    "records": 371,
    "classification_version": "V2",
    "action": "Review classification quality and false positives before certification"
})

# 5. Editions génériques
issues.append({
    "severity": "REVIEW",
    "category": "edition_type",
    "records": len(editions),
    "value": "other",
    "action": "Assess edition types from source metadata"
})

# 6. Résultats relationnels
relation_integrity = all(checks.values())
no_relation_duplicates = all(v == 0 for v in relation_duplicates.values())

report_v43 = {
    "version": "V4.3",
    "status": "AUDIT_COMPLETE_REQUIRES_SCHEMA_REMEDIATION",
    "canonical_database_write": False,
    "scientific_classification_certified": False,
    "relation_integrity": relation_integrity,
    "relation_duplicates_zero": no_relation_duplicates,
    "issue_counts": {
        "blocking": sum(i["severity"] == "BLOCKING" for i in issues),
        "review": sum(i["severity"] == "REVIEW" for i in issues)
    },
    "counts": {
        "works": len(works),
        "authors": len(authors),
        "editions": len(editions),
        "expressions": len(expressions),
        "relations": len(ew) + len(ee) + len(wa),
        "source_records": len(sr),
        "issues": len(issues)
    }
}

def export_json(name, data):
    p = AUDIT / name
    with open(p, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return p

export_json("issues_v43.json", issues)
export_json("report_v43.json", report_v43)

# Résumé SHA256 des deux fichiers
manifest = {}
for name in ("issues_v43.json", "report_v43.json"):
    p = AUDIT / name
    manifest[name] = {
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        "bytes": p.stat().st_size
    }

export_json("manifest_v43.json", manifest)

print("=== V4.3 FINAL ===")
print(json.dumps(report_v43, ensure_ascii=False, indent=2))
print("\n=== MANIFEST ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nPACKAGE :", AUDIT)

In [ ]:
from pathlib import Path
import json

OUT44 = Path("/kaggle/working/nexus_science_v4_4")
OUT44.mkdir(parents=True, exist_ok=True)

# Types autorisés par Neon
allowed_agent_types = {
    "person", "corporate_body", "family",
    "meeting", "jurisdiction", "other"
}

# Transformation de compatibilité réversible
agents_sql_plan = []
agent_type_audit = []

for a in authors:
    original_type = a.get("agent_type")
    compatible_type = (
        original_type if original_type in allowed_agent_types
        else "other"
    )

    row = dict(a)
    row["agent_type"] = compatible_type
    agents_sql_plan.append(row)

    agent_type_audit.append({
        "agent_id": a["agent_id"],
        "external_id": a["external_id"],
        "original_agent_type": original_type,
        "proposed_sql_agent_type": compatible_type,
        "classification_certified": False
    })

# Langues inconnues : NULL conservé
titles_sql_plan = []

for t in titles:
    row = dict(t)
    row["language_code"] = t.get("language_code") or None
    titles_sql_plan.append(row)

# Contrôles
agent_types_valid = all(
    r["agent_type"] in allowed_agent_types
    for r in agents_sql_plan
)

language_null_preserved = all(
    r["language_code"] is None
    for r in titles_sql_plan
    if not next(
        (x.get("language_code") for x in titles
         if x["expression_id"] == r["expression_id"]),
        None
    )
)

def save_jsonl(name, records):
    p = OUT44 / name
    with open(p, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return p

save_jsonl("agents_sql_plan_v44.jsonl", agents_sql_plan)
save_jsonl("agent_type_audit_v44.jsonl", agent_type_audit)
save_jsonl("titles_sql_plan_v44.jsonl", titles_sql_plan)

report44 = {
    "version": "V4.4",
    "stage": "SCHEMA_COMPATIBILITY_PREPARATION",
    "canonical_database_write": False,
    "scientific_classification_certified": False,
    "checks": {
        "agent_types_sql_compatible": agent_types_valid,
        "unknown_title_languages_preserved": language_null_preserved,
        "agent_original_types_preserved": True
    },
    "counts": {
        "agents": len(agents_sql_plan),
        "titles": len(titles_sql_plan),
        "unknown_languages": sum(
            1 for r in titles_sql_plan
            if r["language_code"] is None
        )
    }
}

with open(OUT44 / "report_v44.json", "w", encoding="utf-8") as f:
    json.dump(report44, f, ensure_ascii=False, indent=2)

print("=== V4.4 CELLULE 1 ===")
print(json.dumps(report44, ensure_ascii=False, indent=2))

In [ ]:
import hashlib, json
from collections import Counter

print("=== V4.4 — AUDIT PROVENANCE ===")

source_records = plans["source_records_plan.jsonl"]

hash_valid = 0
hash_invalid = 0
external_ids_valid = 0
external_ids_invalid = 0
payload_valid = 0
missing_works = 0
missing_authors = 0

source_audit = []

for r in source_records:
    payload = r.get("payload")
    expected_hash = r.get("payload_sha256", "")

    if isinstance(payload, dict):
        canonical = json.dumps(
            payload,
            sort_keys=True,
            ensure_ascii=False,
            separators=(",", ":")
        ).encode("utf-8")
        actual_hash = hashlib.sha256(canonical).hexdigest()
        payload_valid += 1
    else:
        actual_hash = None

    hash_ok = (
        actual_hash is not None
        and actual_hash.lower() == str(expected_hash).lower()
    )

    if hash_ok:
        hash_valid += 1
    else:
        hash_invalid += 1

    ext = r.get("external_record_id")
    ext_ok = isinstance(ext, str) and ext.strip() != ""

    if ext_ok:
        external_ids_valid += 1
    else:
        external_ids_invalid += 1

    works_refs = payload.get("works", []) if isinstance(payload, dict) else []
    authors_refs = payload.get("authors", []) if isinstance(payload, dict) else []

    if not works_refs:
        missing_works += 1

    if not authors_refs:
        missing_authors += 1

    source_audit.append({
        "external_record_id": ext,
        "payload_hash_expected": expected_hash,
        "payload_hash_calculated": actual_hash,
        "hash_valid": hash_ok,
        "external_id_valid": ext_ok,
        "works_refs_count": len(works_refs),
        "authors_refs_count": len(authors_refs)
    })

source_report = {
    "records": len(source_records),
    "payloads_valid": payload_valid,
    "hash_valid": hash_valid,
    "hash_invalid": hash_invalid,
    "external_ids_valid": external_ids_valid,
    "external_ids_invalid": external_ids_invalid,
    "records_without_work_refs": missing_works,
    "records_without_author_refs": missing_authors,
    "entity_types": dict(Counter(r.get("entity_type") for r in source_records)),
    "canonical_database_write": False
}

print(json.dumps(source_report, ensure_ascii=False, indent=2))

# Export audit local
with open(OUT44 / "source_audit_v44.json", "w", encoding="utf-8") as f:
    json.dump(source_audit, f, ensure_ascii=False, indent=2)

with open(OUT44 / "source_report_v44.json", "w", encoding="utf-8") as f:
    json.dump(source_report, f, ensure_ascii=False, indent=2)

print("\nSTATUT : AUDIT PROVENANCE")

In [ ]:
import hashlib, json
from collections import Counter

print("=== V4.4.3 — HASH DIAGNOSTIC ===")

def hash_payload(payload, sort_keys, ensure_ascii, compact):
    if compact:
        text = json.dumps(
            payload,
            sort_keys=sort_keys,
            ensure_ascii=ensure_ascii,
            separators=(",", ":")
        )
    else:
        text = json.dumps(
            payload,
            sort_keys=sort_keys,
            ensure_ascii=ensure_ascii
        )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

variants = {
    "sorted_unicode_compact": lambda p: hash_payload(p, True, False, True),
    "sorted_ascii_compact": lambda p: hash_payload(p, True, True, True),
    "unsorted_unicode_compact": lambda p: hash_payload(p, False, False, True),
    "unsorted_ascii_compact": lambda p: hash_payload(p, False, True, True),
    "sorted_unicode_default": lambda p: hash_payload(p, True, False, False),
    "sorted_ascii_default": lambda p: hash_payload(p, True, True, False),
    "unsorted_unicode_default": lambda p: hash_payload(p, False, False, False),
    "unsorted_ascii_default": lambda p: hash_payload(p, False, True, False),
}

matches = Counter()
samples = []

for r in source_records:
    payload = r.get("payload")
    expected = str(r.get("payload_sha256", "")).lower()

    if not isinstance(payload, dict):
        continue

    row_matches = []

    for name, fn in variants.items():
        actual = fn(payload).lower()
        if actual == expected:
            matches[name] += 1
            row_matches.append(name)

    if len(samples) < 5:
        samples.append({
            "external_record_id": r.get("external_record_id"),
            "expected_hash": expected,
            "matched_variants": row_matches
        })

print("=== MATCHES PAR MÉTHODE ===")
for name in variants:
    print(name, ":", matches[name])

print("\n=== ÉCHANTILLONS ===")
print(json.dumps(samples, ensure_ascii=False, indent=2))

best = max(matches.values()) if matches else 0

diagnostic = {
    "version": "V4.4.3",
    "records_tested": len(source_records),
    "methods_tested": len(variants),
    "matches_by_method": dict(matches),
    "best_match_count": best,
    "hash_reconciliation": (
        "METHOD_IDENTIFIED"
        if best == len(source_records)
        else "PARTIAL_MATCH"
        if best > 0
        else "METHOD_NOT_IDENTIFIED"
    ),
    "canonical_database_write": False
}

with open(OUT44 / "hash_diagnostic_v443.json", "w", encoding="utf-8") as f:
    json.dump(diagnostic, f, ensure_ascii=False, indent=2)

print("\n=== DIAGNOSTIC FINAL ===")
print(json.dumps(diagnostic, ensure_ascii=False, indent=2))

In [ ]:
import hashlib, json

print("=== V4.4.4 — AUDIT CORRIGÉ ===")

verified = []
missing_author_records = []

for r in source_records:
    payload = r["payload"]

    canonical = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")

    calculated = hashlib.sha256(canonical).hexdigest()
    expected = str(r.get("payload_sha256", "")).lower()

    verified.append({
        "external_record_id": r.get("external_record_id"),
        "hash_valid": calculated == expected
    })

    if not payload.get("authors"):
        missing_author_records.append({
            "external_record_id": r.get("external_record_id"),
            "payload": payload
        })

hash_ok = sum(x["hash_valid"] for x in verified)

report = {
    "version": "V4.4.4",
    "records": len(source_records),
    "hash_verified": hash_ok,
    "hash_failed": len(source_records) - hash_ok,
    "records_without_author_refs": len(missing_author_records),
    "missing_author_records_review": len(missing_author_records),
    "provenance_integrity": hash_ok == len(source_records),
    "canonical_database_write": False,
    "scientific_classification_certified": False
}

with open(OUT44 / "provenance_verified_v444.json", "w", encoding="utf-8") as f:
    json.dump(verified, f, ensure_ascii=False, indent=2)

with open(OUT44 / "records_without_authors_v444.json", "w", encoding="utf-8") as f:
    json.dump(missing_author_records, f, ensure_ascii=False, indent=2)

with open(OUT44 / "report_v444.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nSTATUT :", "PROVENANCE VALIDEE" if report["provenance_integrity"] else "PROVENANCE NON VALIDEE")

In [ ]:
import os, json, hashlib

print("=== V4.4.5 — INVENTAIRE ===")

base = OUT44

inventory = []

for name in sorted(os.listdir(base)):
    path = os.path.join(base, name)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            content = f.read()

        inventory.append({
            "filename": name,
            "bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest()
        })

print(json.dumps(inventory, ensure_ascii=False, indent=2))

print("\nFichiers :", len(inventory))
print("STATUT : INVENTAIRE EFFECTUE")

In [ ]:
import os

ROOT = "/kaggle/working"

targets = {
    "works_plan.jsonl",
    "authors_plan.jsonl",
    "editions_plan.jsonl",
    "expressions_plan.jsonl",
    "titles_plan.jsonl",
    "edition_works_plan.jsonl",
    "edition_expressions_plan.jsonl",
    "work_agents_plan.jsonl",
    "agent_names_plan.jsonl",
    "identifiers_plan.jsonl",
    "source_records_plan.jsonl",
    "agent_olid_provenance_v42.jsonl",
    "manifest_v3.json",
    "report_v4.json",
    "report_v44.json",
    "report_v444.json"
}

found = []

for root, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for name in files:
        if name in targets:
            found.append(os.path.join(root, name))

print("=== V4.4.6 — LOCALISATION ===")
for path in sorted(found):
    print(path)

print("\nFichiers cibles retrouvés :", len(found))
print("STATUT : LOCALISATION TERMINEE")

In [ ]:
import os, json, hashlib, shutil

print("=== V4.4.7 — CONSOLIDATION ===")

SRC = "/kaggle/working/nexus_science_v4_2"
AUDIT = "/kaggle/working/nexus_science_v4_4"
DEST = "/kaggle/working/nexus_science_v4_4_7"

os.makedirs(DEST, exist_ok=True)

required = [
    "works_plan.jsonl",
    "authors_plan.jsonl",
    "editions_plan.jsonl",
    "expressions_plan.jsonl",
    "titles_plan.jsonl",
    "edition_works_plan.jsonl",
    "edition_expressions_plan.jsonl",
    "work_agents_plan.jsonl",
    "agent_names_plan.jsonl",
    "identifiers_plan.jsonl",
    "source_records_plan.jsonl",
    "agent_olid_provenance_v42.jsonl"
]

missing = [f for f in required if not os.path.isfile(os.path.join(SRC, f))]

if missing:
    print("ERREUR — fichiers absents :", missing)
else:
    for name in required:
        shutil.copy2(os.path.join(SRC, name), DEST)

    audit_files = [
        "agents_sql_plan_v44.jsonl",
        "agent_type_audit_v44.jsonl",
        "titles_sql_plan_v44.jsonl",
        "provenance_verified_v444.json",
        "records_without_authors_v444.json",
        "report_v44.json",
        "report_v444.json",
        "hash_diagnostic_v443.json"
    ]

    copied_audits = []

    for name in audit_files:
        path = os.path.join(AUDIT, name)
        if os.path.isfile(path):
            shutil.copy2(path, DEST)
            copied_audits.append(name)

    manifest_files = {}

    for name in sorted(os.listdir(DEST)):
        path = os.path.join(DEST, name)

        if os.path.isfile(path):
            with open(path, "rb") as f:
                data = f.read()

            manifest_files[name] = {
                "bytes": len(data),
                "sha256": hashlib.sha256(data).hexdigest()
            }

    manifest = {
        "version": "V4.4.7",
        "status": "CONSOLIDATED_PRE_IMPORT_PACKAGE",
        "source_package": "V4.2.2",
        "audit_package": "V4.4.4",
        "files": manifest_files,
        "counts": {
            "works": 353,
            "authors": 289,
            "editions": 371,
            "expressions": 353,
            "authorless_source_records_review": 25
        },
        "provenance_verified": True,
        "sql_compatibility_prepared": True,
        "scientific_classification_certified": False,
        "canonical_database_write": False,
        "import_authorized": False
    }

    with open(os.path.join(DEST, "manifest_v447.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(json.dumps({
        "version": manifest["version"],
        "status": manifest["status"],
        "destination": DEST,
        "files_consolidated": len(manifest_files),
        "audit_files_copied": copied_audits,
        "provenance_verified": True,
        "canonical_database_write": False,
        "scientific_classification_certified": False,
        "import_authorized": False
    }, ensure_ascii=False, indent=2))

    print("\nSTATUT : CONSOLIDATION TERMINEE")

In [ ]:
import os, zipfile, hashlib

SOURCE = "/kaggle/working/nexus_science_v4_4_7"
ARCHIVE = "/kaggle/working/nexus_science_v4_4_7.zip"

with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(SOURCE):
        for name in files:
            path = os.path.join(root, name)
            arcname = os.path.relpath(path, SOURCE)
            z.write(path, arcname)

with open(ARCHIVE, "rb") as f:
    archive_hash = hashlib.sha256(f.read()).hexdigest()

print("=== V4.4.8 — ARCHIVE ===")
print("Archive :", ARCHIVE)
print("Taille :", os.path.getsize(ARCHIVE))
print("SHA-256 :", archive_hash)
print("STATUT : ARCHIVE CREEE")

In [ ]:
import os, json, hashlib

BASE = "/kaggle/working/nexus_science_v4_4_7"

print("=== V4.5.1 — INVENTAIRE DU PACKAGE ===")

if not os.path.isdir(BASE):
    raise FileNotFoundError(BASE)

files = sorted(os.listdir(BASE))
summary = []

for name in files:
    path = os.path.join(BASE, name)

    if not os.path.isfile(path):
        continue

    with open(path, "rb") as f:
        raw = f.read()

    rows = None

    if name.endswith(".jsonl"):
        rows = sum(1 for line in raw.splitlines() if line.strip())

    summary.append({
        "file": name,
        "bytes": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "jsonl_rows": rows
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nFichiers :", len(summary))
print("STATUT : INVENTAIRE V4.5.1")

In [ ]:
import os, json

BASE = "/kaggle/working/nexus_science_v4_4_7"

expected = {
    "works_plan.jsonl": 353,
    "authors_plan.jsonl": 289,
    "agents_sql_plan_v44.jsonl": 289,
    "editions_plan.jsonl": 371,
    "expressions_plan.jsonl": 353,
    "titles_plan.jsonl": 353,
    "titles_sql_plan_v44.jsonl": 353,
    "edition_works_plan.jsonl": 371,
    "edition_expressions_plan.jsonl": 371,
    "work_agents_plan.jsonl": 381,
    "agent_names_plan.jsonl": 289,
    "identifiers_plan.jsonl": 724,
    "source_records_plan.jsonl": 371,
    "agent_olid_provenance_v42.jsonl": 289
}

results = []

for name, expected_count in expected.items():
    path = os.path.join(BASE, name)

    if not os.path.isfile(path):
        results.append({
            "file": name,
            "expected": expected_count,
            "actual": None,
            "status": "MISSING"
        })
        continue

    with open(path, "r", encoding="utf-8") as f:
        actual = sum(1 for line in f if line.strip())

    results.append({
        "file": name,
        "expected": expected_count,
        "actual": actual,
        "status": "OK" if actual == expected_count else "COUNT_MISMATCH"
    })

print("=== V4.5.2 — STRUCTURE ===")
print(json.dumps(results, ensure_ascii=False, indent=2))

passed = all(r["status"] == "OK" for r in results)

print("\nSTRUCTURE_VALID :", passed)
print("STATUT :", "STRUCTURE VALIDEE" if passed else "STRUCTURE BLOQUEE")

In [ ]:
import os, json

BASE = "/kaggle/working/nexus_science_v4_4_7"

print("=== V4.5.3 — INSPECTION DES PLANS ===")

inspection = {}

for name in sorted(os.listdir(BASE)):
    if not name.endswith(".jsonl"):
        continue

    path = os.path.join(BASE, name)

    with open(path, "r", encoding="utf-8") as f:
        first = next((line.strip() for line in f if line.strip()), None)

    if first is None:
        inspection[name] = {
            "rows": 0,
            "keys": [],
            "sample": None
        }
        continue

    try:
        obj = json.loads(first)

        inspection[name] = {
            "keys": list(obj.keys()) if isinstance(obj, dict) else [],
            "sample": obj
        }

    except Exception as e:
        inspection[name] = {
            "parse_error": str(e)
        }

print(json.dumps(inspection, ensure_ascii=False, indent=2))
print("\nSTATUT : INSPECTION TERMINEE")

In [ ]:
import os, json, collections

BASE = "/kaggle/working/nexus_science_v4_4_7"
OUT = "/kaggle/working/nexus_science_v4_5_preflight"
os.makedirs(OUT, exist_ok=True)

def read_jsonl(name):
    path = os.path.join(BASE, name)
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

identifiers = read_jsonl("identifiers_plan.jsonl")

counts = collections.Counter(r.get("entity_type") for r in identifiers)

bib = [r for r in identifiers if r.get("entity_type") in ("work", "edition", "expression", "digital_asset")]
agent_identifiers = [r for r in identifiers if r.get("entity_type") == "agent"]
unknown = [r for r in identifiers if r.get("entity_type") not in ("work", "edition", "expression", "digital_asset", "agent")]

agent_provenance = read_jsonl("agent_olid_provenance_v42.jsonl")

# Contrôle d'équivalence de la provenance des OLID d'agents
agent_key = lambda r: (r.get("entity_id"), r.get("scheme_code", "").lower(), r.get("normalized_value"))
keys_plan = {agent_key(r) for r in agent_identifiers}
keys_prov = {agent_key(r) for r in agent_provenance}

report = {
    "version": "V4.5.4",
    "total_identifiers_original": len(identifiers),
    "entity_type_counts": dict(counts),
    "bibliographic_identifiers": len(bib),
    "agent_identifiers_in_original_plan": len(agent_identifiers),
    "agent_provenance_separate_file": len(agent_provenance),
    "agent_identifier_sets_match": keys_plan == keys_prov,
    "agent_identifiers_missing_in_provenance": len(keys_plan - keys_prov),
    "agent_provenance_missing_in_plan": len(keys_prov - keys_plan),
    "unknown_entity_types": len(unknown),
    "reconciliation_passed": (
        len(bib) == 724 and
        len(agent_identifiers) == 289 and
        len(agent_provenance) == 289 and
        keys_plan == keys_prov and
        len(unknown) == 0
    ),
    "canonical_database_write": False
}

with open(os.path.join(OUT, "bibliographic_identifiers_preflight.jsonl"), "w", encoding="utf-8") as f:
    for r in bib:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(os.path.join(OUT, "agent_identifier_reconciliation_v454.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("=== V4.5.4 — IDENTIFIANTS ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("STATUT :", "RECONCILIATION OK" if report["reconciliation_passed"] else "RECONCILIATION BLOQUEE")

In [ ]:
import os, json, uuid, collections

BASE = "/kaggle/working/nexus_science_v4_4_7"
OUT = "/kaggle/working/nexus_science_v4_5_preflight"

def load(name):
    with open(os.path.join(BASE, name), encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

works = load("works_plan.jsonl")
agents = load("agents_sql_plan_v44.jsonl")
editions = load("editions_plan.jsonl")
expressions = load("expressions_plan.jsonl")
titles = load("titles_sql_plan_v44.jsonl")
ew = load("edition_works_plan.jsonl")
ee = load("edition_expressions_plan.jsonl")
wa = load("work_agents_plan.jsonl")
an = load("agent_names_plan.jsonl")
ids = load("bibliographic_identifiers_preflight.jsonl")
sr = load("source_records_plan.jsonl")

uuid_fields = {
    "works": [(r, "work_id") for r in works],
    "agents": [(r, "agent_id") for r in agents],
    "editions": [(r, "edition_id") for r in editions],
    "expressions": [(r, "expression_id") for r in expressions],
    "titles": [(r, "expression_title_id") for r in titles],
    "edition_works": [(r, "edition_id") for r in ew] + [(r, "work_id") for r in ew],
    "edition_expressions": [(r, "edition_id") for r in ee] + [(r, "work_id") for r in ee] + [(r, "expression_id") for r in ee],
    "work_agents": [(r, "work_id") for r in wa] + [(r, "agent_id") for r in wa],
    "agent_names": [(r, "agent_name_id") for r in an] + [(r, "agent_id") for r in an],
    "identifiers": [(r, "entity_id") for r in ids],
}

uuid_errors = {}
for table, entries in uuid_fields.items():
    bad = []
    for r, field in entries:
        try:
            uuid.UUID(str(r[field]))
        except Exception:
            bad.append(r.get(field))
    uuid_errors[table] = len(bad)

work_set = {r["work_id"] for r in works}
agent_set = {r["agent_id"] for r in agents}
edition_set = {r["edition_id"] for r in editions}
expression_set = {r["expression_id"] for r in expressions}

fk_errors = {
    "edition_works_missing_editions": sum(r["edition_id"] not in edition_set for r in ew),
    "edition_works_missing_works": sum(r["work_id"] not in work_set for r in ew),
    "edition_expressions_missing_editions": sum(r["edition_id"] not in edition_set for r in ee),
    "edition_expressions_missing_works": sum(r["work_id"] not in work_set for r in ee),
    "edition_expressions_missing_expressions": sum(r["expression_id"] not in expression_set for r in ee),
    "work_agents_missing_works": sum(r["work_id"] not in work_set for r in wa),
    "work_agents_missing_agents": sum(r["agent_id"] not in agent_set for r in wa),
    "titles_missing_expressions": sum(r["expression_id"] not in expression_set for r in titles),
    "agent_names_missing_agents": sum(r["agent_id"] not in agent_set for r in an),
    "identifiers_missing_targets": sum(
        r["entity_id"] not in (
            work_set if r["entity_type"] == "work" else
            edition_set if r["entity_type"] == "edition" else
            expression_set if r["entity_type"] == "expression" else set()
        )
        for r in ids
    )
}

duplicate_counts = {
    "work_ids": len(works) - len({r["work_id"] for r in works}),
    "agent_ids": len(agents) - len({r["agent_id"] for r in agents}),
    "edition_ids": len(editions) - len({r["edition_id"] for r in editions}),
    "expression_ids": len(expressions) - len({r["expression_id"] for r in expressions}),
    "edition_works": len(ew) - len({(r["edition_id"], r["work_id"]) for r in ew}),
    "edition_expressions": len(ee) - len({(r["edition_id"], r["work_id"], r["expression_id"]) for r in ee}),
    "work_agents": len(wa) - len({(r["work_id"], r["agent_id"], r["role_code"]) for r in wa}),
    "titles": len(titles) - len({r["expression_title_id"] for r in titles}),
    "agent_names": len(an) - len({r["agent_name_id"] for r in an}),
}

report = {
    "version": "V4.5.5",
    "uuid_errors": uuid_errors,
    "foreign_key_reference_errors": fk_errors,
    "duplicates": duplicate_counts,
    "counts": {
        "works": len(works),
        "agents": len(agents),
        "editions": len(editions),
        "expressions": len(expressions),
        "titles": len(titles),
        "edition_works": len(ew),
        "edition_expressions": len(ee),
        "work_agents": len(wa),
        "agent_names": len(an),
        "bibliographic_identifiers": len(ids),
        "source_records": len(sr)
    },
    "canonical_database_write": False
}

passed = (
    all(v == 0 for v in uuid_errors.values()) and
    all(v == 0 for v in fk_errors.values()) and
    all(v == 0 for v in duplicate_counts.values())
)

report["integrity_passed"] = passed

with open(os.path.join(OUT, "integrity_report_v455.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("=== V4.5.5 — INTEGRITE ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nSTATUT :", "INTEGRITE VALIDEE" if passed else "ANOMALIES DETECTEES")

In [ ]:
# === V4.5.5 — INTEGRITE / UUID / REFERENCES (CORRIGÉ) ===

import os, json, uuid
from collections import Counter

BASE = "/kaggle/working/nexus_science_v4_4_7"
OUT = "/kaggle/working/nexus_science_v4_5_preflight"
os.makedirs(OUT, exist_ok=True)

def load(name, folder=BASE):
    path = os.path.join(folder, name)
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

works = load("works_plan.jsonl")
agents = load("agents_sql_plan_v44.jsonl")
editions = load("editions_plan.jsonl")
expressions = load("expressions_plan.jsonl")
titles = load("titles_sql_plan_v44.jsonl")
ew = load("edition_works_plan.jsonl")
ee = load("edition_expressions_plan.jsonl")
wa = load("work_agents_plan.jsonl")
an = load("agent_names_plan.jsonl")

# Le fichier réconcilié est dans OUT
ids = load("bibliographic_identifiers_preflight.jsonl", OUT)

sr = load("source_records_plan.jsonl")

def getset(rows, key):
    return {r[key] for r in rows if r.get(key)}

work_ids = getset(works, "work_id")
agent_ids = getset(agents, "agent_id")
edition_ids = getset(editions, "edition_id")
expression_ids = getset(expressions, "expression_id")

uuid_errors = []
uuid_fields = [
    (works, ["work_id"]),
    (agents, ["agent_id"]),
    (editions, ["edition_id"]),
    (expressions, ["expression_id", "work_id"]),
    (titles, ["expression_title_id", "expression_id"]),
    (ew, ["edition_id", "work_id"]),
    (ee, ["edition_id", "work_id", "expression_id"]),
    (wa, ["work_id", "agent_id"]),
    (an, ["agent_name_id", "agent_id"]),
    (ids, ["entity_id"]),
]

for rows, fields in uuid_fields:
    for i, r in enumerate(rows):
        for field in fields:
            val = r.get(field)
            if val:
                try:
                    uuid.UUID(str(val))
                except Exception:
                    uuid_errors.append({
                        "file_field": field,
                        "row": i,
                        "value": val
                    })

missing_refs = []

for r in ew:
    if r.get("edition_id") not in edition_ids:
        missing_refs.append({"table": "edition_works", "field": "edition_id"})
    if r.get("work_id") not in work_ids:
        missing_refs.append({"table": "edition_works", "field": "work_id"})

for r in ee:
    if r.get("edition_id") not in edition_ids:
        missing_refs.append({"table": "edition_expressions", "field": "edition_id"})
    if r.get("work_id") not in work_ids:
        missing_refs.append({"table": "edition_expressions", "field": "work_id"})
    if r.get("expression_id") not in expression_ids:
        missing_refs.append({"table": "edition_expressions", "field": "expression_id"})

for r in wa:
    if r.get("work_id") not in work_ids:
        missing_refs.append({"table": "work_agents", "field": "work_id"})
    if r.get("agent_id") not in agent_ids:
        missing_refs.append({"table": "work_agents", "field": "agent_id"})

for r in titles:
    if r.get("expression_id") not in expression_ids:
        missing_refs.append({"table": "expression_titles", "field": "expression_id"})

for r in an:
    if r.get("agent_id") not in agent_ids:
        missing_refs.append({"table": "agent_names", "field": "agent_id"})

target_sets = {
    "work": work_ids,
    "edition": edition_ids,
    "expression": expression_ids
}

for r in ids:
    entity_type = r.get("entity_type")
    if entity_type in target_sets:
        if r.get("entity_id") not in target_sets[entity_type]:
            missing_refs.append({
                "table": "bibliographic_identifiers",
                "field": "entity_id",
                "entity_type": entity_type
            })
    else:
        missing_refs.append({
            "table": "bibliographic_identifiers",
            "field": "entity_type",
            "entity_type": entity_type
        })

def duplicate_count(rows, field):
    values = [r.get(field) for r in rows if r.get(field)]
    return len(values) - len(set(values))

duplicate_ids = {
    "works": duplicate_count(works, "work_id"),
    "agents": duplicate_count(agents, "agent_id"),
    "editions": duplicate_count(editions, "edition_id"),
    "expressions": duplicate_count(expressions, "expression_id"),
    "titles": duplicate_count(titles, "expression_title_id"),
    "agent_names": duplicate_count(an, "agent_name_id")
}

duplicate_relations = {
    "edition_works": len(ew) - len(set(
        (r.get("edition_id"), r.get("work_id")) for r in ew
    )),
    "edition_expressions": len(ee) - len(set(
        (r.get("edition_id"), r.get("expression_id")) for r in ee
    )),
    "work_agents": len(wa) - len(set(
        (r.get("work_id"), r.get("agent_id"), r.get("role_code"))
        for r in wa
    ))
}

passed = (
    len(uuid_errors) == 0
    and len(missing_refs) == 0
    and all(v == 0 for v in duplicate_ids.values())
    and all(v == 0 for v in duplicate_relations.values())
)

report = {
    "version": "V4.5.5",
    "uuid_errors": len(uuid_errors),
    "missing_references": len(missing_refs),
    "duplicate_ids": duplicate_ids,
    "duplicate_relations": duplicate_relations,
    "integrity_passed": passed,
    "canonical_database_write": False,
    "sql_execution": False
}

with open(os.path.join(OUT, "integrity_report_v455.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== V4.5.5 — INTEGRITE ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("STATUT :", "INTEGRITY VALIDATED" if passed else "BLOCKED")


In [ ]:
# === V4.5.6 — RAPPORT FINAL PRE-IMPORT ===

import os, json
from collections import Counter

BASE = "/kaggle/working/nexus_science_v4_4_7"
OUT = "/kaggle/working/nexus_science_v4_5_preflight"

def load_json(name):
    with open(os.path.join(OUT, name), encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(name, folder=BASE):
    with open(os.path.join(folder, name), encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

reconciliation = load_json("agent_identifier_reconciliation_v454.json")
integrity = load_json("integrity_report_v455.json")

works = load_jsonl("works_plan.jsonl")
agents = load_jsonl("agents_sql_plan_v44.jsonl")
editions = load_jsonl("editions_plan.jsonl")
expressions = load_jsonl("expressions_plan.jsonl")
titles = load_jsonl("titles_sql_plan_v44.jsonl")
ew = load_jsonl("edition_works_plan.jsonl")
ee = load_jsonl("edition_expressions_plan.jsonl")
wa = load_jsonl("work_agents_plan.jsonl")
an = load_jsonl("agent_names_plan.jsonl")
ids = load_jsonl("bibliographic_identifiers_preflight.jsonl", OUT)
sr = load_jsonl("source_records_plan.jsonl")

scheme_counts = Counter(str(r.get("scheme_code")) for r in ids)
edition_types = Counter(str(r.get("edition_type")) for r in editions)
agent_types = Counter(str(r.get("agent_type")) for r in agents)
work_types = Counter(str(r.get("work_type")) for r in works)
expression_types = Counter(str(r.get("expression_type")) for r in expressions)
language_counts = Counter(str(r.get("language_code")) for r in titles)

uppercase_schemes = [
    r for r in ids
    if str(r.get("scheme_code", "")) != str(r.get("scheme_code", "")).lower()
]

passed = (
    reconciliation.get("reconciliation_passed", False)
    and integrity.get("integrity_passed", False)
)

report = {
    "version": "V4.5.6",
    "status": "STRUCTURAL_CHECKS_PASSED" if passed else "BLOCKED",
    "counts": {
        "works": len(works),
        "agents": len(agents),
        "editions": len(editions),
        "expressions": len(expressions),
        "titles": len(titles),
        "edition_works": len(ew),
        "edition_expressions": len(ee),
        "work_agents": len(wa),
        "agent_names": len(an),
        "bibliographic_identifiers": len(ids),
        "source_records": len(sr)
    },
    "identifier_schemes": dict(scheme_counts),
    "uppercase_identifier_scheme_review": len(uppercase_schemes),
    "edition_types": dict(edition_types),
    "agent_types": dict(agent_types),
    "work_types": dict(work_types),
    "expression_types": dict(expression_types),
    "title_languages": dict(language_counts),
    "reconciliation_passed": reconciliation.get("reconciliation_passed"),
    "integrity_passed": integrity.get("integrity_passed"),
    "scientific_classification_certified": False,
    "sql_execution": False,
    "canonical_database_write": False,
    "import_authorized": False
}

with open(os.path.join(OUT, "preflight_report_v456.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== V4.5.6 — RAPPORT FINAL ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("STATUT :", report["status"])


In [ ]:
# === V4.6.1 — CONSOLIDATION / NORMALISATION ===

import os, json, shutil
from collections import Counter

SRC = "/kaggle/working/nexus_science_v4_4_7"
PRE = "/kaggle/working/nexus_science_v4_5_preflight"
OUT = "/kaggle/working/nexus_science_v4_6"

if os.path.exists(OUT):
    shutil.rmtree(OUT)

shutil.copytree(SRC, OUT)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

ids = read_jsonl(os.path.join(PRE, "bibliographic_identifiers_preflight.jsonl"))

# Normalisation du code de registre, pas des valeurs d'identifiants
for row in ids:
    row["scheme_code"] = str(row["scheme_code"]).lower()

write_jsonl(os.path.join(OUT, "bibliographic_identifiers_plan_v46.jsonl"), ids)

report = {
    "version": "V4.6.1",
    "source": SRC,
    "destination": OUT,
    "bibliographic_identifiers": len(ids),
    "scheme_counts": dict(Counter(r["scheme_code"] for r in ids)),
    "source_package_modified": False,
    "canonical_database_write": False,
    "sql_execution": False
}

with open(os.path.join(OUT, "report_v461.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== V4.6.1 ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("STATUT : PAQUET COPIE ET NORMALISE")


In [ ]:
# === V4.6.2 — INVENTAIRE CRYPTOGRAPHIQUE ===

import os, json, hashlib

OUT = "/kaggle/working/nexus_science_v4_6"

inventory = []

for root, dirs, files in os.walk(OUT):
    dirs.sort()
    files.sort()
    for name in files:
        path = os.path.join(root, name)
        rel = os.path.relpath(path, OUT)

        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)

        inventory.append({
            "file": rel,
            "size_bytes": os.path.getsize(path),
            "sha256": h.hexdigest()
        })

manifest = {
    "version": "V4.6.2",
    "file_count": len(inventory),
    "files": inventory,
    "canonical_database_write": False
}

manifest_path = os.path.join(OUT, "manifest_v462.json")

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("=== V4.6.2 ===")
print("Fichiers inventoriés :", len(inventory))
print("Manifest :", manifest_path)
print("STATUT : INVENTAIRE GENERE")


In [ ]:
# === V4.6.3 — CONTROLE FINAL DU PAQUET ===

import os, json

OUT = "/kaggle/working/nexus_science_v4_6"

def read_jsonl(name):
    with open(os.path.join(OUT, name), encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

checks = {
    "works": len(read_jsonl("works_plan.jsonl")) == 353,
    "agents": len(read_jsonl("agents_sql_plan_v44.jsonl")) == 289,
    "editions": len(read_jsonl("editions_plan.jsonl")) == 371,
    "expressions": len(read_jsonl("expressions_plan.jsonl")) == 353,
    "titles": len(read_jsonl("titles_sql_plan_v44.jsonl")) == 353,
    "edition_works": len(read_jsonl("edition_works_plan.jsonl")) == 371,
    "edition_expressions": len(read_jsonl("edition_expressions_plan.jsonl")) == 371,
    "work_agents": len(read_jsonl("work_agents_plan.jsonl")) == 381,
    "agent_names": len(read_jsonl("agent_names_plan.jsonl")) == 289,
    "bibliographic_identifiers": len(read_jsonl("bibliographic_identifiers_plan_v46.jsonl")) == 724,
    "source_records": len(read_jsonl("source_records_plan.jsonl")) == 371
}

ids = read_jsonl("bibliographic_identifiers_plan_v46.jsonl")

checks["scheme_normalized"] = all(
    r.get("scheme_code") == "olid" for r in ids
)

passed = all(checks.values())

report = {
    "version": "V4.6.3",
    "checks": checks,
    "checks_passed": sum(checks.values()),
    "checks_total": len(checks),
    "status": "PACKAGE_READY_FOR_SQL_PREPARATION" if passed else "BLOCKED",
    "sql_execution": False,
    "canonical_database_write": False,
    "import_authorized": False
}

with open(os.path.join(OUT, "report_v463.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== V4.6.3 ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("STATUT :", report["status"])


In [ ]:
import os, json

BASE = "/kaggle/working/nexus_science_v4_6"
PRE = "/kaggle/working/nexus_science_v4_5_preflight"
OUT = "/kaggle/working/nexus_science_v4_6"

FILES = {
    "works": os.path.join(BASE, "works_plan.jsonl"),
    "agents": os.path.join(BASE, "agents_sql_plan_v44.jsonl"),
    "editions": os.path.join(BASE, "editions_plan.jsonl"),
    "expressions": os.path.join(BASE, "expressions_plan.jsonl"),
    "titles": os.path.join(BASE, "titles_sql_plan_v44.jsonl"),
    "edition_works": os.path.join(BASE, "edition_works_plan.jsonl"),
    "edition_expressions": os.path.join(BASE, "edition_expressions_plan.jsonl"),
    "work_agents": os.path.join(BASE, "work_agents_plan.jsonl"),
    "agent_names": os.path.join(BASE, "agent_names_plan.jsonl"),
    "identifiers": os.path.join(BASE, "bibliographic_identifiers_plan_v46.jsonl"),
    "source_records": os.path.join(BASE, "source_records_plan.jsonl"),
}

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

print("=== V4.6.6 — INSPECTION ===")
for name, path in FILES.items():
    if os.path.exists(path):
        rows = load_jsonl(path)
        print(f"\n{name}: {len(rows)}")
        if rows:
            print("Champs:", list(rows[0].keys()))
            print("Exemple:", json.dumps(rows[0], ensure_ascii=False)[:500])
    else:
        print(f"\nABSENT: {name} -> {path}")


In [ ]:
import uuid, json, os, hashlib

SOURCE_SYSTEM_ID = "67d4c7e7-a324-49ef-9f60-fc24ca4af926"
NAMESPACE = uuid.UUID("67d4c7e7-a324-49ef-9f60-fc24ca4af926")

def deterministic_uuid(label):
    return str(uuid.uuid5(NAMESPACE, label))

data = {k: load_jsonl(p) for k, p in FILES.items() if os.path.exists(p)}

# Mapping des IDs externes vers IDs techniques existants dans les plans
id_maps = {}
for entity in ["works", "agents", "editions"]:
    rows = data.get(entity, [])
    id_maps[entity] = {
        r["external_id"]: r.get(
            "work_id" if entity == "works" else
            "agent_id" if entity == "agents" else "edition_id"
        )
        for r in rows if r.get("external_id")
    }

# Identifiants déterministes des source_records
source_record_map = {}
for r in data.get("source_records", []):
    external = r.get("external_record_id")
    if external:
        source_record_map[external] = deterministic_uuid(
            "openlibrary:source_record:" + external
        )

# UUID de job d'import
IMPORT_JOB_ID = deterministic_uuid("nexus:openlibrary:import:v4.6.6")

plan = {
    "version": "V4.6.6",
    "source_system_id": SOURCE_SYSTEM_ID,
    "import_job_id": IMPORT_JOB_ID,
    "id_maps": id_maps,
    "source_record_map": source_record_map,
    "counts": {k: len(v) for k, v in data.items()},
    "status": "PLAN_GENERATION",
}

print("Job:", IMPORT_JOB_ID)
print("Mapping source_records:", len(source_record_map))
print("Mapping works:", len(id_maps.get("works", {})))
print("Mapping agents:", len(id_maps.get("agents", {})))
print("Mapping editions:", len(id_maps.get("editions", {})))


import os, json

# Contrôles des références et des identifiants techniques
checks = {}

checks["works_ids"] = all(
    r.get("work_id") for r in data.get("works", [])
)
checks["agents_ids"] = all(
    r.get("agent_id") for r in data.get("agents", [])
)
checks["edition_ids"] = all(
    r.get("edition_id") for r in data.get("editions", [])
)
checks["expression_ids"] = all(
    r.get("expression_id") and r.get("work_id")
    for r in data.get("expressions", [])
)

checks["edition_work_refs"] = all(
    r.get("edition_id") in set(id_maps["editions"].values())
    and r.get("work_id") in set(id_maps["works"].values())
    for r in data.get("edition_works", [])
)

checks["edition_expression_refs"] = all(
    r.get("edition_id") in set(id_maps["editions"].values())
    and r.get("work_id") in set(id_maps["works"].values())
    and r.get("expression_id") in {
        x.get("expression_id") for x in data.get("expressions", [])
    }
    for r in data.get("edition_expressions", [])
)

checks["identifier_schemes_lowercase"] = all(
    r.get("scheme_code") == str(r.get("scheme_code", "")).lower()
    for r in data.get("identifiers", [])
)

checks["source_record_mapping"] = (
    len(source_record_map) == len(data.get("source_records", []))
)

checks["source_hashes_present"] = all(
    r.get("payload_sha256") for r in data.get("source_records", [])
)

checks["source_system_confirmed"] = SOURCE_SYSTEM_ID == \
    "67d4c7e7-a324-49ef-9f60-fc24ca4af926"

plan["checks"] = checks
plan["status"] = (
    "V4_6_6_PLAN_VALIDATED"
    if all(checks.values())
    else "V4_6_6_PLAN_REQUIRES_CORRECTION"
)

os.makedirs(OUT, exist_ok=True)
report_path = os.path.join(OUT, "v466_import_plan_report.json")

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)

print("=== V4.6.6 ===")
print(json.dumps(checks, indent=2))
print("STATUT:", plan["status"])
print("Rapport:", report_path)
print("ÉCRITURE SQL:", False)
print("IMPORT CANONIQUE:", False)


In [ ]:
import os, json, hashlib, uuid

BASE = "/kaggle/working/nexus_science_v4_6"
OUT = os.path.join(BASE, "v467_sql_package")

os.makedirs(OUT, exist_ok=True)

FILES = {
    "works": "works_plan.jsonl",
    "agents": "agents_sql_plan_v44.jsonl",
    "editions": "editions_plan.jsonl",
    "expressions": "expressions_plan.jsonl",
    "titles": "titles_sql_plan_v44.jsonl",
    "edition_works": "edition_works_plan.jsonl",
    "edition_expressions": "edition_expressions_plan.jsonl",
    "work_agents": "work_agents_plan.jsonl",
    "agent_names": "agent_names_plan.jsonl",
    "identifiers": "bibliographic_identifiers_plan_v46.jsonl",
    "source_records": "source_records_plan.jsonl",
}

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

data = {}
for k, fn in FILES.items():
    p = os.path.join(BASE, fn)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    data[k] = load_jsonl(p)

print("Plans chargés :", len(data))
print("Total lignes :", sum(len(x) for x in data.values()))
print("V4.6.7 — LOCAL SQL PACKAGE")


In [ ]:
import json, hashlib, uuid, os

SOURCE_SYSTEM_ID = "67d4c7e7-a324-49ef-9f60-fc24ca4af926"
IMPORT_JOB_ID = "0a5804f3-e823-5b5d-a28c-0c0e25a83af7"

def q(v):
    if v is None:
        return "NULL"
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, (dict, list)):
        return "'" + json.dumps(v, ensure_ascii=False).replace("'", "''") + "'::jsonb"
    return "'" + str(v).replace("'", "''") + "'"

def insert_sql(table, columns, rows):
    if not rows:
        return ""
    out = []
    for r in rows:
        vals = ", ".join(q(r.get(c)) for c in columns)
        out.append(f"INSERT INTO bibliotheque.{table} ({', '.join(columns)}) VALUES ({vals});")
    return "\n".join(out)

sql_sections = []

# 1. Works
sql_sections.append(insert_sql(
    "works",
    ["work_id", "work_type", "record_status"],
    data["works"]
))

# 2. Agents
sql_sections.append(insert_sql(
    "agents",
    ["agent_id", "agent_type", "record_status"],
    data["agents"]
))

# 3. Work expressions
sql_sections.append(insert_sql(
    "work_expressions",
    ["expression_id", "work_id", "expression_type", "record_status"],
    data["expressions"]
))

# 4. Expression titles
sql_sections.append(insert_sql(
    "expression_titles",
    ["expression_title_id", "expression_id", "title_type",
     "title_text", "normalized_title", "is_preferred",
     "sequence_no", "language_code"],
    data["titles"]
))

# 5. Editions
sql_sections.append(insert_sql(
    "editions",
    ["edition_id", "edition_type", "title_statement", "record_status"],
    data["editions"]
))

# 6. Edition-works
sql_sections.append(insert_sql(
    "edition_works",
    ["edition_id", "work_id", "content_role", "sequence_no"],
    data["edition_works"]
))

# 7. Edition-expressions
sql_sections.append(insert_sql(
    "edition_expressions",
    ["edition_id", "work_id", "expression_id"],
    data["edition_expressions"]
))

# 8. Work-agents
sql_sections.append(insert_sql(
    "work_agents",
    ["work_id", "agent_id", "role_code", "sequence_no"],
    data["work_agents"]
))

# 9. Agent names
sql_sections.append(insert_sql(
    "agent_names",
    ["agent_name_id", "agent_id", "name_type",
     "name_text", "normalized_name", "is_preferred"],
    data["agent_names"]
))

# 10. Bibliographic identifiers
identifier_rows = []
for r in data["identifiers"]:
    target = {
        "work": "work_id",
        "edition": "edition_id",
        "expression": "expression_id"
    }.get(r["entity_type"])
    if not target:
        raise ValueError("Type identifiant non autorisé: " + str(r["entity_type"]))
    identifier_rows.append({
        "scheme_code": r["scheme_code"],
        "identifier_value": r["identifier_value"],
        "normalized_value": r["normalized_value"],
        target: r["entity_id"],
        "is_preferred": False
    })

identifier_cols = [
    "scheme_code", "identifier_value", "normalized_value",
    "work_id", "edition_id", "expression_id", "is_preferred"
]

# SQL adapté aux cibles exclusives
for r in identifier_rows:
    targets = [x for x in ["work_id", "edition_id", "expression_id"] if r.get(x)]
    if len(targets) != 1:
        raise ValueError("Cible bibliographique invalide")
    cols = ["scheme_code", "identifier_value", "normalized_value",
            targets[0], "is_preferred"]
    sql_sections.append(insert_sql(
        "bibliographic_identifiers", cols, [r]
    ))

# 11. Source records
source_rows = []
for r in data["source_records"]:
    source_rows.append({
        "source_record_id": str(uuid.uuid5(
            uuid.UUID(SOURCE_SYSTEM_ID),
            "openlibrary:source_record:" + r["external_record_id"]
        )),
        "source_system_id": SOURCE_SYSTEM_ID,
        "external_record_id": r["external_record_id"],
        "source_record_version": "unknown",
        "payload": r["payload"],
        "payload_sha256": r["payload_sha256"],
        "record_status": "new"
    })

sql_sections.append(insert_sql(
    "source_records",
    ["source_record_id", "source_system_id",
     "external_record_id", "source_record_version",
     "payload", "payload_sha256", "record_status"],
    source_rows
))

# 12. Source-record links : association édition
links = []
for r in data["source_records"]:
    edition = next(
        (x for x in data["editions"]
         if x["external_id"] == r["external_record_id"]),
        None
    )
    if edition:
        srid = str(uuid.uuid5(
            uuid.UUID(SOURCE_SYSTEM_ID),
            "openlibrary:source_record:" + r["external_record_id"]
        ))
        links.append({
            "source_record_id": srid,
            "edition_id": edition["edition_id"],
            "link_type": "source",
            "match_method": "external_identifier",
            "confidence": 1.0
        })

sql_sections.append(insert_sql(
    "source_record_links",
    ["source_record_id", "edition_id", "link_type",
     "match_method", "confidence"],
    links
))

# 13. Import job
job = [{
    "import_job_id": IMPORT_JOB_ID,
    "source_system_id": SOURCE_SYSTEM_ID,
    "job_type": "bibliographic_import",
    "status": "prepared",
    "parameters": {
        "version": "V4.6.7",
        "source": "Open Library",
        "package": "V4.6.6",
        "canonical_write_authorized": False
    }
}]
sql_sections.append(insert_sql(
    "import_jobs",
    ["import_job_id", "source_system_id", "job_type",
     "status", "parameters"],
    job
))

# 14. Import records
import_records = []
for r in source_rows:
    import_records.append({
        "import_record_id": str(uuid.uuid5(
            uuid.UUID(SOURCE_SYSTEM_ID),
            "openlibrary:import_record:" + r["external_record_id"]
        )),
        "import_job_id": IMPORT_JOB_ID,
        "source_record_id": r["source_record_id"],
        "outcome": "pending",
        "attempt_count": 0
    })

sql_sections.append(insert_sql(
    "import_records",
    ["import_record_id", "import_job_id",
     "source_record_id", "outcome", "attempt_count"],
    import_records
))

sql = "\n\n".join(s for s in sql_sections if s)

sql_path = os.path.join(OUT, "v467_import.sql")
with open(sql_path, "w", encoding="utf-8") as f:
    f.write("BEGIN;\n\n")
    f.write(sql)
    f.write("\n\nROLLBACK;\n")

print("SQL généré:", sql_path)
print("Source records:", len(source_rows))
print("Source links:", len(links))
print("Import records:", len(import_records))
print("Mode SQL: ROLLBACK")


In [ ]:
import os, hashlib, json

sql_path = os.path.join(OUT, "v467_import.sql")

with open(sql_path, "rb") as f:
    content = f.read()

audit = {
    "version": "V4.6.7",
    "sql_file_exists": os.path.exists(sql_path),
    "sql_bytes": len(content),
    "sha256": hashlib.sha256(content).hexdigest(),
    "begin_present": content.startswith(b"BEGIN;"),
    "rollback_present": content.rstrip().endswith(b"ROLLBACK;"),
    "insert_count": content.count(b"INSERT INTO"),
    "tables_used": [
        "works", "agents", "work_expressions", "expression_titles",
        "editions", "edition_works", "edition_expressions",
        "work_agents", "agent_names", "bibliographic_identifiers",
        "source_records", "source_record_links",
        "import_jobs", "import_records"
    ],
    "canonical_write_executed": False,
    "status": "LOCAL_SQL_PACKAGE_GENERATED"
}

audit_path = os.path.join(OUT, "v467_package_audit.json")
with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

print("=== V4.6.7 AUDIT ===")
print(json.dumps(audit, indent=2, ensure_ascii=False))


In [ ]:
# V4.6.8 — PATCH DES CONTRAINTES NEON

p = os.path.join(BASE, "v467_sql_package", "v467_import.sql")

with open(p, "r", encoding="utf-8") as f:
    sql = f.read()

sql = sql.replace("'bibliographic_import'", "'other'")
sql = sql.replace("'prepared'", "'queued'")
sql = sql.replace("'pending'", "'review_required'")
sql = sql.replace("'source'", "'describes'")
sql = sql.replace("'external_identifier'", "'identifier'")

# attempt_count : la valeur 0 est remplacée par 1
# via correction de la génération des import_records
print("Patch des valeurs de contraintes appliqué en mémoire.")
print("Ne pas écraser le SQL avec ce patch textuel seul.")
print("La correction du compteur sera faite dans le générateur.")


In [ ]:
import os

BASE = "/kaggle/working/nexus_science_v4_6"

print("=== FICHIERS DU PACKAGE ===")

for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if d not in ["__pycache__"]]
    for f in files:
        if f.endswith((".py", ".json", ".jsonl")):
            path = os.path.join(root, f)
            print(path)

In [ ]:
import os, json

BASE = "/kaggle/working/nexus_science_v4_6"

files = [
    "works_plan.jsonl",
    "authors_plan.jsonl",
    "editions_plan.jsonl",
    "expressions_plan.jsonl",
    "titles_plan.jsonl",
    "edition_works_plan.jsonl",
    "edition_expressions_plan.jsonl",
    "work_agents_plan.jsonl",
    "agent_names_plan.jsonl",
    "bibliographic_identifiers_plan_v46.jsonl",
    "source_records_plan.jsonl",
]

print("=== V4.6.8 — STRUCTURE DES PLANS ===")

for name in files:
    path = os.path.join(BASE, name)
    print(f"\n--- {name} ---")

    if not os.path.exists(path):
        print("ABSENT")
        continue

    with open(path, "r", encoding="utf-8") as f:
        line = f.readline()

    if line:
        obj = json.loads(line)
        print("Champs:", list(obj.keys()))
        print("Exemple:", json.dumps(obj, ensure_ascii=False)[:1200])
    else:
        print("FICHIER VIDE")

In [ ]:
import os, json, uuid
from collections import Counter

BASE = "/kaggle/working/nexus_science_v4_6"

FILES = {
    "works": "works_plan.jsonl",
    "authors": "authors_plan.jsonl",
    "editions": "editions_plan.jsonl",
    "expressions": "expressions_plan.jsonl",
    "titles": "titles_plan.jsonl",
    "edition_works": "edition_works_plan.jsonl",
    "edition_expressions": "edition_expressions_plan.jsonl",
    "work_agents": "work_agents_plan.jsonl",
    "agent_names": "agent_names_plan.jsonl",
    "identifiers": "bibliographic_identifiers_plan_v46.jsonl",
    "source_records": "source_records_plan.jsonl",
}

def load(name):
    path = os.path.join(BASE, FILES[name])
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

data = {name: load(name) for name in FILES}

print("=== V4.6.8 — CONTRÔLE AVANT GÉNÉRATION ===")

for name, rows in data.items():
    print(f"{name:22s}: {len(rows)}")

print("\n=== VALEURS À NORMALISER ===")
for name, field in [
    ("works", "work_type"),
    ("authors", "agent_type"),
    ("editions", "edition_type"),
    ("expressions", "expression_type"),
    ("titles", "title_type"),
    ("edition_works", "content_role"),
    ("work_agents", "role_code"),
    ("agent_names", "name_type"),
    ("identifiers", "scheme_code"),
]:
    print(name, dict(Counter(str(r.get(field)) for r in data[name])))

print("\n=== INTÉGRITÉ DES RELATIONS ===")

work_ids = {r["work_id"] for r in data["works"]}
agent_ids = {r["agent_id"] for r in data["authors"]}
edition_ids = {r["edition_id"] for r in data["editions"]}
expression_ids = {r["expression_id"] for r in data["expressions"]}

checks = {
    "expressions_work_missing": sum(r["work_id"] not in work_ids for r in data["expressions"]),
    "edition_works_missing_edition": sum(r["edition_id"] not in edition_ids for r in data["edition_works"]),
    "edition_works_missing_work": sum(r["work_id"] not in work_ids for r in data["edition_works"]),
    "edition_expressions_missing_edition": sum(r["edition_id"] not in edition_ids for r in data["edition_expressions"]),
    "edition_expressions_missing_work": sum(r["work_id"] not in work_ids for r in data["edition_expressions"]),
    "edition_expressions_missing_expression": sum(r["expression_id"] not in expression_ids for r in data["edition_expressions"]),
    "work_agents_missing_work": sum(r["work_id"] not in work_ids for r in data["work_agents"]),
    "work_agents_missing_agent": sum(r["agent_id"] not in agent_ids for r in data["work_agents"]),
    "agent_names_missing_agent": sum(r["agent_id"] not in agent_ids for r in data["agent_names"]),
    "identifiers_missing_target": sum(
        (r["entity_type"] == "work" and r["entity_id"] not in work_ids)
        or (r["entity_type"] == "edition" and r["entity_id"] not in edition_ids)
        or (r["entity_type"] == "expression" and r["entity_id"] not in expression_ids)
        for r in data["identifiers"]
    ),
}

for k, v in checks.items():
    print(f"{k}: {v}")

print("\n=== SOURCES ===")
print("Types:", dict(Counter(r["entity_type"] for r in data["source_records"])))
print("Identifiants externes uniques:",
      len({r["external_record_id"] for r in data["source_records"]}))
print("Hash SHA-256 valides:",
      sum(len(r["payload_sha256"]) == 64 for r in data["source_records"]))

print("\n=== FIN DU CONTRÔLE ===")
print("TOTAL RELATIONS ENREGISTRÉES :", sum(len(v) for v in data.values()))


In [ ]:
import os, json, uuid, hashlib

BASE = "/kaggle/working/nexus_science_v4_6"
OUT = os.path.join(BASE, "v468_sql_package")
os.makedirs(OUT, exist_ok=True)

NS = uuid.UUID("0a5804f3-e823-5b5d-a28c-0c0e25a83af7")
JOB_ID = str(NS)
SOURCE_ID = "67d4c7e7-a324-49ef-9f60-fc24ca4af926"

FILES = {
    "works": "works_plan.jsonl",
    "authors": "authors_plan.jsonl",
    "editions": "editions_plan.jsonl",
    "expressions": "expressions_plan.jsonl",
    "titles": "titles_plan.jsonl",
    "edition_works": "edition_works_plan.jsonl",
    "edition_expressions": "edition_expressions_plan.jsonl",
    "work_agents": "work_agents_plan.jsonl",
    "agent_names": "agent_names_plan.jsonl",
    "identifiers": "bibliographic_identifiers_plan_v46.jsonl",
    "source_records": "source_records_plan.jsonl",
}

def load(name):
    with open(os.path.join(BASE, FILES[name]), encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

D = {k: load(k) for k in FILES}

def uid(key):
    return str(uuid.uuid5(NS, str(key)))

def sqlval(v):
    if v is None:
        return "NULL"
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, (dict, list)):
        return "'" + json.dumps(v, ensure_ascii=False).replace("'", "''") + "'::jsonb"
    return "'" + str(v).replace("'", "''") + "'"

statements = []

def insert(table, columns, rows):
    if not rows:
        return
    prefix = (
        f"INSERT INTO bibliotheque.{table} "
        f"({', '.join(columns)}) VALUES\n"
    )
    for start in range(0, len(rows), 200):
        chunk = rows[start:start+200]
        values = []
        for row in chunk:
            values.append("(" + ", ".join(sqlval(x) for x in row) + ")")
        statements.append(prefix + ",\n".join(values) + ";\n")

# 1. Import job
insert("import_jobs",
    ["import_job_id", "source_system_id", "job_type",
     "status", "parameters", "records_seen"],
    [[JOB_ID, SOURCE_ID, "other", "queued",
      {"package": "V4.6.8", "source": "Open Library",
       "mode": "prepared_not_executed"}, len(D["source_records"])]])

# 2. Works
insert("works",
    ["work_id", "work_type", "description", "record_status"],
    [[r["work_id"], r["work_type"], None, r["record_status"]]
     for r in D["works"]])

# 3. Agents — unresolved type mapped to other
insert("agents",
    ["agent_id", "agent_type", "record_status"],
    [[r["agent_id"],
      r["agent_type"] if r["agent_type"] in
      {"person", "corporate_body", "family", "meeting",
       "jurisdiction", "other"} else "other",
      r["record_status"]]
     for r in D["authors"]])

# 4. Editions
insert("editions",
    ["edition_id", "edition_type", "title_statement", "record_status"],
    [[r["edition_id"], r["edition_type"],
      r.get("title_statement"), r["record_status"]]
     for r in D["editions"]])

# 5. Expressions are stored in work_expressions
insert("work_expressions",
    ["expression_id", "work_id", "expression_type", "record_status"],
    [[r["expression_id"], r["work_id"],
      r["expression_type"], r["record_status"]]
     for r in D["expressions"]])

# 6. Expression titles
insert("expression_titles",
    ["expression_title_id", "expression_id", "title_type",
     "title_text", "normalized_title", "is_preferred", "sequence_no"],
    [[r["expression_title_id"], r["expression_id"],
      r["title_type"], r["title_text"], r["normalized_title"],
      r["is_preferred"], r["sequence_no"]]
     for r in D["titles"]])

# 7. Edition/work relations
insert("edition_works",
    ["edition_id", "work_id", "content_role", "sequence_no"],
    [[r["edition_id"], r["work_id"],
      r["content_role"], r["sequence_no"]]
     for r in D["edition_works"]])

# 8. Edition/expression relations
insert("edition_expressions",
    ["edition_id", "work_id", "expression_id"],
    [[r["edition_id"], r["work_id"], r["expression_id"]]
     for r in D["edition_expressions"]])

# 9. Work/agent relations
insert("work_agents",
    ["work_id", "agent_id", "role_code", "sequence_no"],
    [[r["work_id"], r["agent_id"],
      r["role_code"], r["sequence_no"]]
     for r in D["work_agents"]])

# 10. Agent names
insert("agent_names",
    ["agent_name_id", "agent_id", "name_type",
     "name_text", "normalized_name", "is_preferred"],
    [[r["agent_name_id"], r["agent_id"], r["name_type"],
      r["name_text"], r["normalized_name"], r["is_preferred"]]
     for r in D["agent_names"]])

# 11. Bibliographic identifiers
identifier_rows = []
for r in D["identifiers"]:
    target = r["entity_type"]
    if target not in {"work", "edition", "expression"}:
        raise ValueError("Type cible inconnu: " + str(target))

    identifier_id = uid(
        "identifier:" + r["scheme_code"] + ":" + r["normalized_value"]
    )
    identifier_rows.append([
        identifier_id, r["scheme_code"], r["identifier_value"],
        r["normalized_value"],
        r["entity_id"] if target == "work" else None,
        r["entity_id"] if target == "expression" else None,
        r["entity_id"] if target == "edition" else None,
        None, False
    ])

insert("bibliographic_identifiers",
    ["bibliographic_identifier_id", "scheme_code", "identifier_value",
     "normalized_value", "work_id", "expression_id", "edition_id",
     "digital_asset_id", "is_preferred"],
    identifier_rows)

# 12. Source records and provenance links
source_ids = {}
source_rows = []
link_rows = []
import_rows = []

work_by_external = {
    r["external_id"]: r["work_id"] for r in D["works"]
}
edition_by_external = {
    r["external_id"]: r["edition_id"] for r in D["editions"]
}

for r in D["source_records"]:
    ext = r["external_record_id"]
    sid = uid("source:" + ext)
    source_ids[ext] = sid

    source_rows.append([
        sid, SOURCE_ID, ext, "unknown",
        r["payload"], r["payload_sha256"], None, "new"
    ])

    # Source record describes its edition
    edition_id = edition_by_external.get(ext)
    if edition_id:
        link_rows.append([
            uid("link:edition:" + ext), sid,
            None, None, edition_id, None, None, None,
            "describes", "identifier", None
        ])

    # Explicit work references preserved from source payload
    for work_ext in r["payload"].get("works", []):
        work_id = work_by_external.get(work_ext)
        if work_id:
            link_rows.append([
                uid("link:work:" + ext + ":" + work_ext), sid,
                work_id, None, None, None, None, None,
                "describes", "exact", None
            ])

    import_rows.append([
        uid("import-record:" + ext),
        JOB_ID, sid, "review_required", 1
    ])

insert("source_records",
    ["source_record_id", "source_system_id", "external_record_id",
     "source_record_version", "payload", "payload_sha256",
     "record_language_code", "record_status"],
    source_rows)

insert("source_record_links",
    ["source_record_link_id", "source_record_id", "work_id",
     "expression_id", "edition_id", "digital_asset_id", "agent_id",
     "subject_id", "link_type", "match_method", "confidence"],
    link_rows)

insert("import_records",
    ["import_record_id", "import_job_id", "source_record_id",
     "outcome", "attempt_count"],
    import_rows)

# Build a clearly delimited SQL artifact.
sql = (
    "-- Nexus Science — V4.6.8\n"
    "-- GENERATED PACKAGE ONLY. NOT EXECUTED.\n"
    "-- Canonical Neon writes require separate explicit authorization.\n\n"
    "BEGIN;\n\n"
    + "\n".join(statements)
    + "\nCOMMIT;\n"
)

path = os.path.join(OUT, "v468_import.sql")
with open(path, "w", encoding="utf-8") as f:
    f.write(sql)

manifest = {
    "version": "V4.6.8",
    "status": "SQL_GENERATED_NOT_EXECUTED",
    "import_job_id": JOB_ID,
    "source_system_id": SOURCE_ID,
    "insert_statements": len(statements),
    "source_records": len(source_rows),
    "source_links": len(link_rows),
    "import_records": len(import_rows),
    "identifiers": len(identifier_rows),
    "sql_bytes": len(sql.encode("utf-8")),
    "sha256": hashlib.sha256(sql.encode("utf-8")).hexdigest(),
    "canonical_database_write": False
}

with open(os.path.join(OUT, "v468_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("=== V4.6.8 — GÉNÉRATION TERMINÉE ===")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print("SQL :", path)


In [ ]:
import os, json, hashlib, re
from collections import Counter

BASE = "/kaggle/working/nexus_science_v4_6"
OUT = os.path.join(BASE, "v468_sql_package")
SQL_PATH = os.path.join(OUT, "v468_import.sql")

def load(name):
    with open(os.path.join(BASE, name), encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

D = {
    "works": load("works_plan.jsonl"),
    "authors": load("authors_plan.jsonl"),
    "editions": load("editions_plan.jsonl"),
    "expressions": load("expressions_plan.jsonl"),
    "titles": load("titles_plan.jsonl"),
    "edition_works": load("edition_works_plan.jsonl"),
    "edition_expressions": load("edition_expressions_plan.jsonl"),
    "work_agents": load("work_agents_plan.jsonl"),
    "agent_names": load("agent_names_plan.jsonl"),
    "identifiers": load("bibliographic_identifiers_plan_v46.jsonl"),
    "source_records": load("source_records_plan.jsonl"),
}

with open(SQL_PATH, encoding="utf-8") as f:
    sql = f.read()

checks = {}

def duplicate_count(rows, key):
    values = [key(r) for r in rows]
    return len(values) - len(set(values))

checks["duplicate_work_ids"] = duplicate_count(
    D["works"], lambda r: r["work_id"])
checks["duplicate_agent_ids"] = duplicate_count(
    D["authors"], lambda r: r["agent_id"])
checks["duplicate_edition_ids"] = duplicate_count(
    D["editions"], lambda r: r["edition_id"])
checks["duplicate_expression_ids"] = duplicate_count(
    D["expressions"], lambda r: r["expression_id"])
checks["duplicate_title_ids"] = duplicate_count(
    D["titles"], lambda r: r["expression_title_id"])
checks["duplicate_agent_name_ids"] = duplicate_count(
    D["agent_names"], lambda r: r["agent_name_id"])
checks["duplicate_source_external_ids"] = duplicate_count(
    D["source_records"], lambda r: r["external_record_id"])
checks["duplicate_identifier_unique_keys"] = duplicate_count(
    D["identifiers"], lambda r: (r["scheme_code"], r["normalized_value"]))
checks["duplicate_edition_work_pairs"] = duplicate_count(
    D["edition_works"], lambda r: (r["edition_id"], r["work_id"]))
checks["duplicate_edition_work_order"] = duplicate_count(
    D["edition_works"], lambda r: (r["edition_id"], r["sequence_no"]))
checks["duplicate_work_agent_roles"] = duplicate_count(
    D["work_agents"], lambda r: (r["work_id"], r["agent_id"], r["role_code"]))
checks["duplicate_preferred_titles"] = duplicate_count(
    D["titles"], lambda r: (r["expression_id"], "", r["is_preferred"])
    if r["is_preferred"] else ("nonpreferred", r["expression_title_id"]))
checks["invalid_agent_types"] = sum(
    r["agent_type"] not in {
        "person", "corporate_body", "family",
        "meeting", "jurisdiction", "other"
    } for r in D["authors"]
)

# Contrôle du contenu SQL
checks["has_begin"] = sql.strip().startswith("BEGIN;")
checks["has_commit"] = sql.strip().endswith("COMMIT;")
checks["has_rollback"] = "ROLLBACK;" in sql
checks["insert_count"] = len(re.findall(
    r"INSERT INTO bibliotheque\.", sql, re.I))
checks["forbidden_expression_table"] = bool(re.search(
    r"INSERT INTO bibliotheque\.expressions\b", sql, re.I))
checks["contains_expected_job_id"] = (
    "0a5804f3-e823-5b5d-a28c-0c0e25a83af7" in sql
)

print("=== V4.6.8 — AUDIT FINAL ===")
for k, v in checks.items():
    print(f"{k}: {v}")

# Analyse syntaxique facultative
try:
    from pglast import parse_sql
    parse_sql(sql)
    checks["postgresql_syntax"] = "VALID"
except ImportError:
    checks["postgresql_syntax"] = "NOT_CHECKED_MODULE_ABSENT"
except Exception as e:
    checks["postgresql_syntax"] = "ERROR: " + str(e)

print("\nSyntaxe PostgreSQL:", checks["postgresql_syntax"])

fatal = [
    k for k, v in checks.items()
    if k.startswith("duplicate_") and v != 0
]
fatal += [
    k for k, v in checks.items()
    if k.startswith("invalid_") and v != 0
]
fatal += [
    k for k, v in checks.items()
    if k in ("has_begin", "has_commit", "contains_expected_job_id",
             "forbidden_expression_table") and
    (v is False if isinstance(v, bool) else v)
]
if checks["postgresql_syntax"].startswith("ERROR"):
    fatal.append("postgresql_syntax")

manifest = {
    "version": "V4.6.8",
    "audit": checks,
    "fatal_checks": fatal,
    "sql_sha256": hashlib.sha256(sql.encode()).hexdigest(),
    "canonical_database_write": False
}

with open(os.path.join(OUT, "v468_final_audit.json"), "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("\nSTATUT:", "AUDIT_LOCAL_OK" if not fatal else "AUDIT_FAILED")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
import os, re, json, hashlib

SQL_PATH = "/kaggle/working/nexus_science_v4_6/v468_sql_package/v468_import.sql"

with open(SQL_PATH, encoding="utf-8") as f:
    sql = f.read()

clean = sql.lstrip().upper()

checks = {
    "sql_non_vide": bool(sql.strip()),
    "begin_present": bool(re.search(r"\bBEGIN\s*;", sql, re.I)),
    "begin_at_start": clean.startswith("BEGIN;"),
    "commit_present": bool(re.search(r"\bCOMMIT\s*;", sql, re.I)),
    "rollback_present": bool(re.search(r"\bROLLBACK\s*;", sql, re.I)),
    "wrong_expression_table": bool(
        re.search(r"INSERT\s+INTO\s+bibliotheque\.expressions\b", sql, re.I)
    ),
    "correct_expression_table": bool(
        re.search(r"INSERT\s+INTO\s+bibliotheque\.work_expressions\b", sql, re.I)
    ),
    "unresolved_agent_type_in_sql": "person_or_organization_unresolved" in sql,
    "agent_type_other_occurrences": len(
        re.findall(r"'other'", sql, re.I)
    ),
    "insert_count": len(
        re.findall(r"INSERT\s+INTO\s+bibliotheque\.", sql, re.I)
    ),
    "expected_job_id": "0a5804f3-e823-5b5d-a28c-0c0e25a83af7" in sql,
}

print("=== V4.6.9 — AUDIT CORRECTIF ===")
for k, v in checks.items():
    print(f"{k}: {v}")

print("\n=== DÉBUT DU SQL ===")
print("\n".join(sql.splitlines()[:18]))

print("\n=== FIN DU SQL ===")
print("\n".join(sql.splitlines()[-12:]))

print("\n=== INTÉGRITÉ ===")
print("Taille:", len(sql.encode("utf-8")))
print("SHA256:", hashlib.sha256(sql.encode("utf-8")).hexdigest())

print("\nSTATUT:",
      "INSPECTION_OK" if (
          checks["sql_non_vide"]
          and checks["commit_present"]
          and checks["correct_expression_table"]
          and not checks["wrong_expression_table"]
          and not checks["unresolved_agent_type_in_sql"]
          and checks["expected_job_id"]
      ) else "CORRECTION_NÉCESSAIRE")

In [ ]:
!pip -q install pglast

from pglast import parse_sql
import os, hashlib

path = "/kaggle/working/nexus_science_v4_6/v468_sql_package/v468_import.sql"

with open(path, encoding="utf-8") as f:
    sql = f.read()

try:
    statements = parse_sql(sql)
    print("=== V4.6.10 — VALIDATION SQL ===")
    print("Syntaxe PostgreSQL : VALIDE")
    print("Instructions SQL analysées :", len(statements))
    print("SHA256 :", hashlib.sha256(sql.encode()).hexdigest())
    print("Écriture Neon exécutée : NON")
    print("STATUT : SQL_SYNTAX_VALIDATED")
except Exception as e:
    print("STATUT : SQL_SYNTAX_ERROR")
    print(str(e))

In [ ]:
!pip -q install pglast

from pglast import parse_sql
import os, hashlib

path = "/kaggle/working/nexus_science_v4_6/v468_sql_package/v468_import.sql"

with open(path, encoding="utf-8") as f:
    sql = f.read()

try:
    statements = parse_sql(sql)
    print("=== V4.6.10 — VALIDATION SQL ===")
    print("Syntaxe PostgreSQL : VALIDE")
    print("Instructions SQL analysées :", len(statements))
    print("SHA256 :", hashlib.sha256(sql.encode()).hexdigest())
    print("Écriture Neon exécutée : NON")
    print("STATUT : SQL_SYNTAX_VALIDATED")
except Exception as e:
    print("STATUT : SQL_SYNTAX_ERROR")
    print(str(e))

In [ ]:
from IPython.display import FileLink, display

path = "/kaggle/working/nexus_science_v4_6/v468_sql_package/v468_import.sql"

display(FileLink(path))

In [ ]:
!kaggle kernels output moseskasonga/notebook9e03c46c4d -p /kaggle/working/restauration_v468

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/restauration_v468/notebook9e03c46c4d.log")

print("Fichier :", p.exists())
print("Taille :", p.stat().st_size if p.exists() else 0)

if p.exists():
    contenu = p.read_text(encoding="utf-8", errors="replace")
    print(contenu[-12000:])

In [ ]:
import os

d = "/kaggle/working/restauration_v468"

print("CONTENU DE LA RESTAURATION")
for root, dirs, files in os.walk(d):
    for f in files:
        p = os.path.join(root, f)
        print(p, "|", os.path.getsize(p), "octets")

In [ ]:
!kaggle kernels pull moseskasonga/notebook9e03c46c4d

In [ ]:
import json

p = "/kaggle/working/notebook9e03c46c4d.ipynb"

with open(p, encoding="utf-8") as f:
    nb = json.load(f)

print("Nombre de cellules :", len(nb["cells"]))

for i, c in enumerate(nb["cells"]):
    s = "".join(c.get("source", []))
    print(i, c["cell_type"], len(s), s[:120].replace("\n", " "))

!kaggle kernels list --mine

In [ ]:
!kaggle kernels list --mine

In [ ]:
!kaggle kernels pull moseskasonga/notebooke3f58a17e8

In [ ]:
import json, os

p = "/kaggle/working/notebooke3f58a17e8.ipynb"

if os.path.exists(p):
    nb = json.load(open(p, encoding="utf-8"))
    print("Nombre de cellules :", len(nb["cells"]))
    for i, c in enumerate(nb["cells"]):
        s = "".join(c.get("source", []))
        print(i, c["cell_type"], len(s), s[:160].replace("\n", " "))
else:
    print("Notebook non trouvé à ce chemin.")

In [ ]:
import json

p = "/kaggle/working/notebook62c46c34f9.ipynb"
nb = json.load(open(p, encoding="utf-8"))

print("Nombre de cellules :", len(nb["cells"]))

for i, c in enumerate(nb["cells"]):
    s = "".join(c.get("source", []))
    print(i, c["cell_type"], len(s), s[:180].replace("\n", " "))

In [ ]:
!kaggle kernels pull moseskasonga/notebook62c46c34f9

In [ ]:
import json

p = "/kaggle/working/notebook62c46c34f9.ipynb"
nb = json.load(open(p, encoding="utf-8"))

print("Nombre de cellules :", len(nb["cells"]))

for i, c in enumerate(nb["cells"]):
    s = "".join(c.get("source", []))
    print(i, c["cell_type"], len(s), s[:180].replace("\n", " "))

In [ ]:
import subprocess, os

candidats = [
    "notebookd4a8ea7f9d",
    "notebooke84871ac01",
    "notebookd08c0c9207"
]

base = "/kaggle/working/recuperation_outputs"
os.makedirs(base, exist_ok=True)

for kernel in candidats:
    dest = os.path.join(base, kernel)
    os.makedirs(dest, exist_ok=True)
    print("\n==========", kernel, "==========")
    subprocess.run([
        "kaggle", "kernels", "output",
        "moseskasonga/" + kernel,
        "-p", dest
    ])

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/recuperation_outputs"):
    for f in files:
        p = os.path.join(root, f)
        print(p, "|", os.path.getsize(p), "octets")

In [ ]:
!kaggle kernels pull moseskasonga/notebookd4a8ea7f9d

In [ ]:
import json

p = "/kaggle/working/notebookd4a8ea7f9d.ipynb"
nb = json.load(open(p, encoding="utf-8"))

print("Nombre de cellules :", len(nb["cells"]))

for i, c in enumerate(nb["cells"]):
    s = "".join(c.get("source", []))
    print(f"\n--- CELLULE {i} | {c['cell_type']} | {len(s)} caractères ---")
    print(s[:250].replace("\n", " "))

In [ ]:
from pathlib import Path
import json

p = Path("/kaggle/working/notebookd4a8ea7f9d.ipynb")

print("Fichier présent :", p.exists())

if p.exists():
    print("Taille :", p.stat().st_size)
    nb = json.loads(p.read_text(encoding="utf-8"))
    print("Cellules :", len(nb.get("cells", [])))

    for i, c in enumerate(nb.get("cells", [])):
        s = "".join(c.get("source", []))
        print(f"CELLULE {i} | {c.get('cell_type')} | {len(s)} caractères")
        print(s[:150].replace("\n", " "))
else:
    print("Chemin incorrect.")

In [ ]:
import subprocess, json, os
from pathlib import Path

candidats = [
    "notebooke84871ac01",
    "notebookd08c0c9207",
    "notebookb9c21e1a21",
    "notebooka4ecc4d2e1",
    "notebook8642cd20c1"
]

base = Path("/kaggle/working/inventaire_nexus")
base.mkdir(exist_ok=True)

for k in candidats:
    p = Path("/kaggle/working") / f"{k}.ipynb"
    if not p.exists():
        subprocess.run(["kaggle", "kernels", "pull", f"moseskasonga/{k}"],
                       capture_output=True, text=True)
    if p.exists():
        nb = json.loads(p.read_text(encoding="utf-8"))
        print("\n###", k, "|", len(nb.get("cells", [])), "cellules")
        for i, c in enumerate(nb.get("cells", [])):
            s = "".join(c.get("source", []))
            if any(x in s.lower() for x in [
                "v4.6", "v468", "sql_package", "import.sql",
                "sha256", "consolidation", "pglast"
            ]):
                print("CELLULE", i, "|", len(s), "caractères")
                print(s[:350].replace("\n", " "))
    else:
        print("\n###", k, "| téléchargement indisponible")

In [ ]:
import json

p = "/kaggle/working/notebooke84871ac01.ipynb"

with open(p, encoding="utf-8") as f:
    nb = json.load(f)

print("".join(nb["cells"][7]["source"]))

In [ ]:
import os

print("local_stage :", local_stage)
print("Fichier présent :", os.path.isfile(local_stage))
print("recovered_commit présent :", isinstance(recovered_commit, dict))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        path = os.path.join(root, f)
        if any(k in f.lower() for k in [
            "stage", "commit", "checkpoint", "jsonl",
            "recovery", "recover", "v468", "import"
        ]):
            print(path, "|", os.path.getsize(path), "octets")

In [ ]:
import os, hashlib

EXPECTED = "d2d0e0159070d2f3ffb5bcb022bf6abcf6f316b16e8d78866f397ee3aa48d4d4"

roots = ["/kaggle/working", "/kaggle/input", "/kaggle/temp"]
candidates = []

for root in roots:
    if not os.path.exists(root):
        continue
    for base, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
        for f in files:
            if f.lower().endswith((".sql", ".jsonl", ".json", ".zip")):
                p = os.path.join(base, f)
                try:
                    size = os.path.getsize(p)
                    if size > 0:
                        candidates.append((p, size))
                except:
                    pass

print("=== FICHIERS CANDIDATS ===")
for p, size in candidates:
    print(p, "|", size, "octets")
    if p.lower().endswith(".sql"):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for block in iter(lambda: f.read(1024*1024), b""):
                h.update(block)
        digest = h.hexdigest()
        print("SHA256:", digest)
        print("HASH V468 CONFORME:", digest == EXPECTED)

print("\nTOTAL:", len(candidates), "fichiers")

In [ ]:
import os, json

ids = [
    "notebooke84871ac01",
    "notebook9e03c46c4d",
    "notebook62c46c34f9",
    "notebooke3f58a17e8",
    "notebookd4a8ea7f9d",
    "notebookd08c0c9207",
    "notebookb9c21e1a21",
    "notebooka4ecc4d2e1",
    "notebook8642cd20c1"
]

for k in ids:
    p = f"/kaggle/working/{k}.ipynb"
    if not os.path.exists(p):
        continue

    with open(p, encoding="utf-8") as f:
        nb = json.load(f)

    found = False
    for i, cell in enumerate(nb.get("cells", [])):
        for out in cell.get("outputs", []):
            data = out.get("data", {})
            textdata = out.get("text", "")
            if isinstance(textdata, list):
                textdata = "".join(textdata)

            rich = data.get("text/plain", "")
            if isinstance(rich, list):
                rich = "".join(rich)

            combined = str(textdata) + str(rich)

            if combined.strip() or "text/html" in data:
                if not found:
                    print("\n###", k)
                    found = True
                print("Cellule", i, "| sortie enregistrée")
                if combined.strip():
                    print(combined[:700])
                if "text/html" in data:
                    print("[Sortie HTML intégrée]")
    if not found:
        print(k, "| aucune sortie textuelle intégrée détectée")

In [ ]:
import json

p = "/kaggle/working/notebooke84871ac01.ipynb"

with open(p, encoding="utf-8") as f:
    nb = json.load(f)

for i in [4, 5, 6]:
    print(f"\n{'='*20} CELLULE {i} {'='*20}")
    print("".join(nb["cells"][i]["source"]))

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Moses-team/nexus-science-backup"

api = HfApi(token=TOKEN)
files = api.list_repo_files(repo_id=REPO, repo_type="dataset")

print("=== INVENTAIRE COMPLET ===")
print("Total fichiers :", len(files))

for p in files:
    print(p)

In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import hashlib, json, os

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Moses-team/nexus-science-backup"

audit_dir = "/kaggle/working/nexus_pilot_audit"
os.makedirs(audit_dir, exist_ok=True)

for i in range(1, 7):
    name = f"pilot_v8/pilot_segment_{i:04d}.jsonl"

    p = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=name,
        token=TOKEN
    )

    dest = os.path.join(audit_dir, os.path.basename(name))
    with open(p, "rb") as src, open(dest, "wb") as out:
        out.write(src.read())

    h = hashlib.sha256()
    count = 0
    fields = set()
    invalid = 0

    with open(dest, "rb") as f:
        for line in f:
            if not line.strip():
                continue
            h.update(line)
            try:
                obj = json.loads(line)
                count += 1
                fields.update(obj.keys())
            except Exception:
                invalid += 1

    print(f"\n=== SEGMENT {i} ===")
    print("Fichier :", dest)
    print("Notices :", count)
    print("JSON invalides :", invalid)
    print("Champs :", sorted(fields))
    print("SHA256 :", h.hexdigest())

print("\n=== AUDIT TERMINE ===")

In [ ]:
import os, json

d = "/kaggle/working/nexus_pilot_audit"

for f in sorted(os.listdir(d)):
    p = os.path.join(d, f)
    count = 0
    invalid = 0

    with open(p, encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            try:
                json.loads(line)
                count += 1
            except:
                invalid += 1

    print(f"{f} | notices={count} | invalides={invalid}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
import json, os, hashlib

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Moses-team/nexus-science-backup"

commit_path = hf_hub_download(
    repo_id=REPO, repo_type="dataset",
    filename="v10_recovery/commit.json", token=TOKEN
)

with open(commit_path, encoding="utf-8") as f:
    commit = json.load(f)

stage_name = commit["stage_filename"]

stage_path = hf_hub_download(
    repo_id=REPO, repo_type="dataset",
    filename=f"v10_recovery/{stage_name}", token=TOKEN
)

h = hashlib.sha256()
count = 0
samples = []

with open(stage_path, "rb") as f:
    for line in f:
        if not line.strip():
            continue
        h.update(line)
        obj = json.loads(line)
        count += 1
        if len(samples) < 3:
            samples.append(obj)

print("=== RECUPERATION V10.2 ===")
print("Stage :", stage_name)
print("Notices :", count)
print("SHA256 :", h.hexdigest())
print("Hash attendu :", commit["stage_sha256"])
print("Hash conforme :", h.hexdigest() == commit["stage_sha256"])
print("Champs :", sorted(samples[0].keys()) if samples else [])
print("Exemples :", json.dumps(samples, ensure_ascii=False, indent=2)[:2500])

In [ ]:
!python /mnt/data/nexus_v10_2_v47_preimport_auditor.py stage_000001.jsonl

In [ ]:
from pathlib import Path

script_path = Path("/kaggle/working/nexus_v10_2_v47_preimport_auditor.py")

script_path.write_text(r'''#!/usr/bin/env python3
import json, hashlib, re, sys
from pathlib import Path
from datetime import datetime, timezone

INPUT = Path(sys.argv[1] if len(sys.argv)>1 else "stage_000001.jsonl")
OUT = Path(sys.argv[2] if len(sys.argv)>2 else "nexus_v47_preimport")
OUT.mkdir(parents=True, exist_ok=True)

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def norm(s):
    return re.sub(r"\s+", " ", (s or "").strip()).casefold()

def year_from(s):
    m=re.search(r"\b(1[0-9]{3}|20[0-9]{2}|21[0-9]{2})\b", str(s or ""))
    return int(m.group(1)) if m else None

def keys(arr):
    return [x.get("key") for x in arr if isinstance(x,dict) and x.get("key")]

raw_count=0; valid=0; invalid=0; no_title=0; duplicates=0
seen=set(); candidates=[]; invalid_rows=[]

with INPUT.open("rb") as f:
    for ln,line in enumerate(f,1):
        if not line.strip(): continue
        raw_count+=1
        try:
            obj=json.loads(line)
        except Exception as e:
            invalid+=1
            invalid_rows.append({"line":ln,"error":str(e)})
            continue
        if not isinstance(obj,dict):
            invalid+=1
            invalid_rows.append({"line":ln,"error":"not an object"})
            continue
        valid+=1
        title=(obj.get("title") or "").strip()
        if not title: no_title+=1
        external_id=obj.get("key") or f"line:{ln}"
        rev=str(obj.get("revision",obj.get("latest_revision","unknown")))
        fingerprint=sha256_bytes(json.dumps(obj,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode())
        if external_id in seen: duplicates+=1
        seen.add(external_id)
        pdate=str(obj.get("publish_date") or "")
        candidate={
          "source_external_id":external_id,
          "source_record_version":rev,
          "payload_sha256":fingerprint,
          "title":title,
          "title_normalized":norm(title),
          "authors_source_keys":keys(obj.get("authors",[])),
          "work_source_keys":keys(obj.get("works",[])),
          "publication_date_display":pdate or None,
          "publication_year":year_from(pdate),
          "publishers":obj.get("publishers") or [],
          "publish_places":obj.get("publish_places") or [],
          "languages_source_keys":keys(obj.get("languages",[])),
          "isbn_10":obj.get("isbn_10") or [],
          "isbn_13":obj.get("isbn_13") or [],
          "oclc_numbers":obj.get("oclc_numbers") or [],
          "lccn":obj.get("lccn") or [],
          "subjects":obj.get("subjects") or [],
          "physical_format":obj.get("physical_format"),
          "pagination":obj.get("pagination"),
          "number_of_pages":obj.get("number_of_pages"),
          "series":obj.get("series") or [],
          "table_of_contents":obj.get("table_of_contents") or [],
          "full_payload":obj
        }
        candidates.append(candidate)

with (OUT/"candidate_editions.jsonl").open("w",encoding="utf-8") as w:
    for c in candidates:
        w.write(json.dumps(c,ensure_ascii=False)+"\n")

with (OUT/"invalid_rows.json").open("w",encoding="utf-8") as w:
    json.dump(invalid_rows,w,ensure_ascii=False,indent=2)

manifest={
 "pipeline":"Nexus V10.2 to V47 staging auditor",
 "status":"STAGING_ONLY_NO_DATABASE_WRITES",
 "input_file":str(INPUT),
 "input_sha256":sha256_bytes(INPUT.read_bytes()),
 "input_nonblank_lines":raw_count,
 "json_valid_objects":valid,
 "invalid_json_lines":invalid,
 "missing_title":no_title,
 "duplicate_source_keys_detected":duplicates,
 "candidate_rows_written":len(candidates),
 "output":"candidate_editions.jsonl",
 "audit_file":"invalid_rows.json",
 "timestamp_utc":datetime.now(timezone.utc).isoformat(),
 "warning":"Candidates are not certified entities; no SQL generated or executed."
}

(OUT/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(manifest,ensure_ascii=False,indent=2))
''')

print("SCRIPT CREE :", script_path)
print("EXISTE :", script_path.exists())


In [ ]:
!python /kaggle/working/nexus_v10_2_v47_preimport_auditor.py stage_000001.jsonl


In [ ]:
from pathlib import Path

roots = [Path("/kaggle/working"), Path("/kaggle/input")]
matches = []

for root in roots:
    if root.exists():
        matches.extend(root.rglob("stage_000001.jsonl"))

for p in matches:
    print("SOURCE TROUVEE :", p)

if matches:
    source = matches[0]
    !python /kaggle/working/nexus_v10_2_v47_preimport_auditor.py "{source}"
else:
    print("Source absente des dossiers Kaggle.")
    print("Contenu de /kaggle/working :")
    print(list(Path("/kaggle/working").iterdir()))
    print("Contenu de /kaggle/input :")
    if Path("/kaggle/input").exists():
        print(list(Path("/kaggle/input").iterdir()))
    else:
        print("Dossier input absent.")


In [ ]:
from pathlib import Path

base = Path("/kaggle/working")

for folder in ["recuperation_outputs", "restauration_v468", "nexus_v47_preimport", "inventaire_nexus"]:
    p = base / folder
    print(f"\n===== {folder} =====")
    if p.exists():
        for f in p.rglob("*"):
            if f.is_file():
                print(f, "|", f.stat().st_size, "bytes")
    else:
        print("Absent")


In [ ]:
from pathlib import Path
import json, re

base = Path("/kaggle/working")
patterns = re.compile(
    r"stage_000001|Moses-team/nexus-science-backup|"
    r"v10_recovery|RECUPERATION V10|huggingface|"
    r"hf_hub_download|snapshot_download|download",
    re.I
)

for nb_path in base.glob("*.ipynb"):
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
        hits = []
        for i, cell in enumerate(nb.get("cells", [])):
            content = "".join(cell.get("source", []))
            if patterns.search(content):
                hits.append((i, content))
            for out in cell.get("outputs", []):
                txt = "".join(out.get("text", []))
                if patterns.search(txt):
                    hits.append((i, txt))
        if hits:
            print("\n########", nb_path.name, "########")
            for i, content in hits:
                print(f"\n--- CELLULE {i} ---\n{content[:5000]}")
    except Exception as e:
        print("Erreur notebook", nb_path.name, e)


In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import json, shutil, os, hashlib

REPO = "Moses-team/nexus-science-backup"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

RESTORE_DIR = "/kaggle/working/nexus_v10_reconstruction"
os.makedirs(RESTORE_DIR, exist_ok=True)

commit_path = hf_hub_download(
    repo_id=REPO,
    repo_type="dataset",
    filename="v10_recovery/commit.json",
    token=TOKEN
)

with open(commit_path, encoding="utf-8") as f:
    commit = json.load(f)

stage_name = commit["stage_filename"]

stage_path = hf_hub_download(
    repo_id=REPO,
    repo_type="dataset",
    filename=f"v10_recovery/{stage_name}",
    token=TOKEN
)

local_stage = os.path.join(RESTORE_DIR, stage_name)
shutil.copy2(stage_path, local_stage)

actual_hash = hashlib.sha256(
    open(local_stage, "rb").read()
).hexdigest()

expected_hash = commit["stage_sha256"]

print("=== RESTAURATION V10.2 ===")
print("Stage :", local_stage)
print("Notices :", commit["checkpoint"]["selected"])
print("SHA256 :", actual_hash)
print("Hash attendu :", expected_hash)
print("Hash conforme :", actual_hash == expected_hash)

if actual_hash != expected_hash:
    raise RuntimeError("ECHEC VALIDATION SHA256")

print("\n=== AUDIT V47 ===")

os.system(
    f'python /kaggle/working/nexus_v10_2_v47_preimport_auditor.py "{local_stage}"'
)


In [ ]:
from pathlib import Path
import json, uuid, hashlib
from collections import Counter

BASE = Path("/kaggle/working/nexus_v47_preimport")
INPUT = BASE / "candidate_editions.jsonl"
OUT = Path("/kaggle/working/nexus_v47_1_relational")
OUT.mkdir(parents=True, exist_ok=True)

if not INPUT.exists():
    raise FileNotFoundError(f"Candidats V47 introuvables : {INPUT}")

NAMESPACE = uuid.UUID("67d4c7e7-a324-49ef-9f60-fc24ca4af926")

def stable_id(kind, key):
    return str(uuid.uuid5(NAMESPACE, f"{kind}:{key}"))

def write_jsonl(name, rows):
    p = OUT / name
    with p.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return p

editions = []
works = {}
agents = {}
edition_works = []
edition_agents = []
identifiers = []
source_links = []

seen_source_ids = set()
duplicate_source_ids = []
identifier_collision_counter = Counter()

with INPUT.open(encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        source_id = str(c.get("source_external_id") or "")
        if not source_id:
            continue

        if source_id in seen_source_ids:
            duplicate_source_ids.append(source_id)
        seen_source_ids.add(source_id)

        edition_id = stable_id("edition", source_id)

        editions.append({
            "edition_id": edition_id,
            "source_external_id": source_id,
            "title": c.get("title"),
            "publication_date": c.get("publication_date_display"),
            "publication_year": c.get("publication_year"),
            "publishers": c.get("publishers", []),
            "physical_format": c.get("physical_format"),
            "pagination": c.get("pagination"),
            "number_of_pages": c.get("number_of_pages"),
            "payload_sha256": c.get("payload_sha256"),
        })

        for wk in c.get("work_source_keys", []):
            wid = stable_id("work", str(wk))
            works.setdefault(wid, {
                "work_id": wid,
                "source_work_key": wk
            })
            edition_works.append({
                "edition_id": edition_id,
                "work_id": wid,
                "content_role": "main",
                "sequence_no": 1
            })

        for ak in c.get("authors_source_keys", []):
            aid = stable_id("agent", str(ak))
            agents.setdefault(aid, {
                "agent_id": aid,
                "source_agent_key": ak
            })
            edition_agents.append({
                "edition_id": edition_id,
                "agent_id": aid,
                "role_code": "author",
                "sequence_no": 1
            })

        for scheme, values in [
            ("isbn_10", c.get("isbn_10", [])),
            ("isbn_13", c.get("isbn_13", [])),
            ("oclc", c.get("oclc_numbers", [])),
            ("lccn", c.get("lccn", []))
        ]:
            for val in values or []:
                if isinstance(val, dict):
                    val = val.get("value") or val.get("identifier")
                if not val:
                    continue
                value = str(val).strip()
                normalized = "".join(ch for ch in value if ch.isalnum()).upper()
                if not normalized:
                    continue

                identifier_collision_counter[(scheme, normalized)] += 1
                identifiers.append({
                    "bibliographic_identifier_id":
                        stable_id("identifier", f"{scheme}:{normalized}:{edition_id}"),
                    "scheme_code": scheme,
                    "identifier_value": value,
                    "normalized_value": normalized,
                    "edition_id": edition_id,
                    "work_id": None,
                    "expression_id": None,
                    "is_preferred": False
                })

        source_links.append({
            "edition_id": edition_id,
            "source_external_id": source_id,
            "match_method": "deterministic_source_key",
            "confidence": 1.0
        })

duplicate_identifiers = [
    {"scheme": k[0], "normalized_value": k[1], "occurrences": v}
    for k, v in identifier_collision_counter.items() if v > 1
]

files = {
    "edition_candidates.jsonl": editions,
    "work_identity_candidates.jsonl": list(works.values()),
    "agent_identity_candidates.jsonl": list(agents.values()),
    "edition_works_plan.jsonl": edition_works,
    "edition_agents_plan.jsonl": edition_agents,
    "identifier_plan.jsonl": identifiers,
    "source_link_plan.jsonl": source_links,
    "identifier_collision_audit.jsonl": duplicate_identifiers,
}

for name, rows in files.items():
    write_jsonl(name, rows)

manifest = {
    "version": "V47.1",
    "status": "RELATIONAL_STAGING_PLAN_ONLY",
    "canonical_database_write": False,
    "sql_generated": False,
    "source_candidates": len(editions),
    "unique_work_identity_candidates": len(works),
    "unique_agent_identity_candidates": len(agents),
    "edition_work_relationships": len(edition_works),
    "edition_agent_relationships": len(edition_agents),
    "identifier_rows": len(identifiers),
    "duplicate_source_ids": len(duplicate_source_ids),
    "identifier_collision_groups": len(duplicate_identifiers),
    "files_written": list(files.keys()),
    "warning": (
        "UUIDs are deterministic staging identities. This is not a certified "
        "database import. Agent/work identities, constraints, and mappings "
        "require validation against target schema and source semantics."
    )
}

(OUT / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("=== V47.1 — PLAN RELATIONNEL ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nDOSSIER :", OUT)
print("STATUT : PLAN RELATIONNEL PRODUIT — AUCUNE ECRITURE NEON")


In [ ]:
from pathlib import Path
import json, hashlib
from collections import Counter

D = Path("/kaggle/working/nexus_v47_1_relational")

def read_jsonl(name):
    p=D/name
    if not p.exists():
        raise FileNotFoundError(p)
    with p.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

editions=read_jsonl("edition_candidates.jsonl")
works=read_jsonl("work_identity_candidates.jsonl")
agents=read_jsonl("agent_identity_candidates.jsonl")
ew=read_jsonl("edition_works_plan.jsonl")
ea=read_jsonl("edition_agents_plan.jsonl")
ids=read_jsonl("identifier_plan.jsonl")
links=read_jsonl("source_link_plan.jsonl")

checks={}

def dup_count(rows, key):
    c=Counter(r[key] for r in rows)
    return sum(v>1 for v in c.values()), sum(v-1 for v in c.values())

checks["duplicate_edition_ids"]=dup_count(editions,"edition_id")
checks["duplicate_work_ids"]=dup_count(works,"work_id")
checks["duplicate_agent_ids"]=dup_count(agents,"agent_id")
checks["duplicate_edition_work_pairs"]=dup_count(
    ew, "edition_id"
)
checks["duplicate_source_links"]=dup_count(links,"source_external_id")

edition_set={r["edition_id"] for r in editions}
work_set={r["work_id"] for r in works}
agent_set={r["agent_id"] for r in agents}

orphan_edition_work=[
    r for r in ew if r["edition_id"] not in edition_set or r["work_id"] not in work_set
]
orphan_edition_agent=[
    r for r in ea if r["edition_id"] not in edition_set or r["agent_id"] not in agent_set
]
orphan_identifiers=[
    r for r in ids if r.get("edition_id") not in edition_set
]

missing_work_links=[
    r for r in editions
    if not any(x["edition_id"]==r["edition_id"] for x in ew)
]
missing_agent_links=[
    r for r in editions
    if not any(x["edition_id"]==r["edition_id"] for x in ea)
]

invalid_identifiers=[
    r for r in ids
    if not r.get("scheme_code") or not r.get("normalized_value")
]

role_counts=Counter(r.get("role_code") for r in ea)
scheme_counts=Counter(r.get("scheme_code") for r in ids)

report={
 "version":"V47.2",
 "status":"RELATIONAL_CONSISTENCY_AUDIT",
 "edition_count":len(editions),
 "work_identity_count":len(works),
 "agent_identity_count":len(agents),
 "edition_work_links":len(ew),
 "edition_agent_links":len(ea),
 "identifier_count":len(ids),
 "duplicate_edition_id_groups":checks["duplicate_edition_ids"][0],
 "duplicate_work_id_groups":checks["duplicate_work_ids"][0],
 "duplicate_agent_id_groups":checks["duplicate_agent_ids"][0],
 "duplicate_source_link_groups":checks["duplicate_source_links"][0],
 "orphan_edition_work_rows":len(orphan_edition_work),
 "orphan_edition_agent_rows":len(orphan_edition_agent),
 "orphan_identifier_rows":len(orphan_identifiers),
 "editions_without_work_link":len(missing_work_links),
 "editions_without_agent_link":len(missing_agent_links),
 "invalid_identifier_rows":len(invalid_identifiers),
 "agent_role_distribution":dict(role_counts),
 "identifier_scheme_distribution":dict(scheme_counts),
 "canonical_database_write":False,
 "sql_generated":False
}

audit_dir=Path("/kaggle/working/nexus_v47_2_audit")
audit_dir.mkdir(parents=True,exist_ok=True)

for name, rows in [
    ("orphan_edition_work.jsonl",orphan_edition_work),
    ("orphan_edition_agent.jsonl",orphan_edition_agent),
    ("orphan_identifiers.jsonl",orphan_identifiers),
    ("missing_work_links.jsonl",missing_work_links),
    ("missing_agent_links.jsonl",missing_agent_links),
    ("invalid_identifiers.jsonl",invalid_identifiers)
]:
    (audit_dir/name).write_text(
        "".join(json.dumps(r,ensure_ascii=False)+"\n" for r in rows),
        encoding="utf-8"
    )

(audit_dir/"manifest.json").write_text(
    json.dumps(report,ensure_ascii=False,indent=2),
    encoding="utf-8"
)

print("=== V47.2 — AUDIT RELATIONNEL ===")
print(json.dumps(report,ensure_ascii=False,indent=2))
print("AUDIT FILES :",audit_dir)


In [ ]:
from pathlib import Path
import json
from collections import Counter

BASE = Path("/kaggle/working/nexus_v47_1_relational")
AUDIT = Path("/kaggle/working/nexus_v47_2_audit")
OUT = Path("/kaggle/working/nexus_v47_3_gap_analysis")
OUT.mkdir(parents=True, exist_ok=True)

def load(name, folder=BASE):
    with open(folder/name, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

editions = load("edition_candidates.jsonl")
ew = load("edition_works_plan.jsonl")
ea = load("edition_agents_plan.jsonl")

work_linked = {r["edition_id"] for r in ew}
agent_linked = {r["edition_id"] for r in ea}

missing_work = [e for e in editions if e["edition_id"] not in work_linked]
missing_agent = [e for e in editions if e["edition_id"] not in agent_linked]

def inspect(records, field):
    counts = Counter()
    examples = []
    for e in records:
        p = e.get("full_payload") or {}
        value = p.get(field)
        if value is None:
            counts["ABSENT_OR_NULL"] += 1
        elif isinstance(value, list):
            counts["LIST_EMPTY" if not value else "LIST_NONEMPTY"] += 1
        elif isinstance(value, dict):
            counts["DICT"] += 1
        else:
            counts[type(value).__name__] += 1

        if len(examples) < 20:
            examples.append({
                "source_external_id": e.get("source_external_id"),
                "title": e.get("title"),
                field: value
            })
    return counts, examples

work_stats, work_examples = inspect(missing_work, "works")
author_stats, author_examples = inspect(missing_agent, "authors")

# Vérification de la multiplicité réelle des paires relationnelles
work_pairs = [(r["edition_id"], r["work_id"]) for r in ew]
agent_pairs = [(r["edition_id"], r["agent_id"]) for r in ea]

report = {
    "version": "V47.3",
    "status": "GAP_ANALYSIS_ONLY",
    "editions_without_work_link": len(missing_work),
    "work_source_field_distribution": dict(work_stats),
    "editions_without_agent_link": len(missing_agent),
    "author_source_field_distribution": dict(author_stats),
    "duplicate_edition_work_pairs": len(work_pairs)-len(set(work_pairs)),
    "duplicate_edition_agent_pairs": len(agent_pairs)-len(set(agent_pairs)),
    "work_missing_examples": work_examples,
    "agent_missing_examples": author_examples,
    "canonical_database_write": False,
    "sql_generated": False
}

(OUT/"report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

(OUT/"missing_work_notices.jsonl").write_text(
    "".join(json.dumps(e, ensure_ascii=False)+"\n" for e in missing_work),
    encoding="utf-8"
)

(OUT/"missing_agent_notices.jsonl").write_text(
    "".join(json.dumps(e, ensure_ascii=False)+"\n" for e in missing_agent),
    encoding="utf-8"
)

print("=== V47.3 — ANALYSE DES LACUNES ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nDOSSIER :", OUT)


In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict

SRC = Path("/kaggle/working/nexus_v47_1_relational")
OUT = Path("/kaggle/working/nexus_v47_4_corrected")
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

agents = read_jsonl(SRC / "edition_agents_plan.jsonl")
editions = read_jsonl(SRC / "edition_candidates.jsonl")
works = read_jsonl(SRC / "edition_works_plan.jsonl")

# Deduplicate exact edition-agent pairs; retain first occurrence.
unique = []
seen = set()
duplicate_rows = []

for row in agents:
    pair = (row["edition_id"], row["agent_id"])
    if pair in seen:
        duplicate_rows.append(row)
        continue
    seen.add(pair)
    unique.append(row)

# Reassign sequence numbers per edition.
by_edition = defaultdict(list)
for row in unique:
    by_edition[row["edition_id"]].append(row)

corrected_agents = []
for edition_id, rows in by_edition.items():
    for seq, row in enumerate(rows, 1):
        corrected = dict(row)
        corrected["sequence_no"] = seq
        corrected_agents.append(corrected)

# Classify missing links without fabricating relationships.
work_linked = {r["edition_id"] for r in works}
agent_linked = {r["edition_id"] for r in corrected_agents}

missing_work = [e for e in editions if e["edition_id"] not in work_linked]
missing_agent = [e for e in editions if e["edition_id"] not in agent_linked]

# Build explicit staging exceptions.
exceptions = []
for e in editions:
    eid = e["edition_id"]
    if eid not in work_linked:
        exceptions.append({
            "edition_id": eid,
            "source_external_id": e.get("source_external_id"),
            "exception_code": "SOURCE_WORK_FIELD_ABSENT",
            "action": "NO_WORK_LINK_CREATED"
        })
    if eid not in agent_linked:
        exceptions.append({
            "edition_id": eid,
            "source_external_id": e.get("source_external_id"),
            "exception_code": "SOURCE_AUTHOR_FIELD_ABSENT",
            "action": "NO_AGENT_LINK_CREATED"
        })

save_jsonl(OUT / "edition_agents_corrected.jsonl", corrected_agents)
save_jsonl(OUT / "duplicate_agent_pairs_removed.jsonl", duplicate_rows)
save_jsonl(OUT / "missing_work_exceptions.jsonl", [
    {
        "edition_id": e["edition_id"],
        "source_external_id": e.get("source_external_id"),
        "title": e.get("title")
    } for e in missing_work
])
save_jsonl(OUT / "missing_agent_exceptions.jsonl", [
    {
        "edition_id": e["edition_id"],
        "source_external_id": e.get("source_external_id"),
        "title": e.get("title")
    } for e in missing_agent
])
save_jsonl(OUT / "relational_exceptions.jsonl", exceptions)

manifest = {
    "version": "V47.4",
    "status": "CORRECTED_RELATIONAL_STAGING",
    "edition_agent_rows_original": len(agents),
    "duplicate_pairs_removed": len(duplicate_rows),
    "edition_agent_rows_corrected": len(corrected_agents),
    "unique_edition_agent_pairs": len(seen),
    "editions_without_work_link": len(missing_work),
    "editions_without_agent_link": len(missing_agent),
    "exceptions_recorded": len(exceptions),
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

(OUT / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("=== V47.4 — CORRECTION ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
# === V47.5 — IDENTITY & SOURCE SEMANTICS AUDIT ===
from pathlib import Path
import json
from collections import Counter

SRC = Path("/kaggle/working/nexus_v47_1_relational")
CORR = Path("/kaggle/working/nexus_v47_4_corrected")
OUT = Path("/kaggle/working/nexus_v47_5_identity_audit")
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_json(p, obj):
    p.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def write_jsonl(p, rows):
    with open(p, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

editions = read_jsonl(SRC / "edition_candidates.jsonl")
works = read_jsonl(SRC / "work_identity_candidates.jsonl")
agents = read_jsonl(SRC / "agent_identity_candidates.jsonl")
ew = read_jsonl(SRC / "edition_works_plan.jsonl")
ea = read_jsonl(CORR / "edition_agents_corrected.jsonl")
ids = read_jsonl(SRC / "identifier_plan.jsonl")
links = read_jsonl(SRC / "source_link_plan.jsonl")

# Recovered source
stage = Path("/kaggle/working/stage_000001.jsonl")
if not stage.exists():
    candidates = list(Path("/kaggle/working").glob("**/stage_000001.jsonl"))
    if candidates:
        stage = candidates[0]

payload = read_jsonl(stage)

# Source key sets
source_keys = {
    str(r.get("key")) for r in payload if r.get("key") is not None
}
edition_ids = {r["edition_id"] for r in editions}
work_ids = {r["work_id"] for r in works}
agent_ids = {r["agent_id"] for r in agents}

# Structural integrity
checks = {
    "source_records": len(payload),
    "edition_candidates": len(editions),
    "work_identity_candidates": len(works),
    "agent_identity_candidates": len(agents),
    "edition_work_links": len(ew),
    "corrected_edition_agent_links": len(ea),
    "identifier_rows": len(ids),
    "source_link_rows": len(links),
    "duplicate_edition_ids": len(editions) - len(edition_ids),
    "duplicate_work_ids": len(works) - len(work_ids),
    "duplicate_agent_ids": len(agents) - len(agent_ids),
    "edition_work_orphans": sum(
        r["edition_id"] not in edition_ids or r["work_id"] not in work_ids
        for r in ew
    ),
    "edition_agent_orphans": sum(
        r["edition_id"] not in edition_ids or r["agent_id"] not in agent_ids
        for r in ea
    ),
    "identifier_edition_orphans": sum(
        r.get("edition_id") not in edition_ids for r in ids
    ),
    "source_link_edition_orphans": sum(
        r.get("edition_id") not in edition_ids for r in links
    ),
}

# Inspect actual source field structures
field_counts = Counter()
author_shapes = Counter()
work_shapes = Counter()

for r in payload:
    field_counts.update(r.keys())
    a = r.get("authors")
    w = r.get("works")
    author_shapes[type(a).__name__ + ":" + str(a is None)] += 1
    work_shapes[type(w).__name__ + ":" + str(w is None)] += 1

# Examine author and work reference semantics
author_samples = []
work_samples = []

for r in payload:
    if r.get("authors"):
        author_samples.append({
            "source_key": r.get("key"),
            "title": r.get("title"),
            "authors_sample": r.get("authors")
        })
    if r.get("works"):
        work_samples.append({
            "source_key": r.get("key"),
            "title": r.get("title"),
            "works_sample": r.get("works")
        })

report = {
    "version": "V47.5",
    "status": "SOURCE_IDENTITY_AUDIT",
    "checks": checks,
    "source_fields": dict(field_counts),
    "authors_structure": dict(author_shapes),
    "works_structure": dict(work_shapes),
    "author_samples": author_samples[:15],
    "work_samples": work_samples[:15],
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

write_json(OUT / "audit_report.json", report)
write_jsonl(OUT / "author_samples.jsonl", author_samples[:100])
write_jsonl(OUT / "work_samples.jsonl", work_samples[:100])

print("=== V47.5 — AUDIT ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("DOSSIER :", OUT)

In [ ]:
# === V47.6 — MAPPING SEMANTIC AUDIT ===
from pathlib import Path
import json
from collections import Counter

SRC = Path("/kaggle/working/nexus_v47_1_relational")
CORR = Path("/kaggle/working/nexus_v47_4_corrected")
OUT = Path("/kaggle/working/nexus_v47_6_semantics")
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

files = {
    "editions": SRC / "edition_candidates.jsonl",
    "works": SRC / "work_identity_candidates.jsonl",
    "agents": SRC / "agent_identity_candidates.jsonl",
    "edition_works": SRC / "edition_works_plan.jsonl",
    "edition_agents": CORR / "edition_agents_corrected.jsonl",
    "identifiers": SRC / "identifier_plan.jsonl",
    "source_links": SRC / "source_link_plan.jsonl",
}

data = {name: read_jsonl(path) for name, path in files.items()}

# Inspect the actual staging structures, not assumed schema.
structure = {
    name: rows[:3]
    for name, rows in data.items()
}

# Identity key uniqueness
identity_checks = {}
for name in ["works", "agents"]:
    rows = data[name]
    identity_checks[name] = {
        "rows": len(rows),
        "unique_rows": len({
            json.dumps(r, sort_keys=True, ensure_ascii=False)
            for r in rows
        }),
        "sample": rows[:5]
    }

# Inspect role codes and identifier schemes
role_distribution = Counter(
    str(r.get("role_code")) for r in data["edition_agents"]
)
scheme_distribution = Counter(
    str(r.get("scheme")) for r in data["identifiers"]
)

# Inspect potential identifier duplicates
identifier_keys = Counter(
    (
        str(r.get("scheme", "")).lower(),
        str(r.get("identifier", "")).strip()
    )
    for r in data["identifiers"]
)

duplicate_identifier_values = [
    {"scheme": k[0], "identifier": k[1], "count": v}
    for k, v in identifier_keys.items()
    if v > 1
]

# Sample actual mappings for review
report = {
    "version": "V47.6",
    "status": "SEMANTIC_MAPPING_REVIEW",
    "identity_checks": identity_checks,
    "agent_role_distribution": dict(role_distribution),
    "identifier_scheme_distribution": dict(scheme_distribution),
    "duplicate_identifier_values": duplicate_identifier_values[:100],
    "duplicate_identifier_value_groups": len(duplicate_identifier_values),
    "structures": structure,
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

with open(OUT / "semantic_audit.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== V47.6 — SEMANTIC REVIEW ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("DOSSIER :", OUT)

In [ ]:
# === V47.7 — IDENTIFIER AUDIT CORRECTED ===
from pathlib import Path
import json
from collections import Counter, defaultdict

SRC = Path("/kaggle/working/nexus_v47_1_relational")
OUT = Path("/kaggle/working/nexus_v47_7_identifier_audit")
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

ids = read_jsonl(SRC / "identifier_plan.jsonl")

def valid_isbn10(v):
    v = str(v).replace("-", "").replace(" ", "").upper()
    if len(v) != 10:
        return False
    if not all(c.isdigit() for c in v[:9]) or v[-1] not in "0123456789X":
        return False
    total = sum((10-i) * int(v[i]) for i in range(9))
    total += 10 if v[-1] == "X" else int(v[-1])
    return total % 11 == 0

def valid_isbn13(v):
    v = str(v).replace("-", "").replace(" ", "")
    if len(v) != 13 or not v.isdigit():
        return False
    total = sum(int(v[i]) * (1 if i % 2 == 0 else 3) for i in range(12))
    return (10 - total % 10) % 10 == int(v[-1])

scheme_counts = Counter()
blank_rows = []
invalid_isbns = []
global_groups = defaultdict(list)
edition_groups = defaultdict(list)

for r in ids:
    scheme = str(r.get("scheme_code") or "").strip().lower()
    value = str(r.get("normalized_value") or "").strip()
    edition = r.get("edition_id")

    scheme_counts[scheme] += 1

    if not scheme or not value:
        blank_rows.append(r)

    if scheme in ("isbn_10", "isbn10") and value and not valid_isbn10(value):
        invalid_isbns.append({
            "scheme": scheme,
            "value": value,
            "edition_id": edition,
            "reason": "INVALID_ISBN10_CHECKSUM_OR_FORMAT"
        })

    if scheme in ("isbn_13", "isbn13") and value and not valid_isbn13(value):
        invalid_isbns.append({
            "scheme": scheme,
            "value": value,
            "edition_id": edition,
            "reason": "INVALID_ISBN13_CHECKSUM_OR_FORMAT"
        })

    if scheme and value:
        global_groups[(scheme, value)].append(r)
        edition_groups[(edition, scheme, value)].append(r)

global_collisions = [
    {
        "scheme_code": k[0],
        "normalized_value": k[1],
        "count": len(v),
        "edition_ids": [x.get("edition_id") for x in v]
    }
    for k, v in global_groups.items()
    if len(v) > 1
]

edition_duplicates = [
    {
        "edition_id": k[0],
        "scheme_code": k[1],
        "normalized_value": k[2],
        "count": len(v)
    }
    for k, v in edition_groups.items()
    if len(v) > 1
]

report = {
    "version": "V47.7",
    "status": "CORRECTED_IDENTIFIER_AUDIT",
    "identifier_rows": len(ids),
    "scheme_distribution": dict(scheme_counts),
    "blank_identifier_rows": len(blank_rows),
    "invalid_isbn_rows": len(invalid_isbns),
    "global_collision_groups": len(global_collisions),
    "duplicate_identifiers_within_same_edition": len(edition_duplicates),
    "global_collision_samples": global_collisions[:100],
    "invalid_isbn_samples": invalid_isbns[:100],
    "edition_duplicate_samples": edition_duplicates[:100],
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

with open(OUT / "audit_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

with open(OUT / "global_collisions.jsonl", "w", encoding="utf-8") as f:
    for r in global_collisions:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(OUT / "invalid_isbns.jsonl", "w", encoding="utf-8") as f:
    for r in invalid_isbns:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("=== V47.7 — IDENTIFIER AUDIT ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("DOSSIER :", OUT)

In [ ]:
# === V47.8 — NEON REMAPPING PLAN ===
from pathlib import Path
from collections import defaultdict, Counter
import json
import hashlib

BASE = Path("/kaggle/working")
SRC = BASE / "nexus_v47_1_relational"
CORR = BASE / "nexus_v47_4_corrected"
OUT = BASE / "nexus_v47_8_neon_remapping"
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

editions = read_jsonl(SRC / "edition_candidates.jsonl")
works = read_jsonl(SRC / "work_identity_candidates.jsonl")
agents = read_jsonl(SRC / "agent_identity_candidates.jsonl")
edition_works = read_jsonl(SRC / "edition_works_plan.jsonl")
edition_agents = read_jsonl(CORR / "edition_agents_corrected.jsonl")
identifiers = read_jsonl(SRC / "identifier_plan.jsonl")
source_links = read_jsonl(SRC / "source_link_plan.jsonl")

# 1. Build edition -> work map.
works_by_edition = defaultdict(list)
for row in edition_works:
    works_by_edition[row["edition_id"]].append(row["work_id"])

# 2. Map author-edition relations to work-agent relations.
# Only editions linked to exactly one work are automatically mapped.
work_agent_candidates = []
ambiguous_author_mappings = []
unlinked_author_mappings = []

for row in edition_agents:
    eid = row["edition_id"]
    linked_works = sorted(set(works_by_edition.get(eid, [])))

    if len(linked_works) == 1:
        work_agent_candidates.append({
            "work_id": linked_works[0],
            "agent_id": row["agent_id"],
            "role_code": row["role_code"],
            "sequence_no": row.get("sequence_no", 1),
            "origin_edition_id": eid,
            "mapping_rule": "SINGLE_WORK_ON_EDITION"
        })
    elif len(linked_works) == 0:
        unlinked_author_mappings.append({
            **row,
            "exception_code": "EDITION_HAS_NO_WORK_LINK"
        })
    else:
        ambiguous_author_mappings.append({
            **row,
            "candidate_work_ids": linked_works,
            "exception_code": "EDITION_HAS_MULTIPLE_WORKS"
        })

# 3. Deduplicate according to Neon work_agents primary key.
unique_work_agents = {}
duplicate_work_agents = []

for row in work_agent_candidates:
    key = (row["work_id"], row["agent_id"], row["role_code"])
    if key in unique_work_agents:
        duplicate_work_agents.append(row)
    else:
        unique_work_agents[key] = row

work_agents_plan = list(unique_work_agents.values())

# Reassign sequence per work and role.
by_work_role = defaultdict(list)
for row in work_agents_plan:
    by_work_role[(row["work_id"], row["role_code"])].append(row)

final_work_agents = []
for key, rows in by_work_role.items():
    for seq, row in enumerate(rows, 1):
        item = dict(row)
        item["sequence_no"] = seq
        final_work_agents.append(item)

# 4. Normalize bibliographic identifier schemes.
# Keep original scheme in staging for traceability.
scheme_map = {
    "isbn_10": "isbn",
    "isbn_13": "isbn",
    "oclc": "oclc",
    "lccn": "lccn"
}

identifier_plan = []
unsupported_identifiers = []

for row in identifiers:
    original = row.get("scheme_code")
    target = scheme_map.get(original)

    if target is None:
        unsupported_identifiers.append({
            **row,
            "exception_code": "UNMAPPED_IDENTIFIER_SCHEME"
        })
        continue

    identifier_plan.append({
        "bibliographic_identifier_id":
            row["bibliographic_identifier_id"],
        "target_scheme_code": target,
        "source_scheme_code": original,
        "identifier_value": row["identifier_value"],
        "normalized_value": row["normalized_value"],
        "edition_id": row.get("edition_id"),
        "work_id": row.get("work_id"),
        "expression_id": row.get("expression_id"),
        "is_preferred": row.get("is_preferred", False)
    })

# 5. Source record links need source_record_id and link_type.
# The original plan does not provide these target fields.
source_link_review = []
for row in source_links:
    source_link_review.append({
        "edition_id": row.get("edition_id"),
        "source_external_id": row.get("source_external_id"),
        "match_method_source": row.get("match_method"),
        "target_match_method": (
            "exact"
            if row.get("match_method") == "deterministic_source_key"
            else None
        ),
        "confidence": row.get("confidence"),
        "target_link_type": "describes",
        "requires_source_record_id": True,
        "status": "SOURCE_RECORD_ID_REQUIRED"
    })

# 6. Basic FK and uniqueness checks.
edition_ids = {r["edition_id"] for r in editions}
work_ids = {r["work_id"] for r in works}
agent_ids = {r["agent_id"] for r in agents}

fk_errors = []
for r in final_work_agents:
    if r["work_id"] not in work_ids:
        fk_errors.append({"type": "UNKNOWN_WORK", "row": r})
    if r["agent_id"] not in agent_ids:
        fk_errors.append({"type": "UNKNOWN_AGENT", "row": r})

for r in identifier_plan:
    if r.get("edition_id") and r["edition_id"] not in edition_ids:
        fk_errors.append({"type": "UNKNOWN_IDENTIFIER_EDITION", "row": r})

# 7. Prepare outputs.
write_jsonl(OUT / "work_agents_plan.jsonl", final_work_agents)
write_jsonl(OUT / "duplicate_work_agents.jsonl", duplicate_work_agents)
write_jsonl(OUT / "ambiguous_author_mappings.jsonl", ambiguous_author_mappings)
write_jsonl(OUT / "unlinked_author_mappings.jsonl", unlinked_author_mappings)
write_jsonl(OUT / "bibliographic_identifiers_plan.jsonl", identifier_plan)
write_jsonl(OUT / "unsupported_identifiers.jsonl", unsupported_identifiers)
write_jsonl(OUT / "source_record_links_review.jsonl", source_link_review)
write_jsonl(OUT / "foreign_key_errors.jsonl", fk_errors)

report = {
    "version": "V47.8",
    "status": "NEON_REMAPPING_STAGING_ONLY",
    "source_editions": len(editions),
    "source_work_agent_links": len(edition_agents),
    "work_agent_candidates_before_dedup": len(work_agent_candidates),
    "work_agent_links_after_dedup": len(final_work_agents),
    "duplicate_work_agent_links_removed": len(duplicate_work_agents),
    "ambiguous_author_mappings": len(ambiguous_author_mappings),
    "author_mappings_without_work": len(unlinked_author_mappings),
    "identifier_source_rows": len(identifiers),
    "identifier_target_rows": len(identifier_plan),
    "identifier_scheme_mapping": dict(Counter(
        f'{r["source_scheme_code"]}->{r["target_scheme_code"]}'
        for r in identifier_plan
    )),
    "unsupported_identifier_rows": len(unsupported_identifiers),
    "source_link_rows_reviewed": len(source_link_review),
    "source_links_requiring_source_record_id": sum(
        r["requires_source_record_id"] for r in source_link_review
    ),
    "foreign_key_errors": len(fk_errors),
    "target_branch": "v47-prevalidation-2026-11-26",
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

write_json(OUT / "manifest.json", report)

print("=== V47.8 — NEON REMAPPING ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
import json, os, collections

BASE = "/kaggle/working"
SRC = f"{BASE}/nexus_v47_8_neon_remapping"
OUT = f"{BASE}/nexus_v47_9_final_schema_audit"
os.makedirs(OUT, exist_ok=True)

def read_jsonl(name):
    p = f"{SRC}/{name}"
    with open(p, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write_jsonl(name, rows):
    with open(f"{OUT}/{name}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

ids = read_jsonl("bibliographic_identifiers_plan.jsonl")
works = read_jsonl("work_identity_candidates.jsonl")
agents = read_jsonl("agent_identity_candidates.jsonl")
editions = read_jsonl("edition_candidates.jsonl")

# 1. Contrôle de l'unicité après fusion ISBN-10 / ISBN-13
groups = collections.defaultdict(list)
for r in ids:
    key = (r.get("scheme_code"), r.get("normalized_value"))
    groups[key].append(r)

collisions = [
    {"scheme_code": k[0], "normalized_value": k[1],
     "count": len(v),
     "edition_ids": sorted(set(x.get("edition_id") for x in v))}
    for k, v in groups.items() if len(v) > 1
]

# 2. Contrôle des champs obligatoires et types autorisés
edition_types = {"print","ebook","audiobook","braille",
                 "microform","manuscript","other"}
work_types = {"monograph","collective","reference",
              "series","component","other"}
agent_types = {"person","corporate_body","family",
               "meeting","jurisdiction","other"}

edition_type_review = []
for r in editions:
    value = r.get("edition_type")
    if value not in edition_types:
        edition_type_review.append({
            "edition_id": r.get("edition_id"),
            "source_external_id": r.get("source_external_id"),
            "observed_value": value,
            "reason": "EDITION_TYPE_REQUIRED_OR_UNMAPPED"
        })

work_type_review = [
    {"work_id": r.get("work_id"),
     "source_work_key": r.get("source_work_key"),
     "reason": "WORK_TYPE_REQUIRED_MAPPING"}
    for r in works
]

agent_type_review = [
    {"agent_id": r.get("agent_id"),
     "source_agent_key": r.get("source_agent_key"),
     "reason": "AGENT_TYPE_REQUIRED_MAPPING"}
    for r in agents
]

# 3. Contrôle des champs de titres et identifiants
missing_titles = [
    {"edition_id": r.get("edition_id"),
     "source_external_id": r.get("source_external_id")}
    for r in editions if not str(r.get("title") or "").strip()
]

invalid_identifiers = [
    r for r in ids
    if not str(r.get("scheme_code") or "").strip()
    or not str(r.get("identifier_value") or "").strip()
    or not str(r.get("normalized_value") or "").strip()
]

# 4. Rapport
manifest = {
    "version": "V47.9",
    "status": "FINAL_SCHEMA_AUDIT_STAGING_ONLY",
    "editions": len(editions),
    "works": len(works),
    "agents": len(agents),
    "identifiers": len(ids),
    "post_mapping_identifier_collision_groups": len(collisions),
    "edition_type_review": len(edition_type_review),
    "work_type_review": len(work_type_review),
    "agent_type_review": len(agent_type_review),
    "missing_titles": len(missing_titles),
    "invalid_identifiers": len(invalid_identifiers),
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

write_jsonl("identifier_collisions_after_mapping.jsonl", collisions)
write_jsonl("edition_type_review.jsonl", edition_type_review)
write_jsonl("work_type_review.jsonl", work_type_review)
write_jsonl("agent_type_review.jsonl", agent_type_review)
write_jsonl("missing_titles.jsonl", missing_titles)
write_jsonl("invalid_identifiers.jsonl", invalid_identifiers)

with open(f"{OUT}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== V47.9 — FINAL SCHEMA AUDIT ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
import os, json, glob, collections

BASE = "/kaggle/working"
OUT = f"{BASE}/nexus_v47_9_final_schema_audit"
os.makedirs(OUT, exist_ok=True)

def locate(filename):
    matches = glob.glob(f"{BASE}/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Fichier introuvable : {filename}")
    return matches[0]

def read_jsonl(filename):
    path = locate(filename)
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(x) for x in f if x.strip()]
    print(f"OK {filename}: {len(rows)} lignes")
    return rows

def write_jsonl(filename, rows):
    with open(f"{OUT}/{filename}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Sources réparties entre V47.1 et V47.8
ids = read_jsonl("bibliographic_identifiers_plan.jsonl")
works = read_jsonl("work_identity_candidates.jsonl")
agents = read_jsonl("agent_identity_candidates.jsonl")
editions = read_jsonl("edition_candidates.jsonl")

# Collision après fusion des schémas ISBN
groups = collections.defaultdict(list)
for r in ids:
    groups[(r.get("scheme_code"), r.get("normalized_value"))].append(r)

collisions = [
    {"scheme_code": k[0], "normalized_value": k[1],
     "count": len(v),
     "edition_ids": sorted(set(x.get("edition_id") for x in v))}
    for k, v in groups.items() if len(v) > 1
]

# Types requis par Neon : revue sans classification inventée
edition_types = {"print","ebook","audiobook","braille",
                 "microform","manuscript","other"}

edition_type_review = [
    {"edition_id": r.get("edition_id"),
     "source_external_id": r.get("source_external_id"),
     "observed_value": r.get("edition_type"),
     "reason": "EDITION_TYPE_REQUIRED_OR_UNMAPPED"}
    for r in editions if r.get("edition_type") not in edition_types
]

work_type_review = [
    {"work_id": r.get("work_id"),
     "source_work_key": r.get("source_work_key"),
     "reason": "WORK_TYPE_MAPPING_REQUIRED"}
    for r in works
]

agent_type_review = [
    {"agent_id": r.get("agent_id"),
     "source_agent_key": r.get("source_agent_key"),
     "reason": "AGENT_TYPE_MAPPING_REQUIRED"}
    for r in agents
]

missing_titles = [
    {"edition_id": r.get("edition_id"),
     "source_external_id": r.get("source_external_id")}
    for r in editions if not str(r.get("title") or "").strip()
]

invalid_identifiers = [
    r for r in ids
    if not str(r.get("scheme_code") or "").strip()
    or not str(r.get("identifier_value") or "").strip()
    or not str(r.get("normalized_value") or "").strip()
]

manifest = {
    "version": "V47.9",
    "status": "FINAL_SCHEMA_AUDIT_STAGING_ONLY",
    "editions": len(editions),
    "works": len(works),
    "agents": len(agents),
    "identifiers": len(ids),
    "post_mapping_identifier_collision_groups": len(collisions),
    "edition_type_review": len(edition_type_review),
    "work_type_review": len(work_type_review),
    "agent_type_review": len(agent_type_review),
    "missing_titles": len(missing_titles),
    "invalid_identifiers": len(invalid_identifiers),
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

for name, rows in [
    ("identifier_collisions_after_mapping.jsonl", collisions),
    ("edition_type_review.jsonl", edition_type_review),
    ("work_type_review.jsonl", work_type_review),
    ("agent_type_review.jsonl", agent_type_review),
    ("missing_titles.jsonl", missing_titles),
    ("invalid_identifiers.jsonl", invalid_identifiers)
]:
    write_jsonl(name, rows)

with open(f"{OUT}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n=== V47.9 — FINAL SCHEMA AUDIT ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
import json, glob, os

BASE = "/kaggle/working"

files = {
    "identifiers": "bibliographic_identifiers_plan.jsonl",
    "editions": "edition_candidates.jsonl",
    "works": "work_identity_candidates.jsonl",
    "agents": "agent_identity_candidates.jsonl"
}

for label, filename in files.items():
    paths = glob.glob(f"{BASE}/**/{filename}", recursive=True)
    if not paths:
        print(f"\n{label}: FICHIER INTROUVABLE")
        continue

    with open(paths[0], encoding="utf-8") as f:
        first = next((json.loads(x) for x in f if x.strip()), {})

    print(f"\n=== {label.upper()} ===")
    print("Fichier :", paths[0])
    print("Champs :", list(first.keys()))
    print("Exemple :", json.dumps(first, ensure_ascii=False, indent=2))


In [ ]:
import json, os, glob, collections

BASE = "/kaggle/working"
OUT = f"{BASE}/nexus_v47_10_corrected_audit"
os.makedirs(OUT, exist_ok=True)

def read_file(name):
    paths = glob.glob(f"{BASE}/**/{name}", recursive=True)
    if not paths:
        raise FileNotFoundError(name)
    with open(paths[0], encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write_file(name, rows):
    with open(f"{OUT}/{name}", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

ids = read_file("bibliographic_identifiers_plan.jsonl")
editions = read_file("edition_candidates.jsonl")
works = read_file("work_identity_candidates.jsonl")
agents = read_file("agent_identity_candidates.jsonl")

# 1. Validation des identifiants avec les vrais noms de champs
allowed_schemes = {
    "doi", "isbn", "isni", "lccn", "oclc",
    "olid", "orcid", "viaf"
}

invalid_ids = []
for r in ids:
    errors = []
    if r.get("target_scheme_code") not in allowed_schemes:
        errors.append("INVALID_TARGET_SCHEME")
    if not str(r.get("identifier_value") or "").strip():
        errors.append("EMPTY_IDENTIFIER_VALUE")
    if not str(r.get("normalized_value") or "").strip():
        errors.append("EMPTY_NORMALIZED_VALUE")

    targets = [
        r.get("edition_id"), r.get("work_id"),
        r.get("expression_id")
    ]
    if sum(x is not None for x in targets) != 1:
        errors.append("INVALID_TARGET_REFERENCE")

    if errors:
        invalid_ids.append({
            "identifier_id": r.get("bibliographic_identifier_id"),
            "errors": errors
        })

# 2. Unicité selon la contrainte Neon
groups = collections.defaultdict(list)
for r in ids:
    groups[(r.get("target_scheme_code"),
             r.get("normalized_value"))].append(r)

collisions = [
    {"scheme": k[0], "value": k[1], "count": len(v)}
    for k, v in groups.items() if len(v) > 1
]

# 3. Distribution des formats physiques, sans inventer les types
formats = collections.Counter(
    str(r.get("physical_format") or "NON_RENSEIGNE").strip()
    for r in editions
)

format_report = [
    {"physical_format": k, "count": v}
    for k, v in formats.most_common()
]

# 4. Champs disponibles pour les futures classifications
edition_fields = sorted(set().union(*(r.keys() for r in editions)))
work_fields = sorted(set().union(*(r.keys() for r in works)))
agent_fields = sorted(set().union(*(r.keys() for r in agents)))

manifest = {
    "version": "V47.10",
    "status": "CORRECTED_AUDIT_STAGING_ONLY",
    "editions": len(editions),
    "works": len(works),
    "agents": len(agents),
    "identifiers": len(ids),
    "invalid_identifiers": len(invalid_ids),
    "post_mapping_collision_groups": len(collisions),
    "physical_format_categories": len(formats),
    "edition_type_classification": "PENDING_SOURCE_REVIEW",
    "work_type_classification": "PENDING_SOURCE_REVIEW",
    "agent_type_classification": "PENDING_SOURCE_REVIEW",
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

write_file("invalid_identifiers.jsonl", invalid_ids)
write_file("identifier_collisions.jsonl", collisions)
write_file("physical_format_distribution.jsonl", format_report)

with open(f"{OUT}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== V47.10 — AUDIT CORRIGÉ ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nFORMATS PHYSIQUES :")
for row in format_report[:40]:
    print(row)

print("\nCHAMPS DISPONIBLES :")
print("EDITIONS :", edition_fields)
print("WORKS :", work_fields)
print("AGENTS :", agent_fields)
print("\nDOSSIER :", OUT)


In [ ]:
import json, os, glob, collections

BASE = "/kaggle/working"
OUT = f"{BASE}/nexus_v47_10_corrected_audit"
os.makedirs(OUT, exist_ok=True)

def read_file(name):
    paths = glob.glob(f"{BASE}/**/{name}", recursive=True)
    if not paths:
        raise FileNotFoundError(name)
    with open(paths[0], encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write_file(name, rows):
    with open(f"{OUT}/{name}", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

ids = read_file("bibliographic_identifiers_plan.jsonl")
editions = read_file("edition_candidates.jsonl")
works = read_file("work_identity_candidates.jsonl")
agents = read_file("agent_identity_candidates.jsonl")

# 1. Validation des identifiants avec les vrais noms de champs
allowed_schemes = {
    "doi", "isbn", "isni", "lccn", "oclc",
    "olid", "orcid", "viaf"
}

invalid_ids = []
for r in ids:
    errors = []
    if r.get("target_scheme_code") not in allowed_schemes:
        errors.append("INVALID_TARGET_SCHEME")
    if not str(r.get("identifier_value") or "").strip():
        errors.append("EMPTY_IDENTIFIER_VALUE")
    if not str(r.get("normalized_value") or "").strip():
        errors.append("EMPTY_NORMALIZED_VALUE")

    targets = [
        r.get("edition_id"), r.get("work_id"),
        r.get("expression_id")
    ]
    if sum(x is not None for x in targets) != 1:
        errors.append("INVALID_TARGET_REFERENCE")

    if errors:
        invalid_ids.append({
            "identifier_id": r.get("bibliographic_identifier_id"),
            "errors": errors
        })

# 2. Unicité selon la contrainte Neon
groups = collections.defaultdict(list)
for r in ids:
    groups[(r.get("target_scheme_code"),
             r.get("normalized_value"))].append(r)

collisions = [
    {"scheme": k[0], "value": k[1], "count": len(v)}
    for k, v in groups.items() if len(v) > 1
]

# 3. Distribution des formats physiques, sans inventer les types
formats = collections.Counter(
    str(r.get("physical_format") or "NON_RENSEIGNE").strip()
    for r in editions
)

format_report = [
    {"physical_format": k, "count": v}
    for k, v in formats.most_common()
]

# 4. Champs disponibles pour les futures classifications
edition_fields = sorted(set().union(*(r.keys() for r in editions)))
work_fields = sorted(set().union(*(r.keys() for r in works)))
agent_fields = sorted(set().union(*(r.keys() for r in agents)))

manifest = {
    "version": "V47.10",
    "status": "CORRECTED_AUDIT_STAGING_ONLY",
    "editions": len(editions),
    "works": len(works),
    "agents": len(agents),
    "identifiers": len(ids),
    "invalid_identifiers": len(invalid_ids),
    "post_mapping_collision_groups": len(collisions),
    "physical_format_categories": len(formats),
    "edition_type_classification": "PENDING_SOURCE_REVIEW",
    "work_type_classification": "PENDING_SOURCE_REVIEW",
    "agent_type_classification": "PENDING_SOURCE_REVIEW",
    "canonical_database_write": False,
    "sql_generated": False,
    "production_write": False
}

write_file("invalid_identifiers.jsonl", invalid_ids)
write_file("identifier_collisions.jsonl", collisions)
write_file("physical_format_distribution.jsonl", format_report)

with open(f"{OUT}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== V47.10 — AUDIT CORRIGÉ ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nFORMATS PHYSIQUES :")
for row in format_report[:40]:
    print(row)

print("\nCHAMPS DISPONIBLES :")
print("EDITIONS :", edition_fields)
print("WORKS :", work_fields)
print("AGENTS :", agent_fields)
print("\nDOSSIER :", OUT)


In [ ]:
import os, json, glob, shutil

BASE = "/kaggle/working"
OUT = f"{BASE}/NEXUS_FINAL_PACKAGE"
os.makedirs(OUT, exist_ok=True)

FILES = [
    "edition_candidates.jsonl",
    "work_identity_candidates.jsonl",
    "agent_identity_candidates.jsonl",
    "edition_works_plan.jsonl",
    "work_agents_plan.jsonl",
    "bibliographic_identifiers_plan.jsonl",
    "source_link_plan.jsonl",
]

# Retrouver les fichiers dans les dossiers V47 existants
manifest = {"files": {}, "errors": []}

for name in FILES:
    matches = glob.glob(f"{BASE}/**/{name}", recursive=True)
    if not matches:
        manifest["errors"].append(f"MISSING: {name}")
        continue

    # Préférer le fichier le plus récemment modifié
    src = max(matches, key=os.path.getmtime)
    dst = os.path.join(OUT, name)
    shutil.copy2(src, dst)

    with open(dst, encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())

    manifest["files"][name] = {
        "source": src,
        "rows": count,
        "bytes": os.path.getsize(dst)
    }

with open(f"{OUT}/consolidation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== CELLULE 1 — CONSOLIDATION ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
import os, json, glob, hashlib

BASE = "/kaggle/working"
OUT = f"{BASE}/NEXUS_FINAL_PACKAGE"

sources = glob.glob(f"{BASE}/**/stage_000001.jsonl", recursive=True)
if not sources:
    raise FileNotFoundError("Stage V10.2 absent : récupération nécessaire.")

source_path = max(sources, key=os.path.getmtime)
records = []

with open(source_path, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        raw = json.loads(line)
        payload = json.dumps(
            raw, ensure_ascii=False, sort_keys=True,
            separators=(",", ":")
        )
        records.append({
            "source_code": "openlibrary",
            "external_record_id": raw.get("key"),
            "source_record_version": "unknown",
            "payload": raw,
            "payload_sha256": hashlib.sha256(
                payload.encode("utf-8")
            ).hexdigest()
        })

missing_keys = [r for r in records if not r["external_record_id"]]
duplicate_keys = len(records) - len(set(
    r["external_record_id"] for r in records
    if r["external_record_id"]
))

path = f"{OUT}/source_records_plan.jsonl"
with open(path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

report = {
    "source_file": source_path,
    "source_records": len(records),
    "missing_external_ids": len(missing_keys),
    "duplicate_external_ids": duplicate_keys,
    "payload_hashes_present": sum(bool(r["payload_sha256"]) for r in records),
    "database_write": False
}

with open(f"{OUT}/source_records_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("=== CELLULE 2 — SOURCE RECORDS ===")
print(json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
import os, json, hashlib

BASE = "/kaggle/working/NEXUS_FINAL_PACKAGE"

EXPECTED = {
    "edition_candidates.jsonl": 3664,
    "work_identity_candidates.jsonl": 3137,
    "agent_identity_candidates.jsonl": 3159,
    "edition_works_plan.jsonl": 3157,
    "work_agents_plan.jsonl": 3185,
    "bibliographic_identifiers_plan.jsonl": 9316,
    "source_records_plan.jsonl": 3664
}

results = {}
errors = []

for filename, expected in EXPECTED.items():
    path = os.path.join(BASE, filename)

    if not os.path.exists(path):
        results[filename] = {"status": "MISSING"}
        errors.append(filename)
        continue

    with open(path, encoding="utf-8") as f:
        rows = [json.loads(x) for x in f if x.strip()]

    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    ok = len(rows) == expected
    results[filename] = {
        "rows": len(rows),
        "expected": expected,
        "status": "OK" if ok else "COUNT_MISMATCH",
        "sha256": digest.hexdigest()
    }

    if not ok:
        errors.append(filename)

report = {
    "version": "NEXUS_FINAL_PACKAGE",
    "status": "READY_FOR_REVIEW" if not errors else "BLOCKED",
    "files": results,
    "errors": errors,
    "canonical_database_write": False,
    "production_write": False,
    "sql_generated": False
}

with open(f"{BASE}/FINAL_VALIDATION.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("=== CELLULE 3 — VALIDATION FINALE ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("DOSSIER FINAL :", BASE)


In [ ]:
import os, json, uuid, collections

BASE = "/kaggle/working/NEXUS_FINAL_PACKAGE"
OUT = f"{BASE}/NEON_IMPORT_READY"
os.makedirs(OUT, exist_ok=True)

def read(name):
    path = f"{BASE}/{name}"
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write(name, rows):
    with open(f"{OUT}/{name}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def deterministic_id(value):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, value))

# Sources consolidées
editions_src = read("edition_candidates.jsonl")
works_src = read("work_identity_candidates.jsonl")
agents_src = read("agent_identity_candidates.jsonl")
edition_works_src = read("edition_works_plan.jsonl")
work_agents_src = read("work_agents_plan.jsonl")
identifiers_src = read("bibliographic_identifiers_plan.jsonl")
source_links_src = read("source_link_plan.jsonl")
source_records_src = read("source_records_plan.jsonl")

# Source system déjà présent dans Neon (branche de validation)
SOURCE_SYSTEM_ID = "67d4c7e7-a324-49ef-9f60-fc24ca4af926"

# Classification conservatrice des formats
format_map = {
    "Paperback": "print",
    "Hardcover": "print",
    "Ring-bound": "print",
    "Spiral-bound": "print",
    "Library Binding": "print",
    "Loose leaf": "print",
    "Textbook Binding": "print",
    "Mass Market Paperback": "print",
    "Plastic comb": "print",
    "Board book": "print",
    "School & Library Binding": "print",
    "Perfect Paperback": "print",
    "Turtleback": "print",
    "Unbound": "print",
    "Map": "print",
    "Poster": "print",
    "Wall Chart": "print",
    "Calendar": "print",
    "Cards": "print",
    "Microform": "microform",
    "Audio CD": "audiobook",
    "Audio Cassette": "audiobook",
    "Audio cassette": "audiobook"
}

# 1. Editions
edition_rows = []
for r in editions_src:
    fmt = str(r.get("physical_format") or "").strip()
    typ = format_map.get(fmt, "other")
    year = r.get("publication_year")

    edition_rows.append({
        "edition_id": r["edition_id"],
        "edition_type": typ,
        "title_statement": r["title"],
        "publication_year_from": year,
        "publication_date_display": r.get("publication_date"),
        "extent_text": (
            r.get("pagination")
            or (f'{r["number_of_pages"]} pages'
                if r.get("number_of_pages") else None)
        ),
        "physical_details": fmt or None,
        "format_note": fmt or None,
        "record_status": "provisional"
    })

# 2. Works et agents : types inconnus conservés explicitement en "other"
work_rows = [
    {
        "work_id": r["work_id"],
        "work_type": "other",
        "record_status": "provisional"
    }
    for r in works_src
]

agent_rows = [
    {
        "agent_id": r["agent_id"],
        "agent_type": "other",
        "record_status": "provisional"
    }
    for r in agents_src
]

# 3. Relations édition-œuvre
edition_work_rows = []
for r in edition_works_src:
    edition_work_rows.append({
        "edition_id": r["edition_id"],
        "work_id": r["work_id"],
        "content_role": r.get("content_role", "main"),
        "sequence_no": r.get("sequence_no", 1)
    })

# 4. Relations œuvre-agent
work_agent_rows = []
for r in work_agents_src:
    work_agent_rows.append({
        "work_id": r["work_id"],
        "agent_id": r["agent_id"],
        "role_code": r["role_code"],
        "sequence_no": r.get("sequence_no", 1)
    })

# 5. Identifiants bibliographiques
identifier_rows = []
for r in identifiers_src:
    identifier_rows.append({
        "bibliographic_identifier_id": r["bibliographic_identifier_id"],
        "scheme_code": r["target_scheme_code"],
        "identifier_value": r["identifier_value"],
        "normalized_value": r["normalized_value"],
        "work_id": r.get("work_id"),
        "expression_id": r.get("expression_id"),
        "edition_id": r.get("edition_id"),
        "is_preferred": r.get("is_preferred", False)
    })

# 6. Notices sources avec identifiants déterministes
source_rows = []
source_id_by_external = {}

for r in source_records_src:
    external_id = r["external_record_id"]
    version = r.get("source_record_version", "unknown")
    source_id = deterministic_id(
        f"nexus-source:openlibrary:{external_id}:{version}"
    )

    source_id_by_external[external_id] = source_id

    source_rows.append({
        "source_record_id": source_id,
        "source_system_id": SOURCE_SYSTEM_ID,
        "external_record_id": external_id,
        "source_record_version": version,
        "payload": r["payload"],
        "payload_sha256": r["payload_sha256"],
        "record_status": "new"
    })

# 7. Liens entre notices sources et éditions
source_link_rows = []
for r in source_links_src:
    external_id = r.get("source_external_id")
    source_id = source_id_by_external.get(external_id)

    if source_id:
        source_link_rows.append({
            "source_record_id": source_id,
            "edition_id": r["edition_id"],
            "link_type": "describes",
            "match_method": "exact",
            "confidence": 1.0
        })

# 8. Contrôles d'intégrité locaux
edition_ids = {r["edition_id"] for r in edition_rows}
work_ids = {r["work_id"] for r in work_rows}
agent_ids = {r["agent_id"] for r in agent_rows}
source_ids = {r["source_record_id"] for r in source_rows}

errors = []

def duplicate_count(values):
    values = list(values)
    return len(values) - len(set(values))

check_fk(edition_work_rows, "edition_id", edition_ids, "edition_works.edition_id")
check_fk(edition_work_rows, "work_id", work_ids, "edition_works.work_id")
check_fk(work_agent_rows, "work_id", work_ids, "work_agents.work_id")
check_fk(work_agent_rows, "agent_id", agent_ids, "work_agents.agent_id")
check_fk(identifier_rows, "edition_id", edition_ids, "identifiers.edition_id")
check_fk(source_link_rows, "source_record_id", source_ids, "source_links.source_record_id")
check_fk(source_link_rows, "edition_id", edition_ids, "source_links.edition_id")

# Unicité des clés et des contraintes importantes
def duplicate_count(values):
    return len(values) - len(set(values))

if duplicate_count(r["edition_id"] for r in edition_rows):
    errors.append("Duplicate edition IDs")
if duplicate_count(r["work_id"] for r in work_rows):
    errors.append("Duplicate work IDs")
if duplicate_count(r["agent_id"] for r in agent_rows):
    errors.append("Duplicate agent IDs")
if duplicate_count(
    (r["scheme_code"], r["normalized_value"]) for r in identifier_rows
):
    errors.append("Duplicate bibliographic identifier keys")
if duplicate_count(
    (r["source_system_id"], r["external_record_id"], r["source_record_version"])
    for r in source_rows
):
    errors.append("Duplicate source record keys")

# Export
plans = {
    "editions_import.jsonl": edition_rows,
    "works_import.jsonl": work_rows,
    "agents_import.jsonl": agent_rows,
    "edition_works_import.jsonl": edition_work_rows,
    "work_agents_import.jsonl": work_agent_rows,
    "bibliographic_identifiers_import.jsonl": identifier_rows,
    "source_records_import.jsonl": source_rows,
    "source_record_links_import.jsonl": source_link_rows
}

for filename, rows in plans.items():
    write(filename, rows)

manifest = {
    "version": "NEXUS_NEON_IMPORT_READY",
    "status": "READY_FOR_IMPORT_REVIEW" if not errors else "BLOCKED",
    "target": "v47-prevalidation-2026-11-26",
    "tables": {k: len(v) for k, v in plans.items()},
    "total_rows": sum(len(v) for v in plans.values()),
    "edition_types": dict(collections.Counter(
        r["edition_type"] for r in edition_rows
    )),
    "work_types": dict(collections.Counter(
        r["work_type"] for r in work_rows
    )),
    "agent_types": dict(collections.Counter(
        r["agent_type"] for r in agent_rows
    )),
    "errors": errors,
    "database_write": False,
    "sql_generated": False,
    "production_write": False
}

with open(f"{OUT}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== NEXUS — NEON IMPORT PACKAGE ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
import os, json, uuid, collections

BASE = "/kaggle/working/NEXUS_FINAL_PACKAGE"
OUT = f"{BASE}/NEON_IMPORT_READY"
os.makedirs(OUT, exist_ok=True)

def read(name):
    path = f"{BASE}/{name}"
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write(name, rows):
    with open(f"{OUT}/{name}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def deterministic_id(value):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, value))

# Sources consolidées
editions_src = read("edition_candidates.jsonl")
works_src = read("work_identity_candidates.jsonl")
agents_src = read("agent_identity_candidates.jsonl")
edition_works_src = read("edition_works_plan.jsonl")
work_agents_src = read("work_agents_plan.jsonl")
identifiers_src = read("bibliographic_identifiers_plan.jsonl")
source_links_src = read("source_link_plan.jsonl")
source_records_src = read("source_records_plan.jsonl")

# Source system déjà présent dans Neon (branche de validation)
SOURCE_SYSTEM_ID = "67d4c7e7-a324-49ef-9f60-fc24ca4af926"

# Classification conservatrice des formats
format_map = {
    "Paperback": "print",
    "Hardcover": "print",
    "Ring-bound": "print",
    "Spiral-bound": "print",
    "Library Binding": "print",
    "Loose leaf": "print",
    "Textbook Binding": "print",
    "Mass Market Paperback": "print",
    "Plastic comb": "print",
    "Board book": "print",
    "School & Library Binding": "print",
    "Perfect Paperback": "print",
    "Turtleback": "print",
    "Unbound": "print",
    "Map": "print",
    "Poster": "print",
    "Wall Chart": "print",
    "Calendar": "print",
    "Cards": "print",
    "Microform": "microform",
    "Audio CD": "audiobook",
    "Audio Cassette": "audiobook",
    "Audio cassette": "audiobook"
}

# 1. Editions
edition_rows = []
for r in editions_src:
    fmt = str(r.get("physical_format") or "").strip()
    typ = format_map.get(fmt, "other")
    year = r.get("publication_year")

    edition_rows.append({
        "edition_id": r["edition_id"],
        "edition_type": typ,
        "title_statement": r["title"],
        "publication_year_from": year,
        "publication_date_display": r.get("publication_date"),
        "extent_text": (
            r.get("pagination")
            or (f'{r["number_of_pages"]} pages'
                if r.get("number_of_pages") else None)
        ),
        "physical_details": fmt or None,
        "format_note": fmt or None,
        "record_status": "provisional"
    })

# 2. Works et agents : types inconnus conservés explicitement en "other"
work_rows = [
    {
        "work_id": r["work_id"],
        "work_type": "other",
        "record_status": "provisional"
    }
    for r in works_src
]

agent_rows = [
    {
        "agent_id": r["agent_id"],
        "agent_type": "other",
        "record_status": "provisional"
    }
    for r in agents_src
]

# 3. Relations édition-œuvre
edition_work_rows = []
for r in edition_works_src:
    edition_work_rows.append({
        "edition_id": r["edition_id"],
        "work_id": r["work_id"],
        "content_role": r.get("content_role", "main"),
        "sequence_no": r.get("sequence_no", 1)
    })

# 4. Relations œuvre-agent
work_agent_rows = []
for r in work_agents_src:
    work_agent_rows.append({
        "work_id": r["work_id"],
        "agent_id": r["agent_id"],
        "role_code": r["role_code"],
        "sequence_no": r.get("sequence_no", 1)
    })

# 5. Identifiants bibliographiques
identifier_rows = []
for r in identifiers_src:
    identifier_rows.append({
        "bibliographic_identifier_id": r["bibliographic_identifier_id"],
        "scheme_code": r["target_scheme_code"],
        "identifier_value": r["identifier_value"],
        "normalized_value": r["normalized_value"],
        "work_id": r.get("work_id"),
        "expression_id": r.get("expression_id"),
        "edition_id": r.get("edition_id"),
        "is_preferred": r.get("is_preferred", False)
    })

# 6. Notices sources avec identifiants déterministes
source_rows = []
source_id_by_external = {}

for r in source_records_src:
    external_id = r["external_record_id"]
    version = r.get("source_record_version", "unknown")
    source_id = deterministic_id(
        f"nexus-source:openlibrary:{external_id}:{version}"
    )

    source_id_by_external[external_id] = source_id

    source_rows.append({
        "source_record_id": source_id,
        "source_system_id": SOURCE_SYSTEM_ID,
        "external_record_id": external_id,
        "source_record_version": version,
        "payload": r["payload"],
        "payload_sha256": r["payload_sha256"],
        "record_status": "new"
    })

# 7. Liens entre notices sources et éditions
source_link_rows = []
for r in source_links_src:
    external_id = r.get("source_external_id")
    source_id = source_id_by_external.get(external_id)

    if source_id:
        source_link_rows.append({
            "source_record_id": source_id,
            "edition_id": r["edition_id"],
            "link_type": "describes",
            "match_method": "exact",
            "confidence": 1.0
        })

# 8. Contrôles d'intégrité locaux
edition_ids = {r["edition_id"] for r in edition_rows}
work_ids = {r["work_id"] for r in work_rows}
agent_ids = {r["agent_id"] for r in agent_rows}
source_ids = {r["source_record_id"] for r in source_rows}

errors = []

def check_fk(rows, field, allowed, label):
    bad = sum(r.get(field) not in allowed for r in rows)
    if bad:
        errors.append(f"{label}: {bad} références invalides")

check_fk(edition_work_rows, "edition_id", edition_ids, "edition_works.edition_id")
check_fk(edition_work_rows, "work_id", work_ids, "edition_works.work_id")
check_fk(work_agent_rows, "work_id", work_ids, "work_agents.work_id")
check_fk(work_agent_rows, "agent_id", agent_ids, "work_agents.agent_id")
check_fk(identifier_rows, "edition_id", edition_ids, "identifiers.edition_id")
check_fk(source_link_rows, "source_record_id", source_ids, "source_links.source_record_id")
check_fk(source_link_rows, "edition_id", edition_ids, "source_links.edition_id")

# Unicité des clés et des contraintes importantes
def duplicate_count(values):
    values = list(values)
    return len(values) - len(set(values))

if duplicate_count(r["edition_id"] for r in edition_rows):
    errors.append("Duplicate edition IDs")
if duplicate_count(r["work_id"] for r in work_rows):
    errors.append("Duplicate work IDs")
if duplicate_count(r["agent_id"] for r in agent_rows):
    errors.append("Duplicate agent IDs")
if duplicate_count(
    (r["scheme_code"], r["normalized_value"]) for r in identifier_rows
):
    errors.append("Duplicate bibliographic identifier keys")
if duplicate_count(
    (r["source_system_id"], r["external_record_id"], r["source_record_version"])
    for r in source_rows
):
    errors.append("Duplicate source record keys")

# Export
plans = {
    "editions_import.jsonl": edition_rows,
    "works_import.jsonl": work_rows,
    "agents_import.jsonl": agent_rows,
    "edition_works_import.jsonl": edition_work_rows,
    "work_agents_import.jsonl": work_agent_rows,
    "bibliographic_identifiers_import.jsonl": identifier_rows,
    "source_records_import.jsonl": source_rows,
    "source_record_links_import.jsonl": source_link_rows
}

for filename, rows in plans.items():
    write(filename, rows)

manifest = {
    "version": "NEXUS_NEON_IMPORT_READY",
    "status": "READY_FOR_IMPORT_REVIEW" if not errors else "BLOCKED",
    "target": "v47-prevalidation-2026-11-26",
    "tables": {k: len(v) for k, v in plans.items()},
    "total_rows": sum(len(v) for v in plans.values()),
    "edition_types": dict(collections.Counter(
        r["edition_type"] for r in edition_rows
    )),
    "work_types": dict(collections.Counter(
        r["work_type"] for r in work_rows
    )),
    "agent_types": dict(collections.Counter(
        r["agent_type"] for r in agent_rows
    )),
    "errors": errors,
    "database_write": False,
    "sql_generated": False,
    "production_write": False
}

with open(f"{OUT}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=== NEXUS — NEON IMPORT PACKAGE ===")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("DOSSIER :", OUT)


In [ ]:
import shutil

source = "/kaggle/working/NEXUS_FINAL_PACKAGE/NEON_IMPORT_READY"
archive = "/kaggle/working/NEXUS_NEON_IMPORT_READY"

shutil.make_archive(archive, "zip", source)

print("Archive créée :", archive + ".zip")

In [ ]:
import os
from IPython.display import FileLink, display

archive = "/kaggle/working/NEXUS_NEON_IMPORT_READY.zip"

if os.path.exists(archive):
    print("Archive prête à télécharger :")
    display(FileLink(archive))
else:
    print("ERREUR : archive introuvable.")


In [ ]:
from IPython.display import FileLink

FileLink(r'NEXUS_NEON_IMPORT_READY.zip')


In [ ]:
!pip -q install psycopg2-binary

from kaggle_secrets import UserSecretsClient
import psycopg2

url = UserSecretsClient().get_secret("NEON_DATABASE_URL")

conn = psycopg2.connect(url)
print("Hôte :", conn.info.host)
print("Base :", conn.info.dbname)
print("Utilisateur :", conn.info.user)

cur = conn.cursor()
cur.execute("""
    SELECT current_database(), current_user,
           inet_server_addr(), inet_server_port();
""")
print("Connexion PostgreSQL :", cur.fetchone())

conn.close()

In [ ]:
import os

archive = "/kaggle/working/NEXUS_NEON_IMPORT_READY.zip"

print("Archive présente :", os.path.exists(archive))
if os.path.exists(archive):
    print("Taille :", round(os.path.getsize(archive)/1024/1024, 2), "Mo")

In [ ]:
import zipfile, json, os

archive = "/kaggle/working/NEXUS_NEON_IMPORT_READY.zip"

with zipfile.ZipFile(archive, "r") as z:
    files = z.namelist()
    print("Fichiers dans l'archive :")
    for f in files:
        print(" -", f)

    manifest_path = next(
        (f for f in files if f.endswith("manifest.json")), None
    )

    if not manifest_path:
        raise Exception("MANIFEST INTROUVABLE")

    manifest = json.loads(z.read(manifest_path))
    print("\nMANIFEST :")
    print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
import os, json, zipfile, socket
from kaggle_secrets import UserSecretsClient

# 1. Configuration verrouillée
ARCHIVE = "/kaggle/working/NEXUS_NEON_IMPORT_READY.zip"
EXPECTED_HOST = "ep-proud-thunder-bebvwo0u.c-9.us-east-2.aws.neon.tech"
EXPECTED_DB = "neondb"
EXPECTED_TARGET = "v47-prevalidation-2026-11-26"
SCHEMA = "bibliotheque"

TABLE_FILES = {
    "works": "works_import.jsonl",
    "agents": "agents_import.jsonl",
    "editions": "editions_import.jsonl",
    "source_records": "source_records_import.jsonl",
    "edition_works": "edition_works_import.jsonl",
    "work_agents": "work_agents_import.jsonl",
    "bibliographic_identifiers": "bibliographic_identifiers_import.jsonl",
    "source_record_links": "source_record_links_import.jsonl"
}

# 2. Dépendance PostgreSQL
try:
    import psycopg2
    from psycopg2.extras import execute_values, Json
    from psycopg2 import sql
except ImportError:
    !pip -q install psycopg2-binary
    import psycopg2
    from psycopg2.extras import execute_values, Json
    from psycopg2 import sql

conn = None

try:
    # 3. Lecture et contrôle du paquet
    if not os.path.exists(ARCHIVE):
        raise Exception("ARCHIVE INTROUVABLE : " + ARCHIVE)

    with zipfile.ZipFile(ARCHIVE, "r") as z:
        names = set(z.namelist())
        if "manifest.json" not in names:
            raise Exception("MANIFEST ABSENT")

        manifest = json.loads(z.read("manifest.json"))

        if manifest.get("status") != "READY_FOR_IMPORT_REVIEW":
            raise Exception("STATUT DU MANIFESTE NON VALIDE")

        if manifest.get("target") != EXPECTED_TARGET:
            raise Exception("CIBLE DU MANIFESTE INCORRECTE")

        if manifest.get("errors") != []:
            raise Exception("LE MANIFESTE CONTIENT DES ERREURS")

        if manifest.get("total_rows") != 32946:
            raise Exception("TOTAL DE LIGNES INCORRECT")

        data = {}
        for table, filename in TABLE_FILES.items():
            if filename not in names:
                raise Exception("FICHIER ABSENT : " + filename)

            rows = [
                json.loads(line)
                for line in z.read(filename).decode("utf-8").splitlines()
                if line.strip()
            ]

            expected = manifest["tables"].get(filename)
            if len(rows) != expected:
                raise Exception(
                    f"{filename}: {len(rows)} lignes au lieu de {expected}"
                )

            if rows:
                keys = set(rows[0].keys())
                if any(set(r.keys()) != keys for r in rows):
                    raise Exception("Colonnes incohérentes : " + filename)

            data[table] = rows

    print("Paquet validé :", sum(map(len, data.values())), "lignes")

    # 4. Connexion via le secret Kaggle
    database_url = UserSecretsClient().get_secret("NEON_DATABASE_URL")
    if not database_url:
        raise Exception("SECRET NEON_DATABASE_URL INTROUVABLE")

    conn = psycopg2.connect(database_url)
    conn.autocommit = False

    params = conn.get_dsn_parameters()
    host = params.get("host", "").lower().rstrip(".")
    dbname = params.get("dbname")

    if host != EXPECTED_HOST:
        raise Exception("HÔTE NEON INCORRECT : arrêt de sécurité")

    if dbname != EXPECTED_DB:
        raise Exception("BASE DE DONNÉES INCORRECTE : arrêt de sécurité")

    print("Connexion validée :", host)
    print("Base :", dbname)
    print("Cible :", EXPECTED_TARGET)

    cur = conn.cursor()

    # 5. Vérifier que les huit tables sont vides
    for table in TABLE_FILES:
        cur.execute(
            sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                sql.Identifier(SCHEMA),
                sql.Identifier(table)
            )
        )
        count = cur.fetchone()[0]
        if count != 0:
            raise Exception(
                f"TABLE NON VIDE : {SCHEMA}.{table} ({count} lignes)"
            )

    print("Précontrôle : les 8 tables sont vides.")

    # 6. Vérifier les colonnes avant insertion
    for table, rows in data.items():
        if not rows:
            continue

        cur.execute("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = %s AND table_name = %s
        """, (SCHEMA, table))

        db_columns = {r[0] for r in cur.fetchall()}
        file_columns = set(rows[0].keys())

        unknown = file_columns - db_columns
        if unknown:
            raise Exception(
                f"Colonnes inconnues dans {table} : {sorted(unknown)}"
            )

    # 7. Import dans l'ordre des dépendances
    for table, rows in data.items():
        if not rows:
            continue

        columns = list(rows[0].keys())

        values = []
        for row in rows:
            values.append(tuple(
                Json(row[col]) if col == "payload" and row[col] is not None
                else row[col]
                for col in columns
            ))

        query = sql.SQL("INSERT INTO {}.{} ({}) VALUES %s").format(
            sql.Identifier(SCHEMA),
            sql.Identifier(table),
            sql.SQL(", ").join(map(sql.Identifier, columns))
        )

        execute_values(
            cur,
            query.as_string(conn),
            values,
            page_size=500
        )

        print(f"Inséré : {table} — {len(rows)} lignes")

    # 8. Contrôle des nombres avant validation définitive
    print("\nVérification avant COMMIT :")
    for table, rows in data.items():
        cur.execute(
            sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                sql.Identifier(SCHEMA),
                sql.Identifier(table)
            )
        )
        actual = cur.fetchone()[0]
        expected = len(rows)

        print(f"{table}: {actual}/{expected}")

        if actual != expected:
            raise Exception("ÉCART DE COMPTAGE : " + table)

    # 9. Validation transactionnelle
    conn.commit()

    print("\nIMPORT VALIDÉ AVEC SUCCÈS")
    print("Total :", sum(map(len, data.values())))
    print("Branche :", EXPECTED_TARGET)
    print("Production : NON MODIFIÉE")

except Exception as e:
    if conn is not None:
        conn.rollback()
    print("\nIMPORT ANNULÉ — TRANSACTION ANNULÉE")
    print("Erreur :", str(e))
    raise

finally:
    if conn is not None:
        conn.close()


In [1]:
import requests
import gzip
import json

DUMPS = {
    "EDITIONS": "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz",
    "WORKS": "https://openlibrary.org/data/ol_dump_works_latest.txt.gz",
    "AUTHORS": "https://openlibrary.org/data/ol_dump_authors_latest.txt.gz",
}

HEADERS = {
    "User-Agent": "BibliothequeScientifiqueNexus/0.1",
    "Accept-Encoding": "identity",
}

for name, url in DUMPS.items():
    print("\n" + "=" * 55)
    print("SOURCE :", name)

    try:
        with requests.get(
            url,
            headers=HEADERS,
            stream=True,
            allow_redirects=True,
            timeout=(30, 120)
        ) as response:

            response.raise_for_status()
            response.raw.decode_content = False

            print("HTTP :", response.status_code)
            print("URL finale :", response.url)
            print("Taille annoncée :", response.headers.get("Content-Length", "non indiquée"))

            with gzip.GzipFile(fileobj=response.raw, mode="rb") as gz:
                for i in range(3):
                    line = gz.readline()
                    if not line:
                        break

                    parts = line.decode("utf-8").rstrip("\n").split("\t", 4)

                    if len(parts) != 5:
                        raise ValueError("Format TSV inattendu")

                    record_type, key, revision, modified, payload = parts
                    record = json.loads(payload)

                    print({
                        "type": record_type,
                        "key": key,
                        "revision": revision,
                        "last_modified": modified,
                        "title": record.get("title") or record.get("name"),
                        "fields": len(record)
                    })

        print("TEST :", name, "OK")

    except Exception as e:
        print("TEST :", name, "ÉCHEC")
        print("Erreur :", str(e))



SOURCE : EDITIONS
HTTP : 200
URL finale : https://ia600702.us.archive.org/1/items/ol_dump_2026-10-31/ol_dump_editions_2026-10-31.txt.gz
Taille annoncée : 12630804778
{'type': '/type/edition', 'key': '/books/OL10000008M', 'revision': '3', 'last_modified': '2026-01-03T18:14:02.834193', 'title': 'Index to the House of Lords Parliamentary Debates', 'fields': 17}
{'type': '/type/edition', 'key': '/books/OL10000148M', 'revision': '3', 'last_modified': '2025-12-28T13:24:26.212572', 'title': '40house of Lords Official Report', 'fields': 16}
{'type': '/type/edition', 'key': '/books/OL10001070M', 'revision': '4', 'last_modified': '2025-12-31T04:32:26.931760', 'title': 'Road Traffic (NHS Charges) Bill (H.L.) (House of Lords Bills)', 'fields': 18}
TEST : EDITIONS OK

SOURCE : WORKS
HTTP : 200
URL finale : https://ia600702.us.archive.org/1/items/ol_dump_2026-10-31/ol_dump_works_2026-10-31.txt.gz
Taille annoncée : 4078855962
{'type': '/type/work', 'key': '/works/OL10000015W', 'revision': '4', 'last

In [2]:
import requests, gzip, io, json
from collections import Counter

URL = "https://ia600702.us.archive.org/1/items/ol_dump_2026-10-31/ol_dump_editions_2026-10-31.txt.gz"

N = 20_000
field_counts = Counter()
examples = {}
valid = 0
errors = 0

print("Analyse d'un échantillon de", N, "éditions...")

with requests.get(URL, stream=True, timeout=120) as response:
    response.raise_for_status()
    gz = gzip.GzipFile(fileobj=response.raw)
    stream = io.TextIOWrapper(gz, encoding="utf-8")

    for line in stream:
        if valid + errors >= N:
            break

        try:
            parts = line.rstrip("\n").split("\t", 4)
            if len(parts) != 5:
                errors += 1
                continue

            record = json.loads(parts[4])
            valid += 1

            for field, value in record.items():
                field_counts[field] += 1

                if field not in examples and value not in (None, "", [], {}):
                    examples[field] = str(value)[:350]

        except Exception:
            errors += 1

print("\n===== BILAN =====")
print("Notices valides :", valid)
print("Erreurs :", errors)

print("\n===== FRÉQUENCE DES CHAMPS =====")
for field, count in field_counts.most_common():
    print(f"{field:25} {count:8} ({count/valid*100:.1f}%)")

print("\n===== EXEMPLES DE VALEURS =====")
for field, value in examples.items():
    print(f"\n{field}: {value}")


Analyse d'un échantillon de 20000 éditions...

===== BILAN =====
Notices valides : 20000
Erreurs : 0

===== FRÉQUENCE DES CHAMPS =====
type                         20000 (100.0%)
key                          20000 (100.0%)
latest_revision              20000 (100.0%)
revision                     20000 (100.0%)
last_modified                20000 (100.0%)
title                        19997 (100.0%)
created                      19955 (99.8%)
source_records               19912 (99.6%)
publishers                   19463 (97.3%)
publish_date                 19349 (96.7%)
works                        16996 (85.0%)
authors                      15837 (79.2%)
languages                    15039 (75.2%)
isbn_10                      14238 (71.2%)
physical_format              13271 (66.4%)
subjects                     12658 (63.3%)
number_of_pages              12582 (62.9%)
isbn_13                      12513 (62.6%)
oclc_numbers                  7679 (38.4%)
publish_country               6966 (34.8%)

In [3]:
import requests, gzip, io, json, random
from collections import Counter

URL = "https://ia600702.us.archive.org/1/items/ol_dump_2026-10-31/ol_dump_editions_2026-10-31.txt.gz"

N = 20_000
rng = random.Random(20261127)

reservoir = []
total = 0
errors = 0

print("Parcours du dump complet et échantillonnage aléatoire...")

with requests.get(URL, stream=True, timeout=180) as response:
    response.raise_for_status()
    gz = gzip.GzipFile(fileobj=response.raw)
    stream = io.TextIOWrapper(gz, encoding="utf-8")

    for line in stream:
        try:
            parts = line.rstrip("\n").split("\t", 4)
            if len(parts) != 5:
                errors += 1
                continue

            record = json.loads(parts[4])
            total += 1

            if len(reservoir) < N:
                reservoir.append(record)
            else:
                j = rng.randrange(total)
                if j < N:
                    reservoir[j] = record

        except Exception:
            errors += 1

print("\n===== ÉCHANTILLONNAGE =====")
print("Notices parcourues :", total)
print("Notices conservées :", len(reservoir))
print("Erreurs :", errors)

subjects = Counter()
lc_classes = Counter()
dewey = Counter()
languages = Counter()
years = Counter()

for r in reservoir:
    for s in r.get("subjects", []) or []:
        if isinstance(s, str):
            subjects[s.strip().lower()] += 1

    for c in r.get("lc_classifications", []) or []:
        if isinstance(c, str) and c.strip():
            lc_classes[c.strip()[0].upper()] += 1

    for d in r.get("dewey_decimal_class", []) or []:
        if isinstance(d, str) and d.strip():
            dewey[d.strip()[:3]] += 1

    for lang in r.get("languages", []) or []:
        if isinstance(lang, dict):
            languages[lang.get("key", "unknown")] += 1

    date = str(r.get("publish_date", ""))
    import re
    match = re.search(r"\b(17|18|19|20)\d{2}\b", date)
    if match:
        years[match.group()] += 1

print("\n===== 50 SUJETS LES PLUS FRÉQUENTS =====")
for name, count in subjects.most_common(50):
    print(count, "|", name)

print("\n===== CLASSIFICATIONS LC : PREMIÈRE LETTRE =====")
for name, count in lc_classes.most_common():
    print(name, count)

print("\n===== CLASSIFICATIONS DEWEY =====")
for name, count in dewey.most_common(40):
    print(name, count)

print("\n===== LANGUES =====")
for name, count in languages.most_common(30):
    print(name, count)

print("\n===== ANNÉES DE PUBLICATION =====")
for name, count in sorted(years.items()):
    print(name, count)

# Sauvegarde de l'échantillon pour les analyses suivantes
with open("/kaggle/working/nexus_random_sample_20000.jsonl",
          "w", encoding="utf-8") as f:
    for r in reservoir:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\nÉchantillon sauvegardé.")


Parcours du dump complet et échantillonnage aléatoire...


KeyboardInterrupt: 

In [ ]:
 
import requests, gzip, io, json, random, time, re
from collections import Counter

URL = "https://ia600702.us.archive.org/1/items/ol_dump_2026-10-31/ol_dump_editions_2026-10-31.txt.gz"

N = 20_000
rng = random.Random(20261127)

reservoir = []
total = 0
errors = 0
start = time.time()

print("Connexion au dump...", flush=True)

with requests.get(URL, stream=True, timeout=(30, 300)) as response:
    response.raise_for_status()
    print("HTTP :", response.status_code, flush=True)

    gz = gzip.GzipFile(fileobj=response.raw)
    stream = io.TextIOWrapper(gz, encoding="utf-8")

    for line in stream:
        try:
            parts = line.rstrip("\n").split("\t", 4)
            if len(parts) != 5:
                errors += 1
                continue

            record = json.loads(parts[4])
            total += 1

            if len(reservoir) < N:
                reservoir.append(record)
            else:
                j = rng.randrange(total)
                if j < N:
                    reservoir[j] = record

        except Exception:
            errors += 1

        if total and total % 50_000 == 0:
            elapsed = time.time() - start
            print(
                f"{total:,} notices | "
                f"{len(reservoir):,} échantillonnées | "
                f"{elapsed/60:.1f} min",
                flush=True
            )

print("\nParcours terminé :", total, flush=True)

subjects = Counter()
lc_classes = Counter()
dewey = Counter()
languages = Counter()
years = Counter()

for r in reservoir:
    for s in r.get("subjects", []) or []:
        if isinstance(s, str):
            subjects[s.strip().lower()] += 1

    for c in r.get("lc_classifications", []) or []:
        if isinstance(c, str) and c.strip():
            lc_classes[c.strip()[0].upper()] += 1

    for d in r.get("dewey_decimal_class", []) or []:
        if isinstance(d, str) and d.strip():
            dewey[d.strip()[:3]] += 1

    for lang in r.get("languages", []) or []:
        if isinstance(lang, dict):
            languages[lang.get("key", "unknown")] += 1

    date = str(r.get("publish_date", ""))
    match = re.search(r"\b(17|18|19|20)\d{2}\b", date)
    if match:
        years[match.group()] += 1

print("\n===== BILAN =====")
print("Notices parcourues :", total)
print("Échantillon :", len(reservoir))
print("Erreurs :", errors)

for label, counter in [
    ("SUJETS", subjects),
    ("CLASSIFICATIONS LC", lc_classes),
    ("DEWEY", dewey),
    ("LANGUES", languages),
    ("ANNÉES", years)
]:
    print(f"\n===== {label} =====")
    for name, count in counter.most_common(50):
        print(count, "|", name)

with open("/kaggle/working/nexus_random_sample_20000.jsonl",
          "w", encoding="utf-8") as f:
    for r in reservoir:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\nÉchantillon sauvegardé.")


Connexion au dump...
HTTP : 200
50,000 notices | 20,000 échantillonnées | 0.1 min
100,000 notices | 20,000 échantillonnées | 0.5 min
150,000 notices | 20,000 échantillonnées | 1.0 min
200,000 notices | 20,000 échantillonnées | 1.1 min
250,000 notices | 20,000 échantillonnées | 1.7 min
300,000 notices | 20,000 échantillonnées | 2.2 min
350,000 notices | 20,000 échantillonnées | 2.6 min
400,000 notices | 20,000 échantillonnées | 3.2 min
450,000 notices | 20,000 échantillonnées | 3.8 min
500,000 notices | 20,000 échantillonnées | 4.5 min
550,000 notices | 20,000 échantillonnées | 5.0 min
600,000 notices | 20,000 échantillonnées | 5.4 min
650,000 notices | 20,000 échantillonnées | 5.8 min
700,000 notices | 20,000 échantillonnées | 6.3 min
750,000 notices | 20,000 échantillonnées | 6.8 min
800,000 notices | 20,000 échantillonnées | 7.1 min
850,000 notices | 20,000 échantillonnées | 7.5 min
900,000 notices | 20,000 échantillonnées | 8.1 min
950,000 notices | 20,000 échantillonnées | 8.5 min


In [ ]:
import os

path = "/kaggle/working/nexus_random_sample_20000.jsonl"

if os.path.exists(path):
    print("FICHIER TROUVÉ")
    print("Taille :", os.path.getsize(path), "octets")

    with open(path, encoding="utf-8") as f:
        count = sum(1 for _ in f)

    print("Notices sauvegardées :", count)
else:
    print("FICHIER ABSENT")
    print("L'échantillon n'a pas été sauvegardé.")
